# Liver-tumour synthesis on Colab (improved multi-GAN pipeline)

One self-contained notebook: it writes the four modules it needs (`Syn_Img_Pipeline_fixed.py`, `syn_training_improved.py`,
`insertion_improved.py`, `evaluate_synthesis.py`) into the Colab VM, then runs
**prepare → train → size-controlled generation → vessel-aware insertion with mass effect and QC → evaluation**.

**How to use**
1. `Runtime > Change runtime type > T4 GPU` (or better).
2. Pick a `MODE` in the settings cell:
   * `demo`: no data needed. It trains on synthetic patches and inserts into synthetic hosts in about 3 minutes. Use it to check the notebook.
   * `nifti`: the LiTS training volumes (`volume-*.nii[.gz]`, `segmentation-*.nii[.gz]`) in a Google Drive folder.
   * `cache`: the uint8 slice cache of `prepare_data_v2.py` (`images.npy`, `masks.npy`, `vol_index.npy`, `volumes.txt`, `config.json`)
     plus `per_volume.csv`.
3. `Runtime > Run all`.

Results go to Google Drive when `USE_DRIVE` is on, so a disconnect loses nothing. With `RESUME = True`, re-running the notebook
continues training from `checkpoints/gan_last.pt` and keeps finished stages. `MAX_TRAIN_MINUTES` stops training cleanly before the
Colab session limit.

Only the train part of the segmenter's split (`train_test_split(test_size=0.2, random_state=42)` over the sorted volume names) is ever
loaded for GAN training or as an insertion host. The 27 held-out volumes never reach the GAN.

In [ ]:
# ---------------- settings ----------------
MODE = "demo"  # @param ["demo", "nifti", "cache"]
USE_DRIVE = True  # @param {type:"boolean"}
DRIVE_RESULTS = "/content/drive/MyDrive/lits_gan_improved"  # @param {type:"string"}
NIFTI_DIR = "/content/drive/MyDrive/LiTS/raw"  # @param {type:"string"}
COPY_NIFTI_TO_LOCAL = False  # @param {type:"boolean"}
CACHE_DIR = "/content/drive/MyDrive/LiTS/cache_v2"  # @param {type:"string"}
PER_VOLUME_CSV = "/content/drive/MyDrive/LiTS/per_volume.csv"  # @param {type:"string"}
MAX_VOLUMES = 0  # @param {type:"integer"}
EPOCHS = 60  # @param {type:"integer"}
MAX_TRAIN_MINUTES = 150  # @param {type:"number"}
EARLY_STOP_PATIENCE = 8  # @param {type:"integer"}
RESUME = True  # @param {type:"boolean"}
SAMPLES_PER_CLASS = 16  # @param {type:"integer"}
INSERT_SIZE = "medium"  # @param ["small", "medium", "large", "keep"]
TUMOURS_PER_HOST = 1  # @param {type:"integer"}
MAX_HOSTS = 10  # @param {type:"integer"}
VESSEL_AWARE = True  # @param {type:"boolean"}
MASS_EFFECT = True  # @param {type:"boolean"}
CAPSULE = False  # @param {type:"boolean"}
RUN_EVALUATION = True  # @param {type:"boolean"}
SEED = 42  # @param {type:"integer"}

In [ ]:
import os, sys, subprocess, shutil, glob
IN_COLAB = "google.colab" in sys.modules
CODE_DIR = os.environ.get("LITS_COLAB_CODE_DIR", "/content/lits_synthesis" if IN_COLAB else os.path.abspath("lits_synthesis_code"))
os.makedirs(CODE_DIR, exist_ok=True)
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "nibabel"], check=False)
    if USE_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
RESULTS = DRIVE_RESULTS if (IN_COLAB and USE_DRIVE) else os.path.join(os.path.dirname(CODE_DIR), "lits_gan_results")
if MODE == "demo":
    RESULTS = os.path.join(RESULTS, "demo")
os.makedirs(RESULTS, exist_ok=True)
import torch
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
print("Colab:", IN_COLAB, "| GPU:", gpu or "none (CPU only: fine for demo, slow for real training)")
print("code   :", CODE_DIR)
print("results:", RESULTS)
if MODE == "nifti":
    assert os.path.isdir(NIFTI_DIR) and glob.glob(os.path.join(NIFTI_DIR, "volume-*.nii*")), f"no volume-*.nii* in {NIFTI_DIR}"
    if COPY_NIFTI_TO_LOCAL:  # local disk reads are much faster than Drive; needs the space (check with !df -h /content)
        local = "/content/data/raw"
        os.makedirs(local, exist_ok=True)
        for f in sorted(glob.glob(os.path.join(NIFTI_DIR, "*.nii*"))):
            if not os.path.exists(os.path.join(local, os.path.basename(f))):
                shutil.copy2(f, local)
        NIFTI_DIR = local
    print("NIfTI volumes:", len(glob.glob(os.path.join(NIFTI_DIR, "volume-*.nii*"))))
if MODE == "cache":
    for f in ("images.npy", "masks.npy", "vol_index.npy", "volumes.txt", "config.json"):
        assert os.path.exists(os.path.join(CACHE_DIR, f)), f"cache file {f} missing in {CACHE_DIR}"
    assert os.path.exists(PER_VOLUME_CSV), PER_VOLUME_CSV

## The pipeline modules

The next four cells write the code files. Run them as they are. They are the same files as in the repository's `Image Synthesis/` folder.

In [ ]:
%%writefile {CODE_DIR}/Syn_Img_Pipeline_fixed.py
"""Corrected multi-GAN liver-tumour synthesis pipeline (replaces Syn_Img_Pipeline_Code.py, which is kept as is).

    python Syn_Img_Pipeline_fixed.py          # all stages; LITS_GAN_STAGE=<stage> runs one (it reads the earlier stages' files from the results dir)
    python test_syn_pipeline_fixed.py         # unit / smoke tests

Stages (LITS_GAN_STAGE = all | prepare | train | generate | insert):
  prepare   per-volume: crop liver bbox -> resample to 1 mm -> fixed HU window -> [-1,1] -> pad once with air (-1)
            -> tumour patches sampled per connected component (+ healthy patches for evaluation)
  train     two 3D DCGANs, one 3D WGAN-GP, an Aggregator (adversarial, generators detached, L1-anchored to the
            generator mean) and a supervised mask head (BCE+Dice on real patch/mask pairs plus healthy patches as
            all-zero negatives; whole volumes are held out for its evaluation); losses logged, PNG montages, checkpoints
  generate  LITS_GAN_FUSION=aggregator|mixture ('mixture' = one random generator per sample, no aggregator);
            the mask head labels each sample, the mask is cleaned (threshold 0.5, largest component, min voxels,
            not touching the patch border) and invalid samples are rejected
  insert    host-aware insertion into train-part volumes only; augmented files go to <results>/aug (HU int16)
            (cache source: slabs of augmented cache slices -> <results>/aug_slabs.npz + gan_summary.json, see below)

Renames vs the original: StyleTransfer3D -> MaskHead3D (it is now a supervised segmenter, not a style net).
Generator contract: z (B,ld,1,1,1) -> ConvT(ld,8b,seed,1,0) -> seed^3 -> four k4/s2/p1 layers -> PATCH^3 with
seed = PATCH/16, so PATCH must be a multiple of 16 (16, 32, 48, 64, ...; 64 is the default and uses the usual 4^3 seed).
Discriminators end in a patch map that is averaged to ONE score per sample (B,).
Orientation: patches/insertion work in array-voxel space with per-axis spacing from the header zooms; the
raw LiTS arrays have mixed axcodes (LAS/RAS/LPS), so an inserted tumour may be mirrored relative to anatomy
(irrelevant for roughly isotropic blobs).
Device: LITS_DEVICE=auto|cpu|mps|cuda. auto = cuda if present, otherwise cpu: measured on an M4 Pro (batch 4, 64^3, width 32)
one full training batch took ~1.4 s on cpu and ~2.9 s on mps (measured twice, machine not idle), so mps is opt-in (LITS_DEVICE=mps).
PYTORCH_ENABLE_MPS_FALLBACK=1 is set so an op missing on MPS runs on CPU; conv3d / conv_transpose3d and the gradient-penalty
double backward run natively on mps.

Split: the segmenter's original rule, sklearn train_test_split over the lexicographically sorted volume file names of
LITS_DATA_DIR (test_size=0.2, random_state=42; 104 train / 27 held-out of the 131 LiTS volumes). GAN training AND insertion
hosts come only from the train part, so the segmenter's held-out volumes (its validation and test set) never reach the GAN.
Volumes whose image header zooms are exactly 1.0 x 1.0 x 1.0 (LiTS volumes 28-47: placeholder spacing, every mm-based step would
be wrong) are detected from the NIfTI header, logged and dropped from the GAN as well.

Cache source (Colab: no raw NIfTI, only the uint8 slice cache of prepare_data_v2.py): LITS_GAN_SOURCE=cache. A volume is then
rebuilt from the cache: HU-like float = u8/255*(hi-lo)+lo (the exact inverse of the cache window, so x = u8/255*2-1 is the GAN's
[-1,1] window), slice axis moved last -> (256,256,Z), spacing (nx/256*sx, ny/256*sy, sz) from per_volume.csv, canonical orientation as
cached. Everything else (patch sampling per component, networks, mask gate, feathered insertion) is the NIfTI code, run at the
working voxel size LITS_GAN_SPACING_MM (1.0 for nifti, 1.5 for cache; every mm-based constant is converted, at 1.0 they are unchanged).
LITS_GAN_VOLUME_LIST (json list of volume names, or {"volumes": [...]}) restricts GAN patch extraction AND insertion hosts; it is
intersected with the original-split train part and minus placeholder-spacing volumes, so the 27 held-out volumes are never loaded
(CacheSource.load refuses them). The insert stage then writes <results>/aug_slabs.npz (one slab = the z range of an insertion
window, merged when two windows of one host overlap in z; images/masks are the cache slices of the AUGMENTED host, unchanged voxels
bit-identical) and gan_summary.json. aug_slabs.npz keys: images uint8 (M,256,256), masks uint8 {0,1,2}, slab_id int32 (M,), host_vol (S,)
names, z0 (S,), new_tumor_vox (S,), n_slices (S,); the same file with zero slabs is written when nothing was accepted. It keeps inserting until LITS_GAN_NUM_AUG slabs are accepted or LITS_GAN_MAX_ATTEMPTS host
visits are used. Resume after a disconnect: LITS_GAN_RESUME=1 (train continues from checkpoints/gan_last.pt incl. optimiser state,
finished prepare / generate / insert outputs are kept); LITS_GAN_MAX_SECONDS stops training after the epoch that crosses the budget.
GAN steps run fp32 (WGAN-GP double backward), without per-step host syncs (losses stay on the device until the epoch end; the patch
set lives on the device as fp16).

Config via env vars (defaults at the top of the module): LITS_DATA_DIR, LITS_GAN_RESULTS_DIR, LITS_GAN_STAGE, LITS_DEVICE,
LITS_WIN_LO/HI, LITS_GAN_PATCH, LITS_GAN_LATENT, LITS_GAN_WIDTH, LITS_GAN_BATCH, LITS_GAN_EPOCHS, LITS_GAN_SEED,
LITS_GAN_MAX_VOLUMES, LITS_GAN_MAX_SLICE_MM, LITS_GAN_CAP_COMP, LITS_GAN_CAP_VOL, LITS_GAN_CAP_HEALTHY, LITS_GAN_JITTER,
LITS_GAN_MIN_TUMOR_VOX, LITS_GAN_POS_WEIGHT, LITS_GAN_MASK_STEPS, LITS_GAN_HOLDOUT, LITS_GAN_AGG_L1, LITS_GAN_SAMPLE_EVERY,
LITS_GAN_NUM_SAMPLES, LITS_GAN_FUSION, LITS_GAN_MIN_MASK_VOX, LITS_GAN_EDGE_MARGIN, LITS_GAN_INSERTIONS, LITS_GAN_FEATHER_MM,
LITS_GAN_EXCLUDE_MM; cache source: LITS_GAN_SOURCE, LITS_CACHE_DIR, LITS_PER_VOLUME, LITS_GAN_SPACING_MM, LITS_GAN_VOLUME_LIST,
LITS_GAN_NUM_AUG, LITS_GAN_MAX_ATTEMPTS, LITS_GAN_GEN_ATTEMPTS, LITS_GAN_RESUME, LITS_GAN_MAX_SECONDS, LITS_GAN_DATA_MB, LITS_GAN_CKPT_EVERY, LITS_GAN_FORCE.
"""
import os
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")  # must precede the torch import
import re
import csv
import glob
import json
import time
import random
import logging
import numpy as np
from sklearn.model_selection import train_test_split
import nibabel as nib
from scipy import ndimage as ndi
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

ROOT = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))  # repo root; data/ lives here
DATA_DIR = os.environ.get("LITS_DATA_DIR", os.path.join(ROOT, "data", "raw"))
RESULTS_DIR = os.environ.get("LITS_GAN_RESULTS_DIR", os.path.join(ROOT, "results", "gan_fixed"))
WIN_LO = float(os.environ.get("LITS_WIN_LO", -100))  # one window shared with the segmenter pipeline
WIN_HI = float(os.environ.get("LITS_WIN_HI", 250))
PATCH = int(os.environ.get("LITS_GAN_PATCH", 64))
LATENT_DIM = int(os.environ.get("LITS_GAN_LATENT", 64))
WIDTH = int(os.environ.get("LITS_GAN_WIDTH", 32))  # base channel count of every GAN net
BATCH_SIZE = int(os.environ.get("LITS_GAN_BATCH", 4))
EPOCHS = int(os.environ.get("LITS_GAN_EPOCHS", 50))
MAX_VOLUMES = int(os.environ.get("LITS_GAN_MAX_VOLUMES", 0))  # 0 = all usable volumes of the train part
NUM_SAMPLES = int(os.environ.get("LITS_GAN_NUM_SAMPLES", 16))
INSERTIONS = int(os.environ.get("LITS_GAN_INSERTIONS", 1))
FUSION = os.environ.get("LITS_GAN_FUSION", "aggregator")
SEED = int(os.environ.get("LITS_GAN_SEED", 42))
MAX_SLICE_MM = float(os.environ.get("LITS_GAN_MAX_SLICE_MM", 2.5))  # GAN training skips thicker volumes (0 = keep all)
CAP_COMP = int(os.environ.get("LITS_GAN_CAP_COMP", 4))  # patches per tumour component
CAP_VOL = int(os.environ.get("LITS_GAN_CAP_VOL", 24))  # tumour patches per volume
CAP_HEALTHY = int(os.environ.get("LITS_GAN_CAP_HEALTHY", 4))
JITTER = int(os.environ.get("LITS_GAN_JITTER", 4))  # voxels (1 mm) of random centre jitter
MIN_TUMOR_VOX = int(os.environ.get("LITS_GAN_MIN_TUMOR_VOX", 30))  # components smaller than this (1 mm voxels) are ignored
POS_WEIGHT = float(os.environ.get("LITS_GAN_POS_WEIGHT", 5))  # patches are tumour-centred (~10% positive), not 0.5%
MIN_MASK_VOX = int(os.environ.get("LITS_GAN_MIN_MASK_VOX", 100))
EDGE_MARGIN = int(os.environ.get("LITS_GAN_EDGE_MARGIN", 4))  # generated mask must stay this far from the patch border
MASK_STEPS = int(os.environ.get("LITS_GAN_MASK_STEPS", 1))  # mask-head updates per GAN batch (it learns slower than the GANs move)
HOLDOUT = float(os.environ.get("LITS_GAN_HOLDOUT", 0.2))  # fraction of volumes held out of mask-head training (eval only)
AGG_L1 = float(os.environ.get("LITS_GAN_AGG_L1", 5.0))  # anchor of the aggregator output to the generator mean (stops tanh saturation)
SAMPLE_EVERY = int(os.environ.get("LITS_GAN_SAMPLE_EVERY", 5))
FEATHER_MM = float(os.environ.get("LITS_GAN_FEATHER_MM", 2.5))  # alpha gaussian sigma (2-3 voxels at 1 mm)
EXCLUDE_MM = float(os.environ.get("LITS_GAN_EXCLUDE_MM", 5))  # keep-out margin around existing tumours
GAN_SOURCE = os.environ.get("LITS_GAN_SOURCE", "nifti").lower()  # nifti | cache
CACHE_DIR = os.environ.get("LITS_CACHE_DIR", os.path.join(ROOT, "data", "cache_v2"))
PER_VOLUME = os.environ.get("LITS_PER_VOLUME", os.path.join(ROOT, "analysis", "output", "per_volume.csv"))
WORK_MM = float(os.environ.get("LITS_GAN_SPACING_MM", 1.5 if GAN_SOURCE == "cache" else 1.0))  # working voxel size of patches / generator
VOLUME_LIST = os.environ.get("LITS_GAN_VOLUME_LIST", "")
NUM_AUG = int(os.environ.get("LITS_GAN_NUM_AUG", 32))  # cache source: target number of accepted slabs
MAX_ATTEMPTS = int(os.environ.get("LITS_GAN_MAX_ATTEMPTS", 0))  # host visits in the insert stage; 0 = 5 x NUM_AUG
GEN_ATTEMPTS = int(os.environ.get("LITS_GAN_GEN_ATTEMPTS", 0))  # generated candidates tried by the mask gate; 0 = 50 x pool size
RESUME = os.environ.get("LITS_GAN_RESUME", "0") == "1"
MAX_SECONDS = float(os.environ.get("LITS_GAN_MAX_SECONDS", 0))  # 0 = no time limit on GAN training
DATA_MB = float(os.environ.get("LITS_GAN_DATA_MB", 3072))  # patch set kept on the device up to this size, else pinned host memory
CKPT_EVERY = int(os.environ.get("LITS_GAN_CKPT_EVERY", 5))  # gan_last.pt (~240 MB with optimiser state at width 32) every N epochs, always at the end
FORCE = os.environ.get("LITS_GAN_FORCE", "0") == "1"
NOISE_SIGMA_MM = 1.5  # high-pass scale used to estimate / re-inject CT noise
NOISE_PRE_MM = 0.8  # pre-smoothing of the white noise (host high-pass lag-1 autocorrelation measured 0.3-0.76, white gives 0)
REAL_LABEL = 0.9  # one-sided label smoothing for DCGAN discriminators
N_CRITIC, GP_LAMBDA = 5, 10.0

logger = logging.getLogger("gan_fixed")


def pick_device():
    """LITS_DEVICE=auto -> cuda if present, else cpu (mps is opt-in, see docstring). An explicit cuda/mps request that this
    machine cannot serve is an error, not a silent fall back to a 100x slower cpu run."""
    want = os.environ.get("LITS_DEVICE", "auto").lower()
    if want == "auto":
        return torch.device("cuda" if torch.cuda.is_available() else "cpu")
    dev = torch.device(want)
    if dev.type == "cuda" and not torch.cuda.is_available():
        raise RuntimeError("LITS_DEVICE=cuda but torch.cuda.is_available() is False (Colab: Runtime > Change runtime type > GPU)")
    if dev.type == "mps" and not (hasattr(torch.backends, "mps") and torch.backends.mps.is_available()):
        raise RuntimeError("LITS_DEVICE=mps but the mps backend is not available")
    return dev


def setup_backends(device):
    """fp32 everywhere for the GAN (WGAN-GP double backward): TF32 off; cuDNN autotune on cuda (fixed shapes)."""
    if device.type == "cuda":
        torch.backends.cudnn.benchmark = True
        torch.backends.cudnn.allow_tf32 = False
        torch.backends.cuda.matmul.allow_tf32 = False


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    logger.info(f"seeds: random={seed} numpy={seed} torch={seed}")


def setup_logging(results_dir):
    os.makedirs(results_dir, exist_ok=True)
    logging.basicConfig(
        level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s",
        handlers=[logging.FileHandler(os.path.join(results_dir, "pipeline.log"), mode="a"), logging.StreamHandler()],
        force=True)


def check_patch(p):
    assert p >= 16 and p % 16 == 0, f"PATCH must be a multiple of 16, got {p}"


# -----------------------------
# Normalisation, patch extraction, I/O
# -----------------------------
def to_unit(hu, lo=WIN_LO, hi=WIN_HI):
    """HU -> [-1,1] with one fixed window; air (<= lo) maps to -1."""
    return np.clip((hu - lo) / (hi - lo), 0.0, 1.0).astype(np.float32) * 2.0 - 1.0


def from_unit(x, lo=WIN_LO, hi=WIN_HI):
    return (x + 1.0) * 0.5 * (hi - lo) + lo


def assert_unit_range(x, name="data", tol=1e-3):
    lo, hi = float(x.min()), float(x.max())
    assert lo >= -1 - tol and hi <= 1 + tol, f"{name} outside [-1,1]: [{lo:.3f},{hi:.3f}] (double normalisation?)"


def pad_air(vol, size, value):
    """Pad once by size//2 on every side; patch for centre c is then padded[c:c+size] (no low-edge shift)."""
    return np.pad(vol, size // 2, mode="constant", constant_values=value)


def extract_patch(padded, centre, size):
    c = [int(v) for v in centre]
    return padded[c[0]:c[0] + size, c[1]:c[1] + size, c[2]:c[2] + size]


def case_id(name):
    m = re.search(r"(\d+)", os.path.basename(str(name)))
    assert m, f"no case id in {name}"
    return int(m.group(1))


def pair_cases(data_dir, ids=None):
    """{case id: (volume path, segmentation path)}, paired by id (not by sorted position)."""
    vols = {case_id(p): p for p in glob.glob(os.path.join(data_dir, "volume-*.nii*"))}
    segs = {case_id(p): p for p in glob.glob(os.path.join(data_dir, "segmentation-*.nii*"))}
    if ids is None:
        assert set(vols) == set(segs), f"unpaired ids: {sorted(set(vols) ^ set(segs))}"
        ids = sorted(vols)
    for i in ids:
        assert i in vols and i in segs, f"case {i}: volume/segmentation pair missing in {data_dir}"
    return {i: (vols[i], segs[i]) for i in ids}


def sorted_volume_names(data_dir):
    """Volume file names in the segmenter's order (lexicographic: volume-0, volume-1, volume-10, ...)."""
    return sorted(os.path.basename(p) for p in glob.glob(os.path.join(data_dir, "volume-*.nii*")))


def train_part_ids(data_dir):
    """Case ids of the segmenter's train part: the original train_test_split(test_size=0.2, random_state=42) over the sorted names."""
    names = sorted_volume_names(data_dir)
    assert names, f"no volume-*.nii* in {data_dir}"
    train_idx, _ = train_test_split(range(len(names)), test_size=0.2, random_state=42)
    return sorted(case_id(names[i]) for i in train_idx)


def placeholder_spacing(vpath):
    """True when the image header zooms are exactly 1.0 x 1.0 x 1.0 (LiTS volumes 28-47); header only, no voxel data is read."""
    return tuple(float(z) for z in nib.load(vpath).header.get_zooms()[:3]) == (1.0, 1.0, 1.0)


def parse_volume_list(path):
    """Case ids named by a json file: a list of names ('volume-5.nii', 'volume-5' or 5), {"volumes": [...]}, or a dict keyed by volume names."""
    with open(path) as f:
        obj = json.load(f)
    if isinstance(obj, dict):
        if isinstance(obj.get("volumes"), list):
            obj = obj["volumes"]
        elif all(re.match(r"volume-\d+", str(k)) for k in obj):
            obj = list(obj)
        else:
            raise ValueError(f"{path}: expected a list of volume names, {{'volumes': [...]}} or a dict keyed by volume names")
    assert isinstance(obj, list) and obj, f"{path}: empty volume list"
    return {case_id(str(v)) for v in obj}


def split_cases(data_dir=None, max_volumes=None, restrict=None):
    """Case ids for GAN training and insertion hosts: the train part of the original split (the held-out part never
    reaches the GAN), minus volumes with placeholder header spacing (makes every mm-based step wrong).
    restrict: optional set of case ids (LITS_GAN_VOLUME_LIST); the result is its intersection with the above."""
    data_dir = data_dir or DATA_DIR
    max_volumes = MAX_VOLUMES if max_volumes is None else max_volumes
    train = train_part_ids(data_dir)
    paths = {case_id(p): p for p in glob.glob(os.path.join(data_dir, "volume-*.nii*"))}
    cand = train if restrict is None else [i for i in train if i in restrict]
    bad = [i for i in cand if placeholder_spacing(paths[i])]
    if bad:
        logger.info(f"dropping {len(bad)} train-part volumes with placeholder 1x1x1 header spacing: {bad}")
    ids = [i for i in cand if i not in bad]
    if restrict is not None:
        logger.info(f"volume list: {len(restrict)} named, {len(ids)} usable (in the train part, real spacing)")
    assert ids, f"no usable train-part volumes in {data_dir}"
    return ids[:max_volumes] if max_volumes else ids


class NiftiSource:
    """Raw LiTS NIfTI volumes (the original path). load -> (HU float32 (X,Y,Z), seg uint8, spacing mm, affine)."""
    kind = "nifti"

    def __init__(self, data_dir=None, work_mm=1.0):
        self.data_dir, self.work_mm, self._cases = data_dir or DATA_DIR, work_mm, None

    def split_cases(self, restrict=None, max_volumes=None):
        return split_cases(self.data_dir, max_volumes, restrict)

    def prepare(self, ids):
        self._cases = pair_cases(self.data_dir, ids)

    def load(self, cid):
        return load_case(*self._cases[cid])


class CacheSource:
    """The uint8 slice cache of prepare_data_v2.py as a volume source (no NIfTI needed). A volume is rebuilt as
    HU-like float u8/255*(hi-lo)+lo (exact inverse of the cache window), slice axis last -> (256,256,Z), labels uint8,
    spacing (nx/256*sx, ny/256*sy, sz) from per_volume.csv (cache = 2x in-plane downsample of the 512x512 raw slices).
    Axis order and canonical orientation are the cache's own (array axes 0/1 of a slice, not mirrored back)."""
    kind = "cache"

    def __init__(self, cache_dir=None, per_volume=None, work_mm=None):
        self.cache_dir, self.work_mm = cache_dir or CACHE_DIR, WORK_MM if work_mm is None else work_mm
        cd = self.cache_dir
        with open(os.path.join(cd, "config.json")) as f:
            cfg = json.load(f)
        self.lo, self.hi = float(cfg["win_lo"]), float(cfg["win_hi"])
        assert (self.lo, self.hi) == (WIN_LO, WIN_HI), f"cache window [{self.lo},{self.hi}] != LITS_WIN_LO/HI [{WIN_LO},{WIN_HI}]"
        with open(os.path.join(cd, "volumes.txt")) as f:
            self.names = f.read().split()
        self.vol_index = np.load(os.path.join(cd, "vol_index.npy"))
        assert len(self.names) == int(self.vol_index.max()) + 1 and (np.diff(self.vol_index) >= 0).all(), "bad vol_index"
        cnt = np.bincount(self.vol_index, minlength=len(self.names))
        self.start = np.concatenate([[0], np.cumsum(cnt)])
        self.images = np.load(os.path.join(cd, "images.npy"), mmap_mode="r")
        self.masks = np.load(os.path.join(cd, "masks.npy"), mmap_mode="r")
        assert len(self.images) == len(self.masks) == len(self.vol_index), "cache arrays disagree in length"
        self.cid_to_idx = {case_id(n): i for i, n in enumerate(self.names)}
        with open(per_volume or PER_VOLUME, newline="") as f:
            self.per_volume = {r["name"]: r for r in csv.DictReader(f)}
        missing = [n for n in self.names if n not in self.per_volume]
        assert not missing, f"per_volume.csv has no row for {missing[:5]} (spacing source)"
        tr, _ = train_test_split(range(len(self.names)), test_size=0.2, random_state=42)  # the segmenter's rule on volumes.txt order
        self.train_ids = sorted(case_id(self.names[i]) for i in tr)

    def raw_spacing(self, cid):
        r = self.per_volume[self.names[self.cid_to_idx[cid]]]
        return float(r["sx_mm"]), float(r["sy_mm"]), float(r["sz_mm"])

    def spacing(self, cid):
        r = self.per_volume[self.names[self.cid_to_idx[cid]]]
        sx, sy, sz = self.raw_spacing(cid)
        return (float(r["nx"]) / self.images.shape[1] * sx, float(r["ny"]) / self.images.shape[2] * sy, sz)

    def placeholder(self, cid):
        return self.raw_spacing(cid) == (1.0, 1.0, 1.0)  # LiTS 28-47: header carries 1x1x1 although the true spacing is unknown

    def train_part_ids(self):
        return list(self.train_ids)

    def split_cases(self, restrict=None, max_volumes=None):
        max_volumes = MAX_VOLUMES if max_volumes is None else max_volumes
        cand = self.train_ids if restrict is None else [i for i in self.train_ids if i in restrict]
        bad = [i for i in cand if self.placeholder(i)]
        if bad:
            logger.info(f"dropping {len(bad)} train-part volumes with placeholder 1x1x1 header spacing: {bad}")
        ids = [i for i in cand if i not in bad]
        if restrict is not None:
            unknown = sorted(restrict - set(self.cid_to_idx))
            held = sorted((restrict & set(self.cid_to_idx)) - set(self.train_ids))
            logger.info(f"volume list: {len(restrict)} named, {len(ids)} usable; ignored: {len(held)} held-out part {held}, "
                        f"{len(bad)} placeholder, {len(unknown)} unknown {unknown}")
        assert ids, f"no usable train-part volumes in the cache {self.cache_dir}"
        return ids[:max_volumes] if max_volumes else ids

    def prepare(self, ids):
        pass

    def load_u8(self, cid):
        """Cache slices of one TRAIN-part volume: (images (Z,256,256) uint8, masks (Z,256,256) uint8), copied out of the memmap."""
        assert cid in self.train_ids, f"volume {cid} is not in the original-split train part (held-out volumes are never loaded by the GAN)"
        i = self.cid_to_idx[cid]
        a, b = int(self.start[i]), int(self.start[i + 1])
        return np.array(self.images[a:b]), np.array(self.masks[a:b])

    def to_hu(self, u8):
        return u8.astype(np.float32) * np.float32((self.hi - self.lo) / 255.0) + np.float32(self.lo)

    def load(self, cid):
        img, msk = self.load_u8(cid)
        return self.to_hu(np.moveaxis(img, 0, 2)), np.ascontiguousarray(np.moveaxis(msk, 0, 2)), self.spacing(cid), None

    def name(self, cid):
        return self.names[self.cid_to_idx[cid]]


def make_source(kind=None):
    kind = kind or GAN_SOURCE
    assert kind in ("nifti", "cache"), f"LITS_GAN_SOURCE must be nifti or cache, got {kind}"
    return CacheSource() if kind == "cache" else NiftiSource(work_mm=WORK_MM)


def load_case(vpath, spath):
    vimg, simg = nib.load(vpath), nib.load(spath)
    vol = np.asarray(vimg.dataobj).astype(np.float32)
    seg = np.rint(np.asarray(simg.dataobj)).astype(np.uint8)
    assert vol.shape == seg.shape, f"shape mismatch {vol.shape} vs {seg.shape}"
    return vol, seg, tuple(float(z) for z in vimg.header.get_zooms()[:3]), vimg.affine


def save_nifti(arr, affine, path, dtype):
    """Fresh header from the host affine (a reused header could quantise values on disk)."""
    img = nib.Nifti1Image(np.asarray(arr, dtype=dtype), affine)
    img.set_data_dtype(dtype)
    nib.save(img, path)


def save_npz_atomic(path, **arrays):
    """np.savez to a temp file in the same directory, then os.replace: a disconnect never leaves a half-written file under `path`."""
    os.makedirs(os.path.dirname(os.path.abspath(path)), exist_ok=True)
    tmp = path + ".tmp"
    with open(tmp, "wb") as f:
        np.savez(f, **arrays)
    os.replace(tmp, path)


class _Npz(dict):
    @property
    def files(self):
        return list(self)


def load_npz(path):
    """All arrays of an .npz read eagerly and the file closed: a lazy np.load keeps the handle open, and on Windows an open
    file cannot be replaced (save_npz_atomic) or deleted. The result is a dict with an NpzFile-like .files."""
    with np.load(path) as z:
        return _Npz({k: z[k] for k in z.files})


def read_json(path):
    with open(path) as f:
        return json.load(f)


def write_json_atomic(path, obj):
    os.makedirs(os.path.dirname(os.path.abspath(path)), exist_ok=True)
    tmp = path + ".tmp"
    with open(tmp, "w") as f:
        json.dump(obj, f, indent=1)
    os.replace(tmp, path)


def to_int16(hu):
    return np.clip(np.rint(hu), -32768, 32767).astype(np.int16)


def liver_crop(seg, spacing, margin_mm):
    """Slices of the liver bbox plus a margin; None for an empty mask."""
    objs = ndi.find_objects((seg > 0).astype(np.int8))
    if not objs:
        return None
    return tuple(slice(max(0, s.start - int(np.ceil(margin_mm / sp)) - 1), min(n, s.stop + int(np.ceil(margin_mm / sp)) + 1))
                 for s, sp, n in zip(objs[0], spacing, seg.shape))


def resample_iso(arr, spacing, order, work_mm=1.0):
    """To isotropic work_mm voxels (1 mm for the NIfTI path)."""
    return ndi.zoom(arr, np.asarray(spacing) / work_mm, order=order, mode="nearest")


# -----------------------------
# Patch sampling
# -----------------------------
def sample_centres(tumor, rng, cap_comp=CAP_COMP, cap_vol=CAP_VOL, jitter=JITTER, min_vox=MIN_TUMOR_VOX):
    """Centres per connected tumour component (cap per component and per volume, random jitter).
    Round-robin over components so small tumours always get a patch before large ones get their k-th."""
    lab, _ = ndi.label(tumor, structure=np.ones((3, 3, 3)))
    per_comp = []
    for i, sl in enumerate(ndi.find_objects(lab), start=1):
        coords = np.argwhere(lab[sl] == i) + np.array([s.start for s in sl])
        if len(coords) < min_vox:
            continue
        pick = coords[rng.choice(len(coords), min(cap_comp, len(coords)), replace=False)]
        pick = pick + rng.integers(-jitter, jitter + 1, size=pick.shape)
        per_comp.append((i, np.clip(pick, 0, np.array(tumor.shape) - 1)))
    order = rng.permutation(len(per_comp))
    out = []
    for k in range(cap_comp):
        for j in order:
            comp, pick = per_comp[j]
            if k < len(pick) and len(out) < cap_vol:
                out.append((pick[k], comp))
    return out


def sample_healthy(seg_rs, seg_pad, rng, size=PATCH, cap=CAP_HEALTHY):
    """Liver-centred patches that contain no tumour label (negatives / evaluation for the mask head, never for GAN training)."""
    liver = np.argwhere(seg_rs == 1)
    out = []
    if len(liver) == 0:
        return out
    for c in liver[rng.choice(len(liver), min(len(liver), cap * 10), replace=False)]:
        if (extract_patch(seg_pad, c, size) == 2).sum() == 0:
            out.append(c)
        if len(out) >= cap:
            break
    return out


def vox_at(n_vox_1mm, work_mm):
    """A voxel count defined at 1 mm voxels (volume ~ n mm^3) converted to the working voxel size (unchanged at 1.0)."""
    return max(1, int(round(n_vox_1mm / work_mm ** 3)))


def prepare_patches(ids, out_path, data_dir=DATA_DIR, size=PATCH, rng=None, source=None):
    check_patch(size)
    rng = rng or np.random.default_rng(SEED)
    source = source or NiftiSource(data_dir)
    work = source.work_mm
    source.prepare(ids)
    min_vox, jitter = vox_at(MIN_TUMOR_VOX, work), max(1, int(round(JITTER / work)))  # mm-defined constants at the working voxel size
    imgs, masks, healthy, meta, healthy_cid = [], [], [], [], []
    for cid in ids:
        vol, seg, sp, _ = source.load(cid)
        if MAX_SLICE_MM and max(sp) > MAX_SLICE_MM:
            logger.info(f"case {cid}: slice thickness {max(sp):.1f} mm > {MAX_SLICE_MM}, skipped for GAN training")
            continue
        sl = liver_crop(seg, sp, (size / 2 + EDGE_MARGIN) * work)  # crop first, then resample (no whole-volume zoom)
        if sl is None:
            logger.warning(f"case {cid}: empty segmentation, skipped")
            continue
        vol_rs = to_unit(resample_iso(vol[sl], sp, 1, work))
        seg_rs = resample_iso(seg[sl], sp, 0, work)
        vol_pad, seg_pad = pad_air(vol_rs, size, -1.0), pad_air(seg_rs, size, 0)
        cs = sample_centres(seg_rs == 2, rng, jitter=jitter, min_vox=min_vox)
        for c, comp in cs:
            imgs.append(extract_patch(vol_pad, c, size))
            masks.append(extract_patch(seg_pad, c, size) == 2)
            meta.append((cid, comp))
        for c in sample_healthy(seg_rs, seg_pad, rng, size):
            healthy.append(extract_patch(vol_pad, c, size))
            healthy_cid.append(cid)
        logger.info(f"case {cid}: {len(cs)} tumour patches ({len({k for _, k in cs})} components), "
                    f"healthy so far {len(healthy)}")
    assert imgs, "no tumour patches extracted (every listed volume was too thick, had no tumour component above the size limit, or no liver)"
    imgs, masks = np.stack(imgs).astype(np.float16), np.stack(masks).astype(np.uint8)
    healthy = np.stack(healthy).astype(np.float16) if healthy else np.zeros((0, size, size, size), np.float16)
    assert_unit_range(imgs, "patches")
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    save_npz_atomic(out_path, images=imgs, masks=masks, healthy=healthy, meta=np.array(meta), healthy_cid=np.array(healthy_cid, dtype=int),
                    work_mm=np.float32(work), ids=np.array(ids, dtype=int))
    logger.info(f"saved {len(imgs)} tumour / {len(healthy)} healthy patches ({size}^3 at {work} mm) -> {out_path}")
    return out_path


# -----------------------------
# Networks
# -----------------------------
def weights_init(m):
    if isinstance(m, (nn.Conv3d, nn.ConvTranspose3d)):
        nn.init.normal_(m.weight, 0.0, 0.02)
        if m.bias is not None:
            nn.init.zeros_(m.bias)
    elif isinstance(m, nn.BatchNorm3d):
        nn.init.normal_(m.weight, 1.0, 0.02)
        nn.init.zeros_(m.bias)


class DCGAN3DGenerator(nn.Module):
    def __init__(self, ld=LATENT_DIM, patch=PATCH, base=WIDTH):
        super().__init__()
        check_patch(patch)
        b, self.ld, self.patch = base, ld, patch
        # conv bias is dropped before BatchNorm (its gradient is exactly zero there)
        self.net = nn.Sequential(
            nn.ConvTranspose3d(ld, b * 8, patch // 16, 1, 0, bias=False), nn.BatchNorm3d(b * 8), nn.ReLU(True),
            nn.ConvTranspose3d(b * 8, b * 4, 4, 2, 1, bias=False), nn.BatchNorm3d(b * 4), nn.ReLU(True),
            nn.ConvTranspose3d(b * 4, b * 2, 4, 2, 1, bias=False), nn.BatchNorm3d(b * 2), nn.ReLU(True),
            nn.ConvTranspose3d(b * 2, b, 4, 2, 1, bias=False), nn.BatchNorm3d(b), nn.ReLU(True),
            nn.ConvTranspose3d(b, 1, 4, 2, 1), nn.Tanh())

    def forward(self, z):
        assert z.shape[1:] == (self.ld, 1, 1, 1), f"z must be (B,{self.ld},1,1,1), got {tuple(z.shape)}"
        out = self.net(z)
        assert out.shape[2:] == (self.patch,) * 3, f"generator emitted {tuple(out.shape)}"
        return out


class WGAN3DGenerator(DCGAN3DGenerator):  # same architecture, different loss/optimiser
    pass


class _Disc3D(nn.Module):
    """Strided conv stack -> patch map -> mean = one score per sample (B,). No Sigmoid: scores are logits."""

    def __init__(self, norm, base=WIDTH):
        super().__init__()
        b = base

        def block(i, o):
            if norm == "bn":
                return [nn.Conv3d(i, o, 4, 2, 1, bias=False), nn.BatchNorm3d(o), nn.LeakyReLU(0.2, True)]
            return [nn.Conv3d(i, o, 4, 2, 1), nn.GroupNorm(1, o), nn.LeakyReLU(0.2, True)]  # per-sample norm (WGAN-GP)

        self.net = nn.Sequential(nn.Conv3d(1, b, 4, 2, 1), nn.LeakyReLU(0.2, True),
                                 *block(b, b * 2), *block(b * 2, b * 4), *block(b * 4, b * 8),
                                 nn.Conv3d(b * 8, 1, 3, 1, 1, bias=norm == "bn"))  # critic: output bias cancels in the Wasserstein loss

    def forward(self, x):
        return self.net(x).mean(dim=(1, 2, 3, 4))


class DCGAN3DDiscriminator(_Disc3D):
    def __init__(self, base=WIDTH):
        super().__init__("bn", base)


class WGAN3DCritic(_Disc3D):
    def __init__(self, base=WIDTH):
        super().__init__("gn", base)


class Aggregator3DDiscriminator(_Disc3D):
    def __init__(self, base=WIDTH):
        super().__init__("bn", base)


class Aggregator3D(nn.Module):
    """Fuses three generator outputs (B,1,P,P,P each) into one; residual on the channel mean so it starts near averaging."""

    def __init__(self, base=8):
        super().__init__()
        b = base
        self.net = nn.Sequential(
            nn.Conv3d(3, b, 3, 1, 1), nn.LeakyReLU(0.2, True),
            nn.Conv3d(b, b, 3, 1, 1), nn.LeakyReLU(0.2, True),
            nn.Conv3d(b, b, 3, 1, 1), nn.LeakyReLU(0.2, True),
            nn.Conv3d(b, 1, 3, 1, 1))

    def forward(self, a, b, c):
        x = torch.cat([a, b, c], 1)
        return torch.tanh(self.net(x) + x.mean(1, keepdim=True))


class MaskHead3D(nn.Module):
    """Supervised image -> tumour-mask logits (replaces StyleTransfer3D). Runs at half resolution with dilated
    convs (large receptive field, 8x cheaper than full resolution) and upsamples back to the input grid."""

    def __init__(self, base=16):
        super().__init__()
        b = base

        def blk(i, o, s=1, d=1):
            return [nn.Conv3d(i, o, 3, s, d, dilation=d), nn.GroupNorm(4, o), nn.ReLU(True)]

        self.enc = nn.Sequential(*blk(1, b, s=2), *blk(b, b, d=2), *blk(b, b, d=4), *blk(b, b))
        self.out = nn.Conv3d(b, 1, 3, 1, 1)

    def forward(self, x):
        h = F.interpolate(self.enc(x), size=x.shape[2:], mode="trilinear", align_corners=False)
        return self.out(h)


def gradient_penalty(critic, real, fake, lam=GP_LAMBDA):
    """WGAN-GP on per-sample scalar scores; with a GroupNorm critic samples do not interact."""
    a = torch.rand(real.size(0), 1, 1, 1, 1, device=real.device)
    x = (a * real + (1 - a) * fake).requires_grad_(True)
    score = critic(x)
    assert score.shape == (real.size(0),), f"critic must return (B,), got {tuple(score.shape)}"
    g, = torch.autograd.grad(score.sum(), x, create_graph=True)
    return lam * ((g.flatten(1).norm(2, dim=1) - 1) ** 2).mean()


def dice_loss(logits, target, eps=1.0):
    p = torch.sigmoid(logits)
    inter = (p * target).sum(dim=(1, 2, 3, 4))
    return (1 - (2 * inter + eps) / (p.sum(dim=(1, 2, 3, 4)) + target.sum(dim=(1, 2, 3, 4)) + eps)).mean()


class MultiGAN:
    """Two DCGANs, one WGAN-GP, an Aggregator (+ its discriminator) and the supervised mask head."""

    def __init__(self, device, ld=LATENT_DIM, patch=PATCH, width=WIDTH, pos_weight=POS_WEIGHT):
        self.device, self.ld, self.patch, self.width = device, ld, patch, width
        self.dc1_g, self.dc1_d = DCGAN3DGenerator(ld, patch, width), DCGAN3DDiscriminator(width)
        self.dc2_g, self.dc2_d = DCGAN3DGenerator(ld, patch, width), DCGAN3DDiscriminator(width)
        self.w_g, self.w_c = WGAN3DGenerator(ld, patch, width), WGAN3DCritic(width)
        self.aggr, self.ag_d = Aggregator3D(), Aggregator3DDiscriminator(width)
        self.mask = MaskHead3D()
        for m in (self.dc1_g, self.dc1_d, self.dc2_g, self.dc2_d, self.w_g, self.w_c, self.ag_d):
            m.apply(weights_init)
        for m in self.modules().values():
            m.to(device)
        dc = dict(lr=2e-4, betas=(0.5, 0.999))
        wg = dict(lr=1e-4, betas=(0.0, 0.9))
        self.opt = {
            "dc1_g": optim.Adam(self.dc1_g.parameters(), **dc), "dc1_d": optim.Adam(self.dc1_d.parameters(), **dc),
            "dc2_g": optim.Adam(self.dc2_g.parameters(), **dc), "dc2_d": optim.Adam(self.dc2_d.parameters(), **dc),
            "w_g": optim.Adam(self.w_g.parameters(), **wg), "w_c": optim.Adam(self.w_c.parameters(), **wg),
            "aggr": optim.Adam(self.aggr.parameters(), **dc), "ag_d": optim.Adam(self.ag_d.parameters(), **dc),
            "mask": optim.Adam(self.mask.parameters(), lr=1e-3)}
        self.pos_weight = torch.tensor(pos_weight, device=device)
        self.w_iter = 0
        self.sync = True  # True: step_* return python floats (.item(), one host sync each); False: detached device tensors (train loop)
        self.check_range = True  # per-batch [-1,1] assertion needs a host sync; train() validates the whole set once and turns it off

    def _v(self, t):
        return t.item() if self.sync else t.detach()

    def modules(self):
        return {"dc1_g": self.dc1_g, "dc1_d": self.dc1_d, "dc2_g": self.dc2_g, "dc2_d": self.dc2_d, "w_g": self.w_g,
                "w_c": self.w_c, "aggr": self.aggr, "ag_d": self.ag_d, "mask": self.mask}

    def train(self, mode=True):
        for m in self.modules().values():
            m.train(mode)

    def z(self, n):
        return torch.randn(n, self.ld, 1, 1, 1, device=self.device)

    @staticmethod
    def _bce(logits, target):
        return F.binary_cross_entropy_with_logits(logits, torch.full_like(logits, target))

    @staticmethod
    def _step(opt, loss):
        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()

    def step_dcgan(self, which, real):
        g, d = (self.dc1_g, self.dc1_d) if which == 1 else (self.dc2_g, self.dc2_d)
        og, od = self.opt[f"dc{which}_g"], self.opt[f"dc{which}_d"]
        fake = g(self.z(real.size(0)))
        ld = self._bce(d(real), REAL_LABEL) + self._bce(d(fake.detach()), 0.0)
        self._step(od, ld)
        lg = self._bce(d(fake), 1.0)
        self._step(og, lg)
        return {f"d{which}": self._v(ld), f"g{which}": self._v(lg)}

    def step_wgan(self, real, update_g):
        fake = self.w_g(self.z(real.size(0))).detach()
        gp = gradient_penalty(self.w_c, real, fake)
        lc = self.w_c(fake).mean() - self.w_c(real).mean() + gp
        self._step(self.opt["w_c"], lc)
        out = {"w_critic": self._v(lc), "w_gp": self._v(gp)}
        if update_g:  # generator moves once every N_CRITIC critic steps
            lg = -self.w_c(self.w_g(self.z(real.size(0)))).mean()
            self._step(self.opt["w_g"], lg)
            out["w_g"] = self._v(lg)
        return out

    def step_aggregator(self, real):
        with torch.no_grad():  # generators detached; one shared z so the three outputs are views of one latent
            z = self.z(real.size(0))
            fakes = [self.dc1_g(z), self.dc2_g(z), self.w_g(z)]
        fused = self.aggr(*fakes)
        ld = self._bce(self.ag_d(real), REAL_LABEL) + self._bce(self.ag_d(fused.detach()), 0.0)
        self._step(self.opt["ag_d"], ld)
        lg = self._bce(self.ag_d(fused), 1.0)
        l1 = (fused - torch.stack(fakes).mean(0)).abs().mean()  # without it the discriminator wins and the fusion drives voxels to +-1
        self._step(self.opt["aggr"], lg + AGG_L1 * l1)
        return {"ag_d": self._v(ld), "ag_g": self._v(lg), "ag_l1": self._v(l1)}

    def step_mask(self, real, mask):
        logits = self.mask(real)
        bce = F.binary_cross_entropy_with_logits(logits, mask, pos_weight=self.pos_weight)
        loss = bce + dice_loss(logits, mask)
        self._step(self.opt["mask"], loss)
        return {"mask": self._v(loss)}

    def train_batch(self, real, mask, mask_pool=None):
        """mask_pool() -> (images, masks) for the mask head (tumour + healthy negatives); default reuses (real, mask)."""
        if self.check_range:
            assert_unit_range(real, "real batch")
        self.w_iter += 1
        out = {}
        out.update(self.step_dcgan(1, real))
        out.update(self.step_dcgan(2, real))
        out.update(self.step_wgan(real, self.w_iter % N_CRITIC == 0))
        out.update(self.step_aggregator(real))
        for _ in range(MASK_STEPS):
            mx, mm = mask_pool() if mask_pool else (real, mask)
            out.update(self.step_mask(mx, mm))
        return out

    @torch.no_grad()
    def sample_all(self, n):
        self.train(False)
        z = self.z(n)
        out = {"dc1": self.dc1_g(z), "dc2": self.dc2_g(z), "wgan": self.w_g(z)}
        out["aggregator"] = self.aggr(out["dc1"], out["dc2"], out["wgan"])
        self.train(True)
        return out

    @torch.no_grad()
    def sample(self, n, fusion=FUSION):
        assert fusion in ("aggregator", "mixture"), fusion
        self.train(False)
        if fusion == "aggregator":
            z = self.z(n)
            out = self.aggr(self.dc1_g(z), self.dc2_g(z), self.w_g(z))
        else:  # one generator per output, no aggregator
            outs = torch.stack([g(self.z(n)) for g in (self.dc1_g, self.dc2_g, self.w_g)])
            pick = torch.randint(0, 3, (n,), device=self.device)
            out = outs[pick, torch.arange(n, device=self.device)]
        self.train(True)
        return out

    def save(self, path, epoch=None, elapsed=0.0, with_opt=False):
        """Atomic (temp file + replace). with_opt adds optimiser states, epoch and elapsed seconds for LITS_GAN_RESUME."""
        os.makedirs(os.path.dirname(path), exist_ok=True)
        ck = {"cfg": dict(ld=self.ld, patch=self.patch, width=self.width),
              "modules": {k: m.state_dict() for k, m in self.modules().items()}}
        if with_opt:
            ck.update(epoch=epoch, elapsed=elapsed, w_iter=self.w_iter, opt={k: o.state_dict() for k, o in self.opt.items()})
        torch.save(ck, path + ".tmp")
        os.replace(path + ".tmp", path)

    @classmethod
    def load(cls, path, device):
        ck = torch.load(path, map_location=device, weights_only=True)
        net = cls(device, **ck["cfg"])
        for k, m in net.modules().items():
            m.load_state_dict(ck["modules"][k])
        return net

    def load_training_state(self, path):
        """Restore modules + optimisers from a with_opt checkpoint; returns (epoch, elapsed seconds), or (0, 0.0) if it has no optimiser state."""
        ck = torch.load(path, map_location=self.device, weights_only=True)
        assert ck["cfg"] == dict(ld=self.ld, patch=self.patch, width=self.width), f"checkpoint config {ck['cfg']} != this run"
        for k, m in self.modules().items():
            m.load_state_dict(ck["modules"][k])
        if "opt" not in ck:
            return 0, 0.0
        for k, o in self.opt.items():
            o.load_state_dict(ck["opt"][k])
        self.w_iter = int(ck.get("w_iter", 0))
        return int(ck["epoch"]), float(ck.get("elapsed", 0.0))


# -----------------------------
# Visualisation (files only, never plt.show)
# -----------------------------
def save_montage(path, rows):
    """rows: [(label, patches (n,P,P,P) in [-1,1], masks (n,P,P,P) or None)]; shows the axial slice with most mask."""
    n = max(len(a) for _, a, _ in rows)
    fig, axs = plt.subplots(len(rows), n, figsize=(1.8 * n, 1.9 * len(rows)), squeeze=False)
    for r, (label, arr, msk) in enumerate(rows):
        for j in range(n):
            ax = axs[r][j]
            ax.axis("off")
            if j >= len(arr):
                continue
            z = int(np.argmax(msk[j].sum(axis=(0, 1)))) if msk is not None and msk[j].any() else arr.shape[-1] // 2
            ax.imshow(arr[j][:, :, z].T, cmap="gray", vmin=-1, vmax=1, origin="lower")
            if msk is not None and msk[j][:, :, z].any():
                ax.contour(msk[j][:, :, z].T.astype(float), levels=[0.5], colors="r", linewidths=0.8)
            if j == 0:
                ax.set_title(label, fontsize=8, loc="left")
    plt.tight_layout()
    os.makedirs(os.path.dirname(path), exist_ok=True)
    fig.savefig(path, dpi=110)
    plt.close(fig)


def save_insert_montage(path, cases):
    """cases: [(cid, host_hu, aug_hu, seg_old, seg_new, info, spacing)]: host | augmented | augmented + labels."""
    fig, axs = plt.subplots(len(cases), 3, figsize=(9, 3.1 * len(cases)), squeeze=False)
    for r, (cid, host, aug, s0, s1, info, sp) in enumerate(cases):
        w = info["window"]
        z = int(np.clip(info["centre"][2], 0, host.shape[2] - 1))
        x = slice(max(0, w[0].start - 12), min(host.shape[0], w[0].stop + 12))
        y = slice(max(0, w[1].start - 12), min(host.shape[1], w[1].stop + 12))
        for c, (img, title) in enumerate([(host, "host"), (aug, "augmented"), (aug, "augmented + labels")]):
            ax = axs[r][c]
            ax.imshow(img[x, y, z].T, cmap="gray", vmin=WIN_LO, vmax=WIN_HI, origin="lower", aspect=sp[1] / sp[0])
            if c == 2:
                o, n = s0[x, y, z] == 2, s1[x, y, z] == 2
                for t, col in ((o, "lime"), (n & ~o, "red")):  # existing tumours green, inserted voxels red
                    t = t.T.astype(float)
                    if t.any():
                        ax.contour(t, levels=[0.5], colors=col, linewidths=0.9)
            ax.set_title(f"case {cid} z={z}: {title}", fontsize=8)
            ax.axis("off")
    plt.tight_layout()
    os.makedirs(os.path.dirname(path), exist_ok=True)
    fig.savefig(path, dpi=110)
    plt.close(fig)


# -----------------------------
# Training
# -----------------------------
@torch.no_grad()
def eval_mask_head(model, imgs, masks, healthy, n=32, min_vox=MIN_MASK_VOX):
    """Dice on tumour patches and false-positive behaviour on tumour-free (healthy) patches."""
    model.train(False)
    dev = model.device

    def pred(a):  # chunks of 4 keep the 3D activations small
        return np.concatenate([torch.sigmoid(model.mask(torch.from_numpy(a[i:i + 4].astype(np.float32))[:, None].to(dev)))[:, 0].cpu().numpy() > 0.5
                               for i in range(0, len(a), 4)])

    out = {}
    if len(imgs):
        p, t = pred(imgs[:n]), masks[:n] > 0
        out["dice"] = float(np.mean([2 * (a & b).sum() / max(a.sum() + b.sum(), 1) for a, b in zip(p, t)]))
    if len(healthy):
        h = pred(healthy[:n])
        out["healthy_fp_vox_frac"] = float(h.mean())
        out["healthy_fp_patches"] = float(np.mean([x.sum() >= min_vox for x in h]))
    model.train(True)
    return out


def split_holdout(cids, rng, frac=HOLDOUT):
    """Whole volumes held out of mask-head training (patches of one volume are strongly correlated)."""
    cids = np.unique(cids)
    if len(cids) < 2:
        return set()
    return set(rng.choice(cids, min(len(cids) - 1, max(1, int(round(frac * len(cids))))), replace=False).tolist())


class PatchStore:
    """Patch arrays for the training loop. Up to limit_mb they live on the device (fp16 images, uint8 masks): a batch is a
    device-side gather with no host transfer and no sync. Larger sets stay in host memory (pinned on cuda) and each batch is
    copied with non_blocking=True."""

    def __init__(self, arrays, device, limit_mb=DATA_MB):
        self.device = device
        self.t = {k: torch.from_numpy(np.ascontiguousarray(a)) for k, a in arrays.items()}
        mb = sum(t.numel() * t.element_size() for t in self.t.values()) / 2 ** 20
        self.on_device = device.type != "cpu" and mb <= limit_mb
        if self.on_device:
            self.t = {k: t.to(device) for k, t in self.t.items()}
        elif device.type == "cuda":
            self.t = {k: t.pin_memory() for k, t in self.t.items()}
        self.mb = mb

    def gather(self, key, idx):
        """idx: int64 tensor (on the device when on_device, else on the host). Returns float32 (B,1,P,P,P) on the device."""
        x = self.t[key][idx]
        if not self.on_device and self.device.type != "cpu":
            x = x.pin_memory().to(self.device, non_blocking=True) if self.device.type == "cuda" else x.to(self.device)
        return x.float()[:, None]


class MaskPool:
    """mask_pool() callable for MultiGAN.train_batch: half tumour, half healthy (all-zero mask) samples of the non-held-out
    volumes. Index tables are drawn on the host once per refill and moved to the device in one copy (no per-step transfer)."""

    def __init__(self, store, t_idx, h_idx, bs, rng, device):
        self.store, self.t_idx, self.h_idx, self.bs, self.rng, self.device = store, t_idx, h_idx, bs, rng, device
        self.nh = bs // 2 if len(h_idx) else 0
        self.c = self.n = 0

    def refill(self, n):
        ti = self.rng.choice(self.t_idx, (n, self.bs - self.nh))
        hi = self.rng.choice(self.h_idx, (n, self.nh)) if self.nh else np.zeros((n, 0), int)
        mv = lambda a: torch.as_tensor(a, dtype=torch.long).to(self.device if self.store.on_device else "cpu")
        self.ti, self.hi, self.c, self.n = mv(ti), mv(hi), 0, n

    def __call__(self):
        if self.c >= self.n:
            self.refill(max(self.n, 64))
        ti, hi = self.ti[self.c], self.hi[self.c]
        self.c += 1
        x = self.store.gather("images", ti)
        m = self.store.gather("masks", ti)
        if self.nh:
            x = torch.cat([x, self.store.gather("healthy", hi)])
            m = torch.cat([m, torch.zeros((self.nh,) + tuple(m.shape[1:]), dtype=m.dtype, device=m.device)])
        return x, m


def train(patches_path, epochs=EPOCHS, device=None, results_dir=RESULTS_DIR, resume=None, max_seconds=None):
    """Train the GANs + mask head. resume (default LITS_GAN_RESUME): continue from checkpoints/gan_last.pt (optimisers, epoch,
    elapsed time); a finished run (epoch >= epochs) returns at once. max_seconds (default LITS_GAN_MAX_SECONDS): stop after the
    epoch that crosses the budget. Writes checkpoints/gan_last.pt after every epoch and train_metrics.json at every evaluation."""
    device = device or pick_device()
    resume = RESUME if resume is None else resume
    max_seconds = MAX_SECONDS if max_seconds is None else max_seconds
    setup_backends(device)
    d = load_npz(patches_path)
    imgs, masks, healthy = d["images"], d["masks"], d["healthy"]
    t_cid = d["meta"][:, 0]
    h_cid = d["healthy_cid"] if "healthy_cid" in d.files else np.full(len(healthy), -1)
    work = float(d["work_mm"]) if "work_mm" in d.files else 1.0
    patch = imgs.shape[-1]
    assert_unit_range(imgs, "training patches")  # whole set once: the per-batch check (a host sync) is off below
    n, bs = len(imgs), min(BATCH_SIZE, len(imgs))
    rng = np.random.default_rng(SEED)
    # mask head: trained on tumour patches + healthy patches (all-zero masks) of the non-held-out volumes, evaluated on the held-out ones
    held = split_holdout(np.concatenate([t_cid, h_cid]), rng)
    t_tr, h_tr = ~np.isin(t_cid, list(held)), ~np.isin(h_cid, list(held))
    t_tr_i, h_tr_i = np.flatnonzero(t_tr), np.flatnonzero(h_tr)
    assert len(t_tr_i), "no tumour patches left for mask-head training"
    model = MultiGAN(device, patch=patch)
    model.check_range = False
    min_mask_vox = vox_at(MIN_MASK_VOX, work)
    ckpt = os.path.join(results_dir, "checkpoints", "gan_last.pt")
    start, elapsed0 = 0, 0.0
    if resume and os.path.exists(ckpt):
        start, elapsed0 = model.load_training_state(ckpt)
        rng = np.random.default_rng([SEED, start])  # the held-out split above is the same as in the first run (same seed, same data)
        logger.info(f"resumed from {ckpt}: epoch {start}, {elapsed0:.0f}s of training already spent")
    store = PatchStore({"images": imgs, "masks": masks, "healthy": healthy}, device)
    logger.info(f"device {device}; {n} tumour / {len(healthy)} healthy patches {patch}^3 at {work} mm; batch {bs}; epochs {epochs}; "
                f"patch set {store.mb:.0f} MB {'on the device' if store.on_device else 'in host memory'}")
    logger.info(f"mask head: train {len(t_tr_i)} tumour + {len(h_tr_i)} healthy patches; held-out volumes {sorted(held)} "
                f"({int((~t_tr).sum())} tumour + {int((~h_tr).sum())} healthy patches)")
    if not held:
        logger.warning("only one volume: no held-out set, mask-head 'heldout_*' metrics are not available")
    pool = MaskPool(store, t_tr_i, h_tr_i, bs, rng, device)
    metrics_path = os.path.join(results_dir, "train_metrics.json")
    metrics = read_json(metrics_path) if resume and os.path.exists(metrics_path) else {}
    metrics.update(patches=int(n), healthy_patches=int(len(healthy)), patch=int(patch), work_mm=work, epochs_target=int(epochs),
                   held_out_volumes=[int(v) for v in sorted(held)], device=str(device))
    steps = n // bs  # drop the partial last batch (BatchNorm)
    t0, ep, stopped = time.time(), start, "max" if start >= epochs else None
    model.sync = False  # losses stay on the device until the epoch end
    for ep in range(start + 1, epochs + 1):
        perm = torch.as_tensor(rng.permutation(n)[:steps * bs], dtype=torch.long).to(device if store.on_device else "cpu")
        pool.refill(steps * MASK_STEPS)
        sums, cnts = {}, {}
        for i in range(steps):
            idx = perm[i * bs:(i + 1) * bs]
            real, msk = store.gather("images", idx), store.gather("masks", idx)
            for k, v in model.train_batch(real, msk, pool).items():
                sums[k], cnts[k] = sums.get(k, 0.0) + v, cnts.get(k, 0) + 1  # w_g is only produced every N_CRITIC-th batch
        means = {k: float(v) / cnts[k] for k, v in sums.items()}  # the one host sync of the epoch
        msg = " ".join(f"{k}={v:.3f}" for k, v in means.items())
        assert all(np.isfinite(v) for v in means.values()), "non-finite loss"
        spent = elapsed0 + time.time() - t0
        logger.info(f"epoch {ep}/{epochs} ({spent:.0f}s) {msg}")
        if ep % SAMPLE_EVERY == 0 or ep == epochs:
            model.sync = True
            for tag, ev in (("train", eval_mask_head(model, imgs[t_tr_i], masks[t_tr_i], healthy[h_tr_i], min_vox=min_mask_vox)),
                            ("heldout", eval_mask_head(model, imgs[~t_tr], masks[~t_tr], healthy[~h_tr], n=64, min_vox=min_mask_vox) if held else {})):
                if ev:
                    logger.info(f"mask head {tag}: " + " ".join(f"{k}={v:.3f}" for k, v in ev.items()))
                    metrics[f"mask_head_{tag}"] = dict(ev, epoch=ep)
            model.sync = False
            s = {k: v[:, 0].cpu().numpy() for k, v in model.sample_all(4).items()}
            with torch.no_grad():
                model.train(False)
                pm = (torch.sigmoid(model.mask(torch.from_numpy(s["aggregator"])[:, None].to(device)))[:, 0] > 0.5).cpu().numpy()
                model.train(True)
            save_montage(os.path.join(results_dir, "figures", f"train_epoch{ep:03d}.png"),
                         [("real", imgs[:4].astype(np.float32), masks[:4] > 0), ("dcgan 1", s["dc1"], None),
                          ("dcgan 2", s["dc2"], None), ("wgan-gp", s["wgan"], None),
                          ("aggregator + mask head", s["aggregator"], pm)])
        out_of_time = bool(max_seconds) and spent >= max_seconds and ep < epochs
        if ep % max(CKPT_EVERY, 1) == 0 or ep == epochs or out_of_time:  # resume loses at most CKPT_EVERY-1 epochs
            model.save(ckpt, epoch=ep, elapsed=spent, with_opt=True)
        history = [h for h in metrics.get("loss_history", []) if h["epoch"] < ep]  # resume: drop epochs that are re-run
        metrics.update(epochs_done=ep, train_seconds=spent, last_losses=means, loss_history=history + [dict(means, epoch=ep, seconds=spent)])
        write_json_atomic(metrics_path, metrics)
        if out_of_time:
            logger.warning(f"GAN time budget {max_seconds:.0f}s reached after epoch {ep}/{epochs}: stopping early")
            stopped = "time"
            break
    model.sync = True
    metrics.update(epochs_done=ep, stopped=stopped or "max")
    write_json_atomic(metrics_path, metrics)
    return model


# -----------------------------
# Generation
# -----------------------------
def clean_mask(prob, min_vox=MIN_MASK_VOX, edge=EDGE_MARGIN):
    """Threshold 0.5, keep the largest component. Returns (mask or None, reject reason)."""
    lab, n = ndi.label(prob > 0.5, structure=np.ones((3, 3, 3)))
    if n == 0:
        return None, "empty"
    m = lab == (np.argmax(np.bincount(lab.ravel())[1:]) + 1)
    if m.sum() < min_vox:
        return None, "too_small"
    if any(m.take(range(edge), axis=a).any() or m.take(range(m.shape[a] - edge, m.shape[a]), axis=a).any() for a in range(3)):
        return None, "touches_border"
    return m, ""


def generate_samples(model, num=NUM_SAMPLES, fusion=FUSION, max_attempts=None, work_mm=1.0, stats=None):
    """Accept-until-target: sample, label with the mask head, gate, until `num` samples pass or max_attempts candidates were tried.
    min mask size is defined at 1 mm voxels (MIN_MASK_VOX) and converted to work_mm. stats (dict) receives tried/accepted/rejects."""
    max_attempts = max_attempts or num * 50
    min_vox = vox_at(MIN_MASK_VOX, work_mm)
    patches, masks, rejects, tried = [], [], {}, 0
    while len(patches) < num and tried < max_attempts:
        x = model.sample(min(8, num), fusion)
        with torch.no_grad():
            model.train(False)
            prob = torch.sigmoid(model.mask(x))[:, 0].cpu().numpy()
            model.train(True)
        for xi, pi in zip(x[:, 0].cpu().numpy(), prob):
            tried += 1
            m, why = clean_mask(pi, min_vox=min_vox)
            if m is None:
                rejects[why] = rejects.get(why, 0) + 1
            elif len(patches) < num:
                assert_unit_range(xi, "generated patch")
                patches.append(xi)
                masks.append(m)
    logger.info(f"generated {len(patches)}/{num} samples ({fusion}) in {tried} attempts; rejects {rejects}")
    if stats is not None:
        stats.update(tried=tried, accepted=len(patches), rejects=rejects)
    if not patches:
        e = (0,) + (model.patch,) * 3
        return np.zeros(e, np.float32), np.zeros(e, np.uint8)
    return np.stack(patches).astype(np.float32), np.stack(masks).astype(np.uint8)


# -----------------------------
# Host-aware insertion
# -----------------------------
def resample_patch_to_host(patch, mask, spacing, work_mm=1.0):
    """work_mm generated patch + mask -> host voxel spacing. Exact target shape with grid_mode (no corner-alignment stretch),
    box low-pass before downsampling; the mask is thresholded so that its physical volume is preserved
    (a plain order-0 zoom of a thin mask changes its volume by up to 50 % on 5 mm slices). Empty mask = vanished."""
    sp = np.asarray(spacing, dtype=np.float64)
    r = sp / work_mm  # host voxel size in generator voxels (== sp at 1 mm)
    shape = np.maximum(np.rint(np.asarray(patch.shape) / r).astype(int), 1)
    zoom = shape / np.asarray(patch.shape)

    def down(a):
        for ax, s_ in enumerate(r):
            if s_ > 1.0:  # anti-alias: average over the host voxel footprint
                a = ndi.uniform_filter1d(a, size=int(s_) | 1, axis=ax, mode="nearest")
        return ndi.zoom(a, zoom, order=1, mode="nearest", grid_mode=True)

    img = down(patch.astype(np.float32))
    f = down(mask.astype(np.float32))
    k = int(round(mask.sum() * work_mm ** 3 / np.prod(sp)))  # voxels the mask should occupy on the host grid
    if k < 1:
        return img, np.zeros(shape, bool)
    c = np.asarray(ndi.center_of_mass(f))
    dist = np.sqrt(sum(((np.arange(n) - ci) * s_).reshape([-1 if a == ax else 1 for a in range(3)]) ** 2
                       for ax, (n, ci, s_) in enumerate(zip(shape, c, sp))))
    score = f - 1e-5 * dist / max(float(dist.max()), 1e-6)  # exactly k voxels: ties go to the voxels nearest the centroid
    thr = float(np.partition(score.ravel(), -k)[-k])
    return img, (score >= thr) & (f > 1e-3)


def border_taper(shape, spacing, taper_mm=4.0):
    """0 on the window border rising to 1 over taper_mm, so alpha vanishes at the window edge."""
    w = np.ones(shape, np.float32)
    for ax, (n, sp) in enumerate(zip(shape, spacing)):
        d = np.minimum(np.arange(n), np.arange(n)[::-1]) * sp
        s = [1, 1, 1]
        s[ax] = n
        w = w * np.clip(d / taper_mm, 0, 1).reshape(s)
    return w


def liver_noise_std(hu, seg, spacing, sl):
    """Robust std of the high-passed HU over eroded liver voxels (CT noise level); None if too few voxels."""
    x = hu[sl].astype(np.float32)
    lm = ndi.binary_erosion(seg[sl] == 1, iterations=2)
    if lm.sum() < 200:
        return None
    hp = (x - ndi.gaussian_filter(x, NOISE_SIGMA_MM / np.asarray(spacing), mode="nearest"))[lm]
    return float(1.4826 * np.median(np.abs(hp - np.median(hp))))


def _reject(why, key, msg):
    logger.info(f"insert skipped: {msg}")
    if why is not None:
        why[key] = why.get(key, 0) + 1


def insert_one(hu, seg, spacing, patch, mask, rng, feather_mm=FEATHER_MM, exclude_mm=EXCLUDE_MM, work_mm=1.0, why=None):
    """Insert one generated tumour into int16 (or float32) HU `hu` / uint8 `seg` IN PLACE. Returns an info dict, or None (reason
    logged, and counted in the `why` dict when given). Only voxels with alpha > 0 inside a local window change; the label is
    (mask & liver) written as 2. A float32 `hu` keeps the blended values unrounded (cache source: written back to uint8 once)."""
    sp = np.asarray(spacing, dtype=np.float64)
    patch_rs, m_rs = resample_patch_to_host(patch, mask, spacing, work_mm)
    if not m_rs.any():
        _reject(why, "mask_vanished", "mask vanished on resampling to host spacing")
        return None
    ps = np.array(patch_rs.shape)
    coords = np.argwhere(mask)
    r_mm = float(np.linalg.norm((coords - coords.mean(0)) * work_mm, axis=1).max()) + work_mm  # circumscribed radius (mm)
    cen = np.rint(np.argwhere(m_rs).mean(0)).astype(int)  # mask centroid inside the resampled window
    # candidate centres: liver eroded by the tumour radius, away from existing tumours, window fully inside the volume
    box = ndi.find_objects((seg > 0).astype(np.int8))
    if not box:
        _reject(why, "host_no_labels", "host has no labels")
        return None
    box = box[0]
    off = np.array([s.start for s in box])
    liver = np.pad(seg[box] == 1, 1)
    ok = ndi.distance_transform_edt(liver, sampling=sp)[1:-1, 1:-1, 1:-1] >= r_mm
    tum = seg[box] == 2
    if tum.any():
        ok &= ndi.distance_transform_edt(~tum, sampling=sp) >= r_mm + exclude_mm
    lo, hi = cen - off, np.array(seg.shape) - ps + cen - off  # valid centre range in crop coordinates (inclusive)
    inside = np.zeros_like(ok)
    sub = tuple(slice(max(l, 0), min(h + 1, n)) for l, h, n in zip(lo, hi, ok.shape))
    inside[sub] = ok[sub]
    cand = np.argwhere(inside)
    if len(cand) == 0:
        _reject(why, "no_site", f"no valid site (tumour radius {r_mm:.1f} mm)")
        return None
    p = cand[rng.integers(len(cand))] + off
    start = p - cen
    win = tuple(slice(int(s), int(s + n)) for s, n in zip(start, ps))
    hw, sw = hu[win].astype(np.float32), seg[win]  # sw is a view: labels are written through it
    liver_w = sw == 1
    m_eff = m_rs & liver_w
    if not m_eff.any():
        _reject(why, "no_liver_voxels", "mask has no liver voxels")
        return None
    # feathered alpha: gaussian of the dilated mask on the host grid, 1 inside the mask, 0 on the window border
    dil = ndi.distance_transform_edt(~m_eff, sampling=sp) <= 1.0  # 1 mm dilation (a voxel dilation would add a whole 5 mm slice)
    alpha = ndi.gaussian_filter(dil.astype(np.float32), feather_mm / sp, mode="constant")
    alpha = np.maximum(alpha, m_eff) * border_taper(tuple(ps), sp) * liver_w  # liver only: no background, no existing tumour
    alpha[alpha < 0.01] = 0.0
    support = alpha > 0
    # masked intensity transfer: shift so the generated non-tumour mean (ring around the tumour) matches the host liver
    gen = from_unit(patch_rs)
    d = ndi.distance_transform_edt(~m_rs, sampling=sp)
    ring = (d > 2.0) & (d <= 10.0)
    ring = ring if ring.sum() >= 20 else ~m_rs
    local = liver_w & ~support
    host_ref = float(hw[local].mean() if local.sum() >= 100 else hw[liver_w].mean())
    shift = host_ref - float(gen[ring].mean())
    gen = gen + shift
    # host-matched high-frequency noise inside alpha (variance-preserving mix of host and new noise)
    pad = np.ceil(24.0 / sp).astype(int)
    nsl = tuple(slice(max(0, w.start - q), min(n, w.stop + q)) for w, q, n in zip(win, pad, seg.shape))
    sig = liver_noise_std(hu, seg, spacing, nsl)
    if sig is None:
        sig = liver_noise_std(hu, seg, spacing, tuple(slice(s.start, s.stop) for s in box))
    sig = 0.0 if sig is None else sig
    sg = NOISE_SIGMA_MM / sp
    noise = ndi.gaussian_filter(rng.standard_normal(hw.shape).astype(np.float32), NOISE_PRE_MM / sp)  # CT noise is spatially correlated
    noise = noise - ndi.gaussian_filter(noise, sg)
    noise *= sig / max(float(noise.std()), 1e-6)
    host_lp = ndi.gaussian_filter(hw, sg, mode="nearest")
    out = ((1 - alpha) * host_lp + alpha * ndi.gaussian_filter(gen, sg, mode="nearest")
           + np.sqrt(1 - alpha) * (hw - host_lp) + np.sqrt(alpha) * noise)
    hu[win][support] = to_int16(out[support]) if np.issubdtype(hu.dtype, np.integer) else out[support].astype(hu.dtype)
    new = m_eff & liver_w
    sw[new] = 2
    return {"window": win, "support": support, "alpha": alpha, "mask": m_eff, "centre": p, "radius_mm": r_mm,
            "shift_hu": shift, "noise_std": sig, "n_label": int(new.sum()), "host_ref_hu": host_ref}


def insert_tumors(hu, seg, spacing, samples, rng, n=INSERTIONS, work_mm=1.0, why=None, keep_float=False):
    """Up to n non-overlapping insertions (each placement sees the labels of the previous ones).
    Returns (hu_int16, seg_uint8, infos); infos is empty when no site was valid. keep_float: return float32 HU, not rounded to int16."""
    out_hu = hu.astype(np.float32).copy() if keep_float else to_int16(hu)
    out_seg, infos = seg.astype(np.uint8).copy(), []
    patches, masks = samples
    for _ in range(n):
        k = int(rng.integers(len(patches)))
        info = insert_one(out_hu, out_seg, spacing, patches[k], masks[k], rng, work_mm=work_mm, why=why)
        if info is not None:
            infos.append(info)
    return out_hu, out_seg, infos


def run_insert(samples_path, ids, out_dir, fig_path, data_dir=DATA_DIR, n=INSERTIONS, rng=None):
    rng = rng or np.random.default_rng(SEED)
    assert not os.path.abspath(out_dir).startswith(os.path.abspath(data_dir)), "never write augmented files into the raw data dir"
    d = load_npz(samples_path)
    samples = (d["patches"], d["masks"])
    work = float(d["work_mm"]) if "work_mm" in d.files else 1.0
    if not len(samples[0]):
        logger.error("no valid generated samples (all rejected by the mask gate): nothing to insert")
        return 0
    cases, shown, done = pair_cases(data_dir, ids), [], 0
    os.makedirs(out_dir, exist_ok=True)
    for cid in ids:
        vol, seg, sp, aff = load_case(*cases[cid])
        hu, seg_new, infos = insert_tumors(vol, seg, sp, samples, rng, n, work_mm=work)
        if not infos:
            logger.warning(f"case {cid}: no tumour inserted (no valid site), not written")
            continue
        save_nifti(hu, aff, os.path.join(out_dir, f"aug_volume-{cid}.nii"), np.int16)
        save_nifti(seg_new, aff, os.path.join(out_dir, f"aug_segmentation-{cid}.nii"), np.uint8)
        logger.info(f"case {cid}: inserted {len(infos)}, +{sum(i['n_label'] for i in infos)} tumour voxels, "
                    f"shift {[round(i['shift_hu'], 1) for i in infos]} HU, noise std {[round(i['noise_std'], 1) for i in infos]}")
        done += 1
        if len(shown) < 4:
            shown.append((cid, to_int16(vol), hu, seg, seg_new, infos[0], sp))
    if shown:
        save_insert_montage(fig_path, shown)
    logger.info(f"augmented {done}/{len(ids)} hosts -> {out_dir}")
    return done


# -----------------------------
# Cache source: slabs of augmented cache slices
# -----------------------------
def accept_until_target(attempt, target, max_attempts):
    """Call attempt(i) (-> list of accepted items, empty = rejected) until `target` items are accepted or max_attempts calls were
    made. The list is cut at target. Returns (items, attempts used)."""
    out, tried = [], 0
    while len(out) < target and tried < max_attempts:
        out.extend(attempt(tried))
        tried += 1
    return out[:target], tried


def quantise_window(hu, lo=WIN_LO, hi=WIN_HI):
    """HU -> the cache's uint8 window mapping (rounded, clipped); inverse of CacheSource.to_hu up to the 1.37 HU step."""
    return np.clip(np.rint((hu - lo) / (hi - lo) * 255.0), 0, 255).astype(np.uint8)


def host_slabs(source, cid, samples, rng, per_host=1, work_mm=1.0, why=None):
    """Insert up to per_host tumours into ONE cache host and cut slabs: the z range of each insertion window (windows of this host
    that overlap in z are merged, so no slice appears twice). Slab images/masks are the cache slices of the AUGMENTED host: voxels
    that the insertion did not change keep the host's uint8 value bit for bit, only changed voxels are re-quantised.
    Returns a list of dicts (images (n,256,256) u8, masks, host_vol, z0, new_tumor_vox, n_slices, preview); [] if nothing fit."""
    img, msk = source.load_u8(cid)  # (Z,256,256)
    hu, seg = source.to_hu(np.moveaxis(img, 0, 2)), np.moveaxis(msk, 0, 2)  # (256,256,Z): the layout the NIfTI code works in
    aug_hu, seg_new, infos = insert_tumors(hu, seg, source.spacing(cid), samples, rng, per_host, work_mm, why, keep_float=True)
    spans = []
    for a, b in sorted((i["window"][2].start, i["window"][2].stop) for i in infos):
        if spans and a <= spans[-1][1]:
            spans[-1][1] = max(spans[-1][1], b)
        else:
            spans.append([a, b])
    out = []
    for z0, z1 in spans:
        a, h = aug_hu[:, :, z0:z1], hu[:, :, z0:z1]
        changed = a != h  # insert_one only assigns inside alpha support, everything else is a bit-identical copy of the host
        u8 = img[z0:z1].copy()
        np.moveaxis(u8, 0, 2)[changed] = quantise_window(a[changed], source.lo, source.hi)
        m_new = np.ascontiguousarray(np.moveaxis(seg_new[:, :, z0:z1], 2, 0))
        grown = (m_new == 2) & (msk[z0:z1] != 2)
        k = int(np.argmax(grown.sum(axis=(1, 2))))
        out.append(dict(images=u8, masks=m_new, host_vol=source.name(cid), z0=int(z0), n_slices=int(z1 - z0), new_tumor_vox=int(grown.sum()),
                        preview=(img[z0 + k].copy(), u8[k].copy(), msk[z0 + k].copy(), m_new[k].copy(), int(z0 + k))))
    return out


def pack_slabs(slabs, size=256):
    """list of host_slabs dicts -> the aug_slabs.npz arrays (schema in the module docstring); also valid for zero slabs."""
    if not slabs:
        return dict(images=np.zeros((0, size, size), np.uint8), masks=np.zeros((0, size, size), np.uint8), slab_id=np.zeros(0, np.int32),
                    host_vol=np.array([], dtype="<U16"), z0=np.zeros(0, np.int32), new_tumor_vox=np.zeros(0, np.int64), n_slices=np.zeros(0, np.int32))
    n = np.array([s["n_slices"] for s in slabs], np.int32)
    return dict(images=np.concatenate([s["images"] for s in slabs]), masks=np.concatenate([s["masks"] for s in slabs]),
                slab_id=np.repeat(np.arange(len(slabs), dtype=np.int32), n), host_vol=np.array([s["host_vol"] for s in slabs]),
                z0=np.array([s["z0"] for s in slabs], np.int32), new_tumor_vox=np.array([s["new_tumor_vox"] for s in slabs], np.int64), n_slices=n)


def save_slab_montage(path, previews, crop=48):
    """previews: [(title, host u8 slice, augmented u8 slice, old mask, new mask)]: host | augmented | augmented + labels
    (existing tumours green, inserted voxels red). Shown in the cache's own orientation, cropped around the new tumour."""
    fig, axs = plt.subplots(len(previews), 3, figsize=(8.4, 2.9 * len(previews)), squeeze=False)
    for r, (title, h, a, m0, m1) in enumerate(previews):
        grown = (m1 == 2) & (m0 != 2)
        cy, cx = (np.rint(np.argwhere(grown).mean(0)).astype(int) if grown.any() else (np.array(h.shape) // 2))
        ys, xs = slice(max(0, cy - crop), min(h.shape[0], cy + crop)), slice(max(0, cx - crop), min(h.shape[1], cx + crop))
        for c, (img, t) in enumerate([(h, "host (before)"), (a, "augmented (after)"), (a, "augmented + labels")]):
            ax = axs[r][c]
            ax.imshow(img[ys, xs], cmap="gray", vmin=0, vmax=255)
            if c == 2:
                for t_, col in (((m0 == 2)[ys, xs], "lime"), (grown[ys, xs], "red")):
                    if t_.any():
                        ax.contour(t_.astype(float), levels=[0.5], colors=col, linewidths=0.9)
            ax.set_title(f"{title}: {t}" if c == 0 else t, fontsize=8, loc="left")
            ax.axis("off")
    plt.tight_layout()
    os.makedirs(os.path.dirname(os.path.abspath(path)), exist_ok=True)
    fig.savefig(path, dpi=110)
    plt.close(fig)


def run_insert_slabs(source, hosts, samples, out_path, target=NUM_AUG, per_host=INSERTIONS, max_attempts=None, rng=None,
                     work_mm=1.0, fig_path=None):
    """Cache-source insert stage: visit hosts (shuffled, cycling) and insert until `target` slabs are accepted or `max_attempts`
    host visits are used; writes aug_slabs.npz atomically. Returns a stats dict (accepted, attempts, reasons, ...)."""
    rng = rng or np.random.default_rng(SEED)
    max_attempts = max_attempts or 5 * target
    hosts = [int(h) for h in hosts]
    assert hosts, "no insertion hosts"
    assert not set(hosts) - set(source.train_part_ids()), "hosts must come from the original-split train part"
    why, t0, state = {}, time.time(), {"order": []}
    if not len(samples[0]):
        logger.error("no valid generated samples (all rejected by the mask gate): nothing to insert")
        max_attempts = 0

    def attempt(i):
        if not state["order"]:
            state["order"] = [hosts[j] for j in rng.permutation(len(hosts))]  # a new shuffled pass over the hosts once all were visited
        cid = state["order"].pop()
        slabs = host_slabs(source, cid, samples, rng, per_host, work_mm, why)
        logger.info(f"attempt {i + 1}/{max_attempts} host {source.name(cid)}: " +
                    (f"{len(slabs)} slab(s), {[s['n_slices'] for s in slabs]} slices, +{sum(s['new_tumor_vox'] for s in slabs)} tumour voxels" if slabs
                     else "rejected"))
        return slabs

    slabs, tried = accept_until_target(attempt, target, max_attempts)
    save_npz_atomic(out_path, **pack_slabs(slabs))
    if fig_path and slabs:
        pick = np.linspace(0, len(slabs) - 1, min(6, len(slabs))).round().astype(int)
        save_slab_montage(fig_path, [(f"{slabs[k]['host_vol'][:-4]} z={slabs[k]['preview'][4]}",) + tuple(slabs[k]["preview"][:4]) for k in pick])
    st = dict(accepted=len(slabs), target=target, attempts=tried, max_attempts=max_attempts, insert_rejected=why, hosts=len(hosts),
              slab_slices=int(sum(s["n_slices"] for s in slabs)), new_tumor_vox=int(sum(s["new_tumor_vox"] for s in slabs)),
              hosts_used=sorted({s["host_vol"] for s in slabs}), seconds=time.time() - t0)
    if len(slabs) < target:
        logger.warning(f"only {len(slabs)}/{target} slabs accepted in {tried} host visits (reasons {why})")
    logger.info(f"wrote {len(slabs)} slabs ({st['slab_slices']} slices, +{st['new_tumor_vox']} tumour voxels) -> {out_path}")
    return st


def record_stage(results_dir, stage, **kv):
    """Merge per-stage numbers into stage_metrics.json (kept across resumes) and return the whole dict."""
    path = os.path.join(results_dir, "stage_metrics.json")
    cur = read_json(path) if os.path.exists(path) else {}
    cur.setdefault(stage, {}).update(kv)
    write_json_atomic(path, cur)
    return cur


def write_gan_summary(results_dir, extra=None):
    """gan_summary.json: {patches, epochs, accepted, rejected_reasons, mask_head_train_dice, mask_head_heldout_dice, volumes_used, seconds, ...}
    built from stage_metrics.json; rejected_reasons is flat: gate_<reason> (mask gate) and insert_<reason> (insertion)."""
    m = read_json(os.path.join(results_dir, "stage_metrics.json"))
    pre, tr, gen, ins = (m.get(k, {}) for k in ("prepare", "train", "generate", "insert"))
    rej = {f"gate_{k}": v for k, v in gen.get("rejects", {}).items()}
    rej.update({f"insert_{k}": v for k, v in ins.get("insert_rejected", {}).items()})
    by_stage = {k: round(v.get("seconds", 0.0), 1) for k, v in m.items() if isinstance(v, dict) and "seconds" in v}
    mh = lambda tag: (tr.get(f"mask_head_{tag}") or {}).get("dice")
    out = dict(source=GAN_SOURCE, work_mm=pre.get("work_mm"), patch=tr.get("patch", pre.get("patch")), patches=tr.get("patches", pre.get("patches")),
               healthy_patches=tr.get("healthy_patches", pre.get("healthy_patches")), epochs=tr.get("epochs_done"), epochs_target=tr.get("epochs_target"),
               train_stopped=tr.get("stopped"), accepted=ins.get("accepted", 0), target=ins.get("target"), attempts=ins.get("attempts"),
               rejected_reasons=rej, gate_tried=gen.get("tried"), gate_accepted=gen.get("accepted"),
               mask_head_train_dice=mh("train"), mask_head_heldout_dice=mh("heldout"),
               mask_head_train=tr.get("mask_head_train"), mask_head_heldout=tr.get("mask_head_heldout"),
               volumes_listed=pre.get("volumes_listed"), volumes_used=pre.get("volumes_used"), volumes_used_names=pre.get("volumes_used_names"),
               hosts=ins.get("hosts"), slab_slices=ins.get("slab_slices"), new_tumor_vox=ins.get("new_tumor_vox"),
               seconds=round(sum(by_stage.values()), 1), seconds_by_stage=by_stage)
    out.update(extra or {})
    write_json_atomic(os.path.join(results_dir, "gan_summary.json"), out)
    f = lambda v: "n/a" if v is None else f"{v:.3f}" if isinstance(v, float) else str(v)
    print(f"GAN_SUMMARY slabs {out['accepted']}/{f(out['target'])} (visits {f(out['attempts'])}) | patches {f(out['patches'])} from {f(out['volumes_used'])}/"
          f"{f(out['volumes_listed'])} volumes | epochs {f(out['epochs'])} | mask head dice train {f(out['mask_head_train_dice'])} held-out "
          f"{f(out['mask_head_heldout_dice'])} | gate {f(out['gate_accepted'])}/{f(out['gate_tried'])} | rejected {rej} | {out['seconds']}s "
          f"{by_stage} -> {os.path.join(results_dir, 'gan_summary.json')}", flush=True)
    return out


def main_cache(stage, device):
    """Stages prepare/train/generate/insert for LITS_GAN_SOURCE=cache. Every stage is idempotent under LITS_GAN_RESUME=1."""
    source = make_source("cache")
    restrict = parse_volume_list(VOLUME_LIST) if VOLUME_LIST else None
    ids = source.split_cases(restrict)
    logger.info(f"cache {source.cache_dir}: {len(ids)} usable train-part volumes (original split) at {WORK_MM} mm: {ids[:12]}{'...' if len(ids) > 12 else ''}")
    patches = os.path.join(RESULTS_DIR, "patches.npz")
    samples = os.path.join(RESULTS_DIR, "synthetic", "samples.npz")
    slabs_path = os.path.join(RESULTS_DIR, "aug_slabs.npz")
    pool_size = int(os.environ["LITS_GAN_NUM_SAMPLES"]) if "LITS_GAN_NUM_SAMPLES" in os.environ else int(np.clip(NUM_AUG, 16, 256))
    skip = lambda path: RESUME and not FORCE and os.path.exists(path)
    try:
        if stage in ("all", "prepare"):
            t0 = time.time()
            same = False
            if skip(patches):
                z = load_npz(patches)
                same = "ids" in z.files and z["ids"].tolist() == ids and z["images"].shape[-1] == PATCH and float(z["work_mm"]) == WORK_MM
                logger.info(f"patches.npz exists ({'same volume list, kept' if same else 'different settings, rebuilt'})")
            if not same:
                prepare_patches(ids, patches, size=PATCH, source=source)
            z = load_npz(patches)
            used = sorted({int(c) for c in z["meta"][:, 0]})
            record_stage(RESULTS_DIR, "prepare", patches=int(len(z["images"])), healthy_patches=int(len(z["healthy"])), patch=PATCH, work_mm=WORK_MM,
                         volumes_listed=len(ids), volumes_used=len(used), volumes_used_names=[source.name(c) for c in used],
                         **({} if same else {"seconds": time.time() - t0}))  # a kept file keeps the seconds of the run that built it
        if stage in ("all", "train"):
            t0 = time.time()
            train(patches, EPOCHS, device)
            tm = read_json(os.path.join(RESULTS_DIR, "train_metrics.json"))
            record_stage(RESULTS_DIR, "train", seconds=tm.get("train_seconds", time.time() - t0), **tm)
        if stage in ("all", "generate"):
            if skip(samples):
                logger.info(f"{samples} exists, kept")
            else:
                t0 = time.time()
                model = MultiGAN.load(os.path.join(RESULTS_DIR, "checkpoints", "gan_last.pt"), device)
                st = {}
                p, m = generate_samples(model, pool_size, FUSION, GEN_ATTEMPTS or None, work_mm=WORK_MM, stats=st)
                save_npz_atomic(samples, patches=p, masks=m, work_mm=np.float32(WORK_MM))
                if len(p):
                    save_montage(os.path.join(RESULTS_DIR, "figures", "generated.png"), [(f"generated ({FUSION}) + mask head", p[:8], m[:8] > 0)])
                record_stage(RESULTS_DIR, "generate", seconds=time.time() - t0, pool_size=pool_size, **st)
        if stage in ("all", "insert"):
            if skip(slabs_path) and os.path.exists(os.path.join(RESULTS_DIR, "gan_summary.json")):
                logger.info(f"{slabs_path} exists, kept")
            else:
                d = load_npz(samples)
                work = float(d["work_mm"]) if "work_mm" in d.files else WORK_MM
                st = run_insert_slabs(source, ids, (d["patches"], d["masks"]), slabs_path, NUM_AUG, INSERTIONS, MAX_ATTEMPTS or None,
                                      np.random.default_rng(SEED), work, os.path.join(RESULTS_DIR, "figures", "slab_montage.png"))
                record_stage(RESULTS_DIR, "insert", **st)
                write_gan_summary(RESULTS_DIR)
    except AssertionError as e:  # e.g. no tumour patches in the listed volumes: leave a summary the orchestrator can read, then fail
        logger.error(f"cache pipeline failed in stage {stage}: {e}")
        if os.path.exists(os.path.join(RESULTS_DIR, "stage_metrics.json")):
            write_gan_summary(RESULTS_DIR, {"error": str(e), "accepted": 0})
        else:
            write_json_atomic(os.path.join(RESULTS_DIR, "gan_summary.json"), {"error": str(e), "accepted": 0, "rejected_reasons": {}})
        raise SystemExit(4)


# -----------------------------
# Entry point
# -----------------------------
def main():
    stage = os.environ.get("LITS_GAN_STAGE", "all")
    assert stage in ("all", "prepare", "train", "generate", "insert"), stage
    check_patch(PATCH)
    setup_logging(RESULTS_DIR)
    set_seed(SEED)
    device = pick_device()
    setup_backends(device)
    if GAN_SOURCE == "cache":
        logger.info(f"stage={stage} source=cache cache={CACHE_DIR} results={RESULTS_DIR} window=[{WIN_LO},{WIN_HI}] patch={PATCH} "
                    f"work={WORK_MM} mm fusion={FUSION} device={device} num_aug={NUM_AUG} per_host={INSERTIONS}")
        main_cache(stage, device)
        logger.info("done")
        return
    patches = os.path.join(RESULTS_DIR, "patches.npz")
    samples = os.path.join(RESULTS_DIR, "synthetic", "samples.npz")
    logger.info(f"stage={stage} data={DATA_DIR} results={RESULTS_DIR} "
                f"window=[{WIN_LO},{WIN_HI}] patch={PATCH} fusion={FUSION} device={device}")
    n_train = len(train_part_ids(DATA_DIR))
    ids = split_cases(restrict=parse_volume_list(VOLUME_LIST) if VOLUME_LIST else None)
    logger.info(f"{len(ids)} of {n_train} train-part volumes used (original split: test_size=0.2, random_state=42): "
                f"{ids[:10]}{'...' if len(ids) > 10 else ''}")
    if stage in ("all", "prepare"):
        prepare_patches(ids, patches, source=NiftiSource(DATA_DIR, WORK_MM))
    if stage in ("all", "train"):
        train(patches, EPOCHS, device)
    if stage in ("all", "generate"):
        model = MultiGAN.load(os.path.join(RESULTS_DIR, "checkpoints", "gan_last.pt"), device)
        p, m = generate_samples(model, NUM_SAMPLES, FUSION, work_mm=WORK_MM)
        os.makedirs(os.path.dirname(samples), exist_ok=True)
        np.savez(samples, patches=p, masks=m, work_mm=np.float32(WORK_MM))
        if len(p):
            save_montage(os.path.join(RESULTS_DIR, "figures", "generated.png"), [(f"generated ({FUSION}) + mask head", p[:8], m[:8] > 0)])
    if stage in ("all", "insert"):
        run_insert(samples, ids, os.path.join(RESULTS_DIR, "aug"), os.path.join(RESULTS_DIR, "figures", "insertions.png"))
    logger.info("done")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile {CODE_DIR}/syn_training_improved.py
"""Training and generation improvements for Syn_Img_Pipeline_fixed.py, ported from the reference synthesis.py.

    python syn_training_improved.py                       # LITS_GAN_STAGE=all | prepare | train | generate (same env vars as the pipeline)
    LITS_GAN_ES_PATIENCE=8 python syn_training_improved.py

Syn_Img_Pipeline_fixed.py is not modified. This module subclasses its MultiGAN and reuses its data preparation, PatchStore, MaskPool,
mask-head evaluation and montages; the insert stage stays in Syn_Img_Pipeline_fixed.py (or insertion_improved.py).

What changes compared with Syn_Img_Pipeline_fixed.py (and where it comes from in synthesis.py):
  1. Differentiable augmentation of every discriminator / critic input, real and fake, in the D and the G step (synthesis.diffaug, after
     DiffAugment and StyleGAN2-ADA). The LiTS train part yields a few hundred tumour patches; without augmentation the discriminators
     memorise them. Flips of the three axes, a 90-degree in-plane rotation, brightness / contrast / noise jitter. No translation and no
     cutout: patches are tumour-centred and a cutout could erase the tumour.
  2. Exponential moving average of the generation path (dc1_g, dc2_g, w_g, aggr) with the warm-up min(decay, (1+n)/(10+n))
     (synthesis.Ema, after PGGAN). Sampling, montages, quality scores and the "modules" of the checkpoint use the averaged weights, so
     Syn_Img_Pipeline_fixed.MultiGAN.load and evaluate_synthesis.py read an improved checkpoint unchanged. The live weights are stored
     next to them ("live_modules") for resume.
  3. WGAN-GP drift term eps * E[C(real)^2], eps = 1e-3 (synthesis.wgan_critic_loss, after PGGAN), keeps the critic output from drifting.
  4. A per-epoch quality score and early stopping (synthesis.quality_score / early_stop_decision): Wasserstein-1 distance of tumour-voxel
     HU, plus the gap in tumour-minus-ring contrast, plus the gap in sharpness (mean neighbouring-voxel difference, HU per voxel), between
     generated patches (aggregator, fixed latents, labelled by the mask head) and real tumour patches of the held-out volumes. Lower is
     better. The best epoch is written to checkpoints/gan_best.pt. With LITS_GAN_ES_PATIENCE > 0 training stops after that many epochs
     without improvement (never before LITS_GAN_ES_MIN_EPOCHS). This is a consistency score, not a measure of realism.
  5. Size-controlled generation (synthesis.SIZE_RANGES_MM): the multi-GAN is unconditional, so size is controlled by accept-until-target
     per size class. Each candidate is labelled by the mask head and cleaned (Syn_Img_Pipeline_fixed.clean_mask); its equivalent
     spherical radius r = (3V / 4 pi)^(1/3) puts it in small (2.5-6 mm], medium (6-14 mm] or large (14-28 mm]. samples.npz gains
     radius_mm and size_class arrays; patches / masks / work_mm keep their meaning, so Syn_Img_Pipeline_fixed.run_insert still reads it.

Env vars on top of the pipeline's: LITS_GAN_DIFFAUG_P (0.5, 0 = off), LITS_GAN_EMA (0.999, 0 = off), LITS_GAN_WGAN_DRIFT (0.001),
LITS_GAN_ES_PATIENCE (0 = off), LITS_GAN_ES_MIN_EPOCHS (10), LITS_GAN_ES_MIN_DELTA (1.0 HU), LITS_GAN_SHARP_WEIGHT (1.0),
LITS_GAN_QUALITY_N (32 generated patches per score), LITS_GAN_PER_CLASS (8 samples per size class), LITS_GAN_CLASSES (small,medium,large).
"""
import os
import sys
import time
import math
import random
import contextlib
import logging
import numpy as np
import torch
from scipy import ndimage as ndi
from scipy.stats import wasserstein_distance

HERE = os.path.dirname(os.path.abspath(__file__))
if HERE not in sys.path:
    sys.path.insert(0, HERE)
import Syn_Img_Pipeline_fixed as S  # noqa: E402

DIFFAUG_P = float(os.environ.get("LITS_GAN_DIFFAUG_P", 0.5))
EMA_DECAY = float(os.environ.get("LITS_GAN_EMA", 0.999))
WGAN_DRIFT = float(os.environ.get("LITS_GAN_WGAN_DRIFT", 1e-3))
ES_PATIENCE = int(os.environ.get("LITS_GAN_ES_PATIENCE", 0))
ES_MIN_EPOCHS = int(os.environ.get("LITS_GAN_ES_MIN_EPOCHS", 10))
ES_MIN_DELTA = float(os.environ.get("LITS_GAN_ES_MIN_DELTA", 1.0))
SHARP_WEIGHT = float(os.environ.get("LITS_GAN_SHARP_WEIGHT", 1.0))
QUALITY_N = int(os.environ.get("LITS_GAN_QUALITY_N", 32))
PER_CLASS = int(os.environ.get("LITS_GAN_PER_CLASS", 8))
CLASSES = tuple(c.strip() for c in os.environ.get("LITS_GAN_CLASSES", "small,medium,large").split(",") if c.strip())
RING_MM = (3.0, 8.0)  # surrounding ring of the contrast statistic: distance to the mask in (3, 8] mm
VOXELS_PER_PATCH = 2000  # tumour voxels kept per patch for the pooled HU distribution
QUALITY_SEED = 12345  # fixed latents: every epoch is scored on the same z

#: size classes by equivalent spherical radius (mm), as in synthesis.SIZE_RANGES_MM; small includes its lower edge
SIZE_BINS_MM = {"small": (2.5, 6.0), "medium": (6.0, 14.0), "large": (14.0, 28.0)}

logger = logging.getLogger("gan_fixed")


# -----------------------------
# Differentiable augmentation
# -----------------------------
def diffaug(x, p=DIFFAUG_P):
    """Augment a (B,1,X,Y,Z) batch in [-1,1] before a discriminator; differentiable, so the same call is used in the G step.
    Geometry (each with probability p, one draw for the whole batch): flips of the three axes, a 90-degree rotation in the (X,Y) plane
    (Z is the slice axis of the source volumes and is never rotated into the plane). Intensity (each sample with probability p):
    brightness shift +-0.1, contrast 0.9-1.1 around the sample mean, Gaussian noise 0.02 (all on the [-1,1] scale)."""
    if p <= 0:
        return x
    for ax in (2, 3, 4):
        if random.random() < p:
            x = x.flip(ax)
    if random.random() < p:
        x = torch.rot90(x, random.randint(1, 3), (2, 3))
    shape = (x.shape[0],) + (1,) * (x.dim() - 1)
    on = lambda: (torch.rand(shape, device=x.device) < p).to(x.dtype)  # noqa: E731
    x = x + on() * (torch.rand(shape, device=x.device) - 0.5) * 0.2
    mean = x.mean(dim=tuple(range(1, x.dim())), keepdim=True)
    x = (x - mean) * (1.0 + on() * (torch.rand(shape, device=x.device) - 0.5) * 0.2) + mean
    return x + on() * 0.02 * torch.randn_like(x)


# -----------------------------
# EMA of the generation path
# -----------------------------
class Ema:
    """Exponential moving average of named modules; swap() puts the averaged weights into the live modules in place (the optimisers
    keep their parameter objects) and restores the live weights afterwards. Integer buffers (BatchNorm num_batches_tracked) are copied."""

    def __init__(self, modules, names, decay):
        self.names, self.decay, self.n = list(names), float(decay), 0
        self.shadow = {k: {n: t.detach().clone() for n, t in modules[k].state_dict().items()} for k in self.names}

    @torch.no_grad()
    def update(self, modules):
        self.n += 1
        d = min(self.decay, (1.0 + self.n) / (10.0 + self.n))
        for k in self.names:
            for n, t in modules[k].state_dict().items():
                if t.dtype.is_floating_point:
                    self.shadow[k][n].mul_(d).add_(t.detach(), alpha=1.0 - d)
                else:
                    self.shadow[k][n].copy_(t)

    @contextlib.contextmanager
    def swap(self, modules):
        live = {k: {n: t.detach().clone() for n, t in modules[k].state_dict().items()} for k in self.names}
        for k in self.names:
            modules[k].load_state_dict(self.shadow[k])
        try:
            yield
        finally:
            for k in self.names:
                modules[k].load_state_dict(live[k])


def _clone_state(module):
    return {n: t.detach().clone() for n, t in module.state_dict().items()}


# -----------------------------
# Improved multi-GAN
# -----------------------------
class ImprovedMultiGAN(S.MultiGAN):
    """S.MultiGAN with DiffAugment on every discriminator input, a WGAN drift term and an EMA of the generation path."""

    EMA_NAMES = ("dc1_g", "dc2_g", "w_g", "aggr")

    def __init__(self, device, ld=S.LATENT_DIM, patch=S.PATCH, width=S.WIDTH, pos_weight=S.POS_WEIGHT,
                 diffaug_p=DIFFAUG_P, ema_decay=EMA_DECAY, wgan_drift=WGAN_DRIFT):
        super().__init__(device, ld, patch, width, pos_weight)
        self.diffaug_p, self.wgan_drift, self.ema_decay = float(diffaug_p), float(wgan_drift), float(ema_decay)
        self.ema = Ema(self.modules(), self.EMA_NAMES, ema_decay) if ema_decay > 0 else None

    def _aug(self, x):
        return diffaug(x, self.diffaug_p)

    def ema_scope(self):
        """Context in which the generation path holds its averaged weights (no-op when the EMA is off)."""
        return self.ema.swap(self.modules()) if self.ema is not None else contextlib.nullcontext()

    # ---- training steps (same structure as S.MultiGAN, discriminator inputs augmented) ----
    def step_dcgan(self, which, real):
        g, d = (self.dc1_g, self.dc1_d) if which == 1 else (self.dc2_g, self.dc2_d)
        og, od = self.opt[f"dc{which}_g"], self.opt[f"dc{which}_d"]
        fake = g(self.z(real.size(0)))
        ld = self._bce(d(self._aug(real)), S.REAL_LABEL) + self._bce(d(self._aug(fake.detach())), 0.0)
        self._step(od, ld)
        lg = self._bce(d(self._aug(fake)), 1.0)
        self._step(og, lg)
        return {f"d{which}": self._v(ld), f"g{which}": self._v(lg)}

    def step_wgan(self, real, update_g):
        fake = self.w_g(self.z(real.size(0))).detach()
        ra, fa = self._aug(real), self._aug(fake)
        gp = S.gradient_penalty(self.w_c, ra, fa)
        score_real = self.w_c(ra)
        drift = self.wgan_drift * (score_real ** 2).mean()
        lc = self.w_c(fa).mean() - score_real.mean() + gp + drift
        self._step(self.opt["w_c"], lc)
        out = {"w_critic": self._v(lc), "w_gp": self._v(gp), "w_drift": self._v(drift)}
        if update_g:
            lg = -self.w_c(self._aug(self.w_g(self.z(real.size(0))))).mean()
            self._step(self.opt["w_g"], lg)
            out["w_g"] = self._v(lg)
        return out

    def step_aggregator(self, real):
        with torch.no_grad():
            z = self.z(real.size(0))
            fakes = [self.dc1_g(z), self.dc2_g(z), self.w_g(z)]
        fused = self.aggr(*fakes)
        ld = self._bce(self.ag_d(self._aug(real)), S.REAL_LABEL) + self._bce(self.ag_d(self._aug(fused.detach())), 0.0)
        self._step(self.opt["ag_d"], ld)
        lg = self._bce(self.ag_d(self._aug(fused)), 1.0)
        l1 = (fused - torch.stack(fakes).mean(0)).abs().mean()
        self._step(self.opt["aggr"], lg + S.AGG_L1 * l1)
        return {"ag_d": self._v(ld), "ag_g": self._v(lg), "ag_l1": self._v(l1)}

    def train_batch(self, real, mask, mask_pool=None):
        out = super().train_batch(real, mask, mask_pool)
        if self.ema is not None:
            self.ema.update(self.modules())
        return out

    # ---- sampling from the averaged weights ----
    def sample_all(self, n):
        with self.ema_scope():
            return super().sample_all(n)

    def sample(self, n, fusion=S.FUSION):
        with self.ema_scope():
            return super().sample(n, fusion)

    @torch.no_grad()
    def sample_fixed(self, n, seed=QUALITY_SEED):
        """Aggregator samples from latents drawn on the CPU with a fixed seed (comparable across epochs and devices)."""
        z = torch.randn(n, self.ld, 1, 1, 1, generator=torch.Generator().manual_seed(int(seed))).to(self.device)
        with self.ema_scope():
            self.train(False)
            try:
                return self.aggr(self.dc1_g(z), self.dc2_g(z), self.w_g(z))
            finally:
                self.train(True)

    # ---- persistence ----
    def save(self, path, epoch=None, elapsed=0.0, with_opt=False, extra=None):
        """"modules" = averaged generation path + live discriminators / mask head (what S.MultiGAN.load reads); "live_modules" = the
        live weights (resume). Atomic like S.MultiGAN.save."""
        os.makedirs(os.path.dirname(os.path.abspath(path)), exist_ok=True)
        live = {k: _clone_state(m) for k, m in self.modules().items()}
        with self.ema_scope():
            averaged = {k: _clone_state(m) for k, m in self.modules().items()}
        ck = {"cfg": dict(ld=self.ld, patch=self.patch, width=self.width), "modules": averaged,
              "improved": {"diffaug_p": self.diffaug_p, "ema_decay": self.ema_decay, "wgan_drift": self.wgan_drift,
                           "ema_n": self.ema.n if self.ema is not None else 0}}
        if self.ema is not None:
            ck["live_modules"] = live
        if with_opt:
            ck.update(epoch=epoch, elapsed=elapsed, w_iter=self.w_iter, opt={k: o.state_dict() for k, o in self.opt.items()})
        if extra:
            ck["extra"] = extra
        torch.save(ck, path + ".tmp")
        os.replace(path + ".tmp", path)

    @classmethod
    def load(cls, path, device):
        """Model for sampling: "modules" (the averaged weights) are loaded and the EMA shadow is reset to them."""
        net = super().load(path, device)
        if net.ema is not None:
            net.ema = Ema(net.modules(), cls.EMA_NAMES, net.ema_decay)
        return net

    def load_training_state(self, path):
        """Restore live weights, EMA shadow and optimisers; returns (epoch, elapsed seconds). A plain S.MultiGAN checkpoint also works
        (its weights become both the live and the averaged weights)."""
        ck = torch.load(path, map_location=self.device, weights_only=True)
        assert ck["cfg"] == dict(ld=self.ld, patch=self.patch, width=self.width), f"checkpoint config {ck['cfg']} != this run"
        live = ck.get("live_modules", ck["modules"])
        for k, m in self.modules().items():
            m.load_state_dict(live[k])
        if self.ema is not None:
            self.ema = Ema(self.modules(), self.EMA_NAMES, self.ema_decay)
            for k in self.EMA_NAMES:
                self.ema.shadow[k] = {n: t.to(self.device).clone() for n, t in ck["modules"][k].items()}
            self.ema.n = int(ck.get("improved", {}).get("ema_n", 0))
        if "opt" not in ck:
            return 0, 0.0
        for k, o in self.opt.items():
            o.load_state_dict(ck["opt"][k])
        self.w_iter = int(ck.get("w_iter", 0))
        return int(ck["epoch"]), float(ck.get("elapsed", 0.0))


# -----------------------------
# Quality score and early stopping
# -----------------------------
@torch.no_grad()
def mask_prob(model, x, chunk=4):
    """Mask-head probabilities (n,P,P,P) numpy of a (n,1,P,P,P) device batch, in chunks (3D activations are large)."""
    model.train(False)
    try:
        return np.concatenate([torch.sigmoid(model.mask(x[i:i + chunk]))[:, 0].cpu().numpy() for i in range(0, len(x), chunk)])
    finally:
        model.train(True)


def sharpness_hu(hu):
    """Mean absolute difference of neighbouring voxels over the three axes (HU per voxel); a blurred patch scores lower."""
    return float(np.mean([np.abs(np.diff(hu, axis=a)).mean() for a in range(3)]))


def tumour_statistics(patches, masks, work_mm=1.0, seed=0):
    """Pooled tumour-voxel HU (up to VOXELS_PER_PATCH per patch), per-patch contrast (tumour mean minus ring mean, ring 3-8 mm) and
    per-patch sharpness. patches in [-1,1]; patches with an empty mask or ring are skipped."""
    rng = np.random.default_rng(seed)
    vox, contrast, sharp = [], [], []
    for x, m in zip(patches, masks):
        m = np.asarray(m) > 0
        if not m.any():
            continue
        hu = S.from_unit(np.asarray(x, dtype=np.float32))
        d = ndi.distance_transform_edt(~m, sampling=work_mm)
        ring = (d > RING_MM[0]) & (d <= RING_MM[1])
        if not ring.any():
            continue
        t = hu[m]
        vox.append(t if t.size <= VOXELS_PER_PATCH else t[rng.choice(t.size, VOXELS_PER_PATCH, replace=False)])
        contrast.append(float(t.mean() - hu[ring].mean()))
        sharp.append(sharpness_hu(hu))
    return {"vox": np.concatenate(vox) if vox else np.zeros(0, np.float32), "contrast": np.array(contrast), "sharp": np.array(sharp),
            "n": len(contrast)}


def quality_reference(imgs, masks, work_mm=1.0, max_n=64, seed=0):
    """Real statistics the per-epoch score compares against (pass the held-out volumes' tumour patches)."""
    idx = np.arange(len(imgs))
    if len(idx) > max_n:
        idx = np.sort(np.random.default_rng(seed).choice(len(idx), max_n, replace=False))
    ref = tumour_statistics(imgs[idx], masks[idx], work_mm, seed)
    assert ref["n"], "quality reference: no real patch with a tumour mask and a surrounding ring"
    return ref


def quality_score(model, ref, n=QUALITY_N, work_mm=1.0, sharp_weight=SHARP_WEIGHT, seed=QUALITY_SEED):
    """Score (HU, lower is better) = W1(tumour-voxel HU) + |contrast gap| + sharp_weight * |sharpness gap| between n aggregator samples
    (fixed latents, EMA weights, mask head + clean_mask) and `ref`. score is None when no sample passes the mask gate."""
    x = model.sample_fixed(n, seed)
    min_vox = S.vox_at(S.MIN_MASK_VOX, work_mm)
    pats, msks = [], []
    for xi, pi in zip(x[:, 0].cpu().numpy(), mask_prob(model, x)):
        m, _ = S.clean_mask(pi, min_vox=min_vox)
        if m is not None:
            pats.append(xi)
            msks.append(m)
    out = {"score": None, "hu_w1": None, "contrast_err": None, "sharp_err": None, "gate_accept": len(pats) / max(n, 1)}
    gen = tumour_statistics(pats, msks, work_mm, seed)
    if not gen["n"]:
        return out
    w1 = float(wasserstein_distance(gen["vox"], ref["vox"]))
    ce = abs(float(gen["contrast"].mean()) - float(ref["contrast"].mean()))
    se = abs(float(gen["sharp"].mean()) - float(ref["sharp"].mean()))
    out.update(score=w1 + ce + sharp_weight * se, hu_w1=w1, contrast_err=ce, sharp_err=se, contrast_hu=float(gen["contrast"].mean()))
    return out


def early_stop_decision(scores, patience, min_epochs, min_delta):
    """(best epoch, stop?) for per-epoch scores (index 0 = epoch 1), lower is better. A score is a new best only when it beats the best
    by more than min_delta; None / non-finite scores never are. Stop when >= min_epochs epochs ran and the last `patience` brought no
    new best. Pure function of the scores, so a resumed run takes the same decision (same as synthesis.early_stop_decision)."""
    best_epoch, best = None, float("inf")
    for epoch, s in enumerate(scores, start=1):
        if s is not None and np.isfinite(s) and s < best - min_delta:
            best_epoch, best = epoch, float(s)
    stop = patience > 0 and best_epoch is not None and len(scores) >= min_epochs and len(scores) - best_epoch >= patience
    return best_epoch, stop


# -----------------------------
# Training loop
# -----------------------------
def train_improved(patches_path, epochs=S.EPOCHS, device=None, results_dir=S.RESULTS_DIR, resume=None, max_seconds=None,
                   patience=ES_PATIENCE, min_epochs=ES_MIN_EPOCHS, min_delta=ES_MIN_DELTA, width=None, quality_n=QUALITY_N,
                   batch_size=None, sample_every=S.SAMPLE_EVERY):
    """S.train with ImprovedMultiGAN, a quality score every epoch, checkpoints/gan_best.pt and early stopping (patience > 0).
    Writes checkpoints/gan_last.pt (every S.CKPT_EVERY epochs and at the end), train_metrics.json (incl. quality_history) and
    figures/train_epoch*.png like S.train. Returns the model; its live weights are those of the last epoch, gan_best.pt holds the best."""
    device = device or S.pick_device()
    resume = S.RESUME if resume is None else resume
    max_seconds = S.MAX_SECONDS if max_seconds is None else max_seconds
    S.setup_backends(device)
    d = S.load_npz(patches_path)
    imgs, masks, healthy = d["images"], d["masks"], d["healthy"]
    t_cid = d["meta"][:, 0]
    h_cid = d["healthy_cid"] if "healthy_cid" in d.files else np.full(len(healthy), -1)
    work = float(d["work_mm"]) if "work_mm" in d.files else 1.0
    patch = imgs.shape[-1]
    S.assert_unit_range(imgs, "training patches")
    n = len(imgs)
    bs = min(batch_size or S.BATCH_SIZE, n)
    rng = np.random.default_rng(S.SEED)
    held = S.split_holdout(np.concatenate([t_cid, h_cid]), rng)
    t_tr, h_tr = ~np.isin(t_cid, list(held)), ~np.isin(h_cid, list(held))
    t_tr_i, h_tr_i = np.flatnonzero(t_tr), np.flatnonzero(h_tr)
    assert len(t_tr_i), "no tumour patches left for mask-head training"
    ref_idx = np.flatnonzero(~t_tr) if held and (~t_tr).any() else np.arange(n)  # held-out volumes when there are any
    ref = quality_reference(imgs[ref_idx].astype(np.float32), masks[ref_idx], work)
    model = ImprovedMultiGAN(device, patch=patch, width=width or S.WIDTH)
    model.check_range = False
    min_mask_vox = S.vox_at(S.MIN_MASK_VOX, work)
    ckpt = os.path.join(results_dir, "checkpoints", "gan_last.pt")
    best_ckpt = os.path.join(results_dir, "checkpoints", "gan_best.pt")
    metrics_path = os.path.join(results_dir, "train_metrics.json")
    start, elapsed0 = 0, 0.0
    if resume and os.path.exists(ckpt):
        start, elapsed0 = model.load_training_state(ckpt)
        rng = np.random.default_rng([S.SEED, start])
        logger.info(f"resumed from {ckpt}: epoch {start}, {elapsed0:.0f}s of training already spent")
    store = S.PatchStore({"images": imgs, "masks": masks, "healthy": healthy}, device)
    logger.info(f"[improved] device {device}; {n} tumour / {len(healthy)} healthy patches {patch}^3 at {work} mm; batch {bs}; "
                f"epochs {epochs}; diffaug p={model.diffaug_p} ema={model.ema_decay} drift={model.wgan_drift}; quality reference "
                f"{ref['n']} real patches ({'held-out volumes' if len(ref_idx) < n else 'all volumes'}); early stopping "
                f"{'off' if patience <= 0 else f'patience {patience}, min epochs {min_epochs}, min delta {min_delta} HU'}")
    pool = S.MaskPool(store, t_tr_i, h_tr_i, bs, rng, device)
    metrics = S.read_json(metrics_path) if resume and os.path.exists(metrics_path) else {}
    metrics.update(patches=int(n), healthy_patches=int(len(healthy)), patch=int(patch), work_mm=work, epochs_target=int(epochs),
                   held_out_volumes=[int(v) for v in sorted(held)], device=str(device), improved=True)
    qhist = [h for h in metrics.get("quality_history", []) if h["epoch"] <= start]
    best_epoch, stop = early_stop_decision([h["score"] for h in qhist], patience, min_epochs, min_delta)
    steps = n // bs
    assert steps > 0, f"batch size {bs} larger than the patch set"
    t0, ep, stopped = time.time(), start, ("early" if stop else ("max" if start >= epochs else None))
    model.sync = False
    for ep in range(start + 1, epochs + 1):
        if stop:
            break
        perm = torch.as_tensor(rng.permutation(n)[:steps * bs], dtype=torch.long).to(device if store.on_device else "cpu")
        pool.refill(steps * S.MASK_STEPS)
        sums, cnts = {}, {}
        for i in range(steps):
            idx = perm[i * bs:(i + 1) * bs]
            for k, v in model.train_batch(store.gather("images", idx), store.gather("masks", idx), pool).items():
                sums[k], cnts[k] = sums.get(k, 0.0) + v, cnts.get(k, 0) + 1
        means = {k: float(v) / cnts[k] for k, v in sums.items()}
        assert all(np.isfinite(v) for v in means.values()), "non-finite loss"
        model.sync = True
        q = quality_score(model, ref, quality_n, work)
        qhist.append(dict(q, epoch=ep))
        best_epoch, stop = early_stop_decision([h["score"] for h in qhist], patience, min_epochs, min_delta)
        spent = elapsed0 + time.time() - t0
        qs = "none" if q["score"] is None else f"{q['score']:.1f}"
        logger.info(f"epoch {ep}/{epochs} ({spent:.0f}s) " + " ".join(f"{k}={v:.3f}" for k, v in means.items())
                    + f" | quality {qs} HU (gate {q['gate_accept']:.2f}), best epoch {best_epoch}")
        if best_epoch == ep:
            model.save(best_ckpt, epoch=ep, elapsed=spent, extra={"quality": {k: v for k, v in q.items() if v is not None}})
        if ep % sample_every == 0 or ep == epochs or stop:
            for tag, ev in (("train", S.eval_mask_head(model, imgs[t_tr_i], masks[t_tr_i], healthy[h_tr_i], min_vox=min_mask_vox)),
                            ("heldout", S.eval_mask_head(model, imgs[~t_tr], masks[~t_tr], healthy[~h_tr], n=64, min_vox=min_mask_vox) if held else {})):
                if ev:
                    logger.info(f"mask head {tag}: " + " ".join(f"{k}={v:.3f}" for k, v in ev.items()))
                    metrics[f"mask_head_{tag}"] = dict(ev, epoch=ep)
            s = {k: v[:, 0].cpu().numpy() for k, v in model.sample_all(4).items()}
            pm = mask_prob(model, torch.from_numpy(s["aggregator"])[:, None].to(device)) > 0.5
            S.save_montage(os.path.join(results_dir, "figures", f"train_epoch{ep:03d}.png"),
                           [("real", imgs[:4].astype(np.float32), masks[:4] > 0), ("dcgan 1 (EMA)", s["dc1"], None),
                            ("dcgan 2 (EMA)", s["dc2"], None), ("wgan-gp (EMA)", s["wgan"], None), ("aggregator + mask head", s["aggregator"], pm)])
        model.sync = False
        out_of_time = bool(max_seconds) and spent >= max_seconds and ep < epochs
        if ep % max(S.CKPT_EVERY, 1) == 0 or ep == epochs or out_of_time or stop:
            model.save(ckpt, epoch=ep, elapsed=spent, with_opt=True)
        history = [h for h in metrics.get("loss_history", []) if h["epoch"] < ep]
        metrics.update(epochs_done=ep, train_seconds=spent, last_losses=means, loss_history=history + [dict(means, epoch=ep, seconds=spent)],
                       quality_history=qhist, best_epoch=best_epoch)
        S.write_json_atomic(metrics_path, metrics)
        if out_of_time:
            logger.warning(f"GAN time budget {max_seconds:.0f}s reached after epoch {ep}/{epochs}: stopping early")
            stopped = "time"
            break
        if stop:
            logger.info(f"early stopping after epoch {ep}: no better quality score for {patience} epochs (best epoch {best_epoch})")
            stopped = "early"
    model.sync = True
    metrics.update(epochs_done=ep, stopped=stopped or "max", best_epoch=best_epoch)
    S.write_json_atomic(metrics_path, metrics)
    return model


# -----------------------------
# Size-controlled generation
# -----------------------------
def equivalent_radius_mm(mask, spacing=1.0):
    """Radius (mm) of the sphere with the volume of the binary mask (voxel count times voxel volume)."""
    v = float(np.count_nonzero(mask)) * float(np.prod(np.broadcast_to(np.asarray(spacing, dtype=np.float64), (3,))))
    return (3.0 * v / (4.0 * math.pi)) ** (1.0 / 3.0)


def size_class(radius_mm, bins=None):
    """Size class of an equivalent radius (small includes its lower edge), or None outside every class."""
    for name, (lo, hi) in (bins or SIZE_BINS_MM).items():
        if (lo <= radius_mm <= hi) if name == "small" else (lo < radius_mm <= hi):
            return name
    return None


def generate_by_size(model, n_per_class=PER_CLASS, classes=CLASSES, fusion=S.FUSION, work_mm=1.0, max_attempts=None, batch=8, stats=None):
    """Accept-until-target per size class: sample (EMA weights for an ImprovedMultiGAN), label with the mask head, clean the mask
    (threshold, largest component, minimum size, away from the border), measure its equivalent radius at work_mm and keep it when
    its class still needs samples. Returns {class: (patches (n,P,P,P) float32, masks uint8, radii_mm float32)}; a class can come out
    short when the generator rarely produces that size (see stats['rejects'] / stats['radius_hist_mm'])."""
    unknown = [c for c in classes if c not in SIZE_BINS_MM]
    assert not unknown, f"unknown size classes {unknown}; known: {list(SIZE_BINS_MM)}"
    max_attempts = max_attempts or 100 * n_per_class * len(classes)
    min_vox = S.vox_at(S.MIN_MASK_VOX, work_mm)
    bins = {c: SIZE_BINS_MM[c] for c in classes}
    keep = {c: ([], [], []) for c in classes}
    rejects, radii_seen, tried = {}, [], 0
    while any(len(keep[c][0]) < n_per_class for c in classes) and tried < max_attempts:
        x = model.sample(batch, fusion)
        for xi, pi in zip(x[:, 0].cpu().numpy(), mask_prob(model, x)):
            tried += 1
            m, why = S.clean_mask(pi, min_vox=min_vox)
            if m is None:
                rejects[why] = rejects.get(why, 0) + 1
                continue
            r = equivalent_radius_mm(m, work_mm)
            radii_seen.append(r)
            c = size_class(r, bins)
            if c is None:
                rejects["size_out_of_range"] = rejects.get("size_out_of_range", 0) + 1
            elif len(keep[c][0]) >= n_per_class:
                rejects[f"{c}_full"] = rejects.get(f"{c}_full", 0) + 1
            else:
                S.assert_unit_range(xi, "generated patch")
                keep[c][0].append(xi)
                keep[c][1].append(m)
                keep[c][2].append(r)
    out = {}
    for c in classes:
        p, m, r = keep[c]
        e = (0,) + (model.patch,) * 3
        out[c] = (np.stack(p).astype(np.float32) if p else np.zeros(e, np.float32), np.stack(m).astype(np.uint8) if m else np.zeros(e, np.uint8),
                  np.asarray(r, dtype=np.float32))
    hist, edges = np.histogram(radii_seen, bins=np.arange(0, 32, 2)) if radii_seen else (np.zeros(15, int), np.arange(0, 32, 2))
    st = dict(tried=tried, accepted={c: int(len(out[c][0])) for c in classes}, rejects=rejects,
              radius_hist_mm={"edges": edges.tolist(), "counts": hist.tolist()})
    logger.info(f"size-controlled generation ({fusion}): {st['accepted']} in {tried} attempts; rejects {rejects}")
    if stats is not None:
        stats.update(st)
    return out


def save_samples(path, by_size, work_mm=1.0):
    """samples.npz readable by S.run_insert (patches, masks, work_mm) plus radius_mm and size_class per sample."""
    cls = [c for c in by_size for _ in range(len(by_size[c][0]))]
    nonempty = [by_size[c] for c in by_size if len(by_size[c][0])]
    if nonempty:
        p, m, r = (np.concatenate([v[i] for v in nonempty]) for i in range(3))
    else:
        any_c = next(iter(by_size.values()))
        p, m, r = any_c
    S.save_npz_atomic(path, patches=p, masks=m, radius_mm=r.astype(np.float32), size_class=np.array(cls, dtype="U8"), work_mm=np.float32(work_mm))
    return path


# -----------------------------
# Entry point
# -----------------------------
def main():
    stage = os.environ.get("LITS_GAN_STAGE", "all")
    assert stage in ("all", "prepare", "train", "generate"), f"{stage}: the insert stage lives in Syn_Img_Pipeline_fixed.py / insertion_improved.py"
    assert S.GAN_SOURCE == "nifti", "syn_training_improved.py runs the NIfTI source; for the cache source use the patches.npz its prepare stage writes"
    S.check_patch(S.PATCH)
    S.setup_logging(S.RESULTS_DIR)
    S.set_seed(S.SEED)
    device = S.pick_device()
    patches = os.path.join(S.RESULTS_DIR, "patches.npz")
    samples = os.path.join(S.RESULTS_DIR, "synthetic", "samples.npz")
    if stage in ("all", "prepare"):
        ids = S.split_cases(restrict=S.parse_volume_list(S.VOLUME_LIST) if S.VOLUME_LIST else None)
        S.prepare_patches(ids, patches, source=S.NiftiSource(S.DATA_DIR, S.WORK_MM))
    if stage in ("all", "train"):
        train_improved(patches, S.EPOCHS, device)
    if stage in ("all", "generate"):
        ck = os.path.join(S.RESULTS_DIR, "checkpoints", "gan_best.pt")
        ck = ck if os.path.exists(ck) else os.path.join(S.RESULTS_DIR, "checkpoints", "gan_last.pt")
        model = ImprovedMultiGAN.load(ck, device)  # "modules" are the averaged weights already
        logger.info(f"generating from {ck}")
        st = {}
        by_size = generate_by_size(model, PER_CLASS, CLASSES, S.FUSION, S.WORK_MM, S.GEN_ATTEMPTS or None, stats=st)
        save_samples(samples, by_size, S.WORK_MM)
        S.write_json_atomic(os.path.join(S.RESULTS_DIR, "synthetic", "generation_stats.json"), st)
        rows = [(f"{c} (r {r.min():.1f}-{r.max():.1f} mm)", p[:8], m[:8] > 0) for c, (p, m, r) in by_size.items() if len(p)]
        if rows:
            S.save_montage(os.path.join(S.RESULTS_DIR, "figures", "generated_by_size.png"), rows)
    logger.info("done")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile {CODE_DIR}/insertion_improved.py
"""Improved host insertion for Syn_Img_Pipeline_fixed.py, ported from the reference insertion.py (Hu et al., CVPR 2023 style).

    python insertion_improved.py                          # NIfTI hosts of the train part, samples from <results>/synthetic/samples.npz
    LITS_INS_SIZE=large LITS_INS_N=2 python insertion_improved.py
    LITS_INS_NO_VESSELS=1 LITS_INS_NO_MASS=1 python insertion_improved.py      # ablations

Syn_Img_Pipeline_fixed.insert_one picks a random site in the liver eroded by the tumour radius, shifts the patch mean onto the host
liver mean and blends it with a feathered alpha. It does not know about vessels, keeps whatever size the generator produced, has no
mass effect and no automatic quality control. This module keeps its blending (feathered alpha, host-matched high-frequency noise,
volume-preserving mask resampling) and adds, from insertion.py:

  1. Vessel mask: CT smoothed inside the liver (G(ct*liver)/G(liver), sigma 1.5 mm), robust parenchyma mean (eroded liver, sigma
     clipping), vessels = parenchyma voxels brighter than mean + 15 HU, components < 0.05 ml dropped (insertion.segment_vessels).
  2. Vessel-aware placement: a centre is a candidate when its depth in the liver > R + 3 mm, its vessel distance > R + 3 mm and its
     distance to existing tumours > R + LITS_GAN_EXCLUDE_MM (R = equivalent radius on the host grid); then the REAL resampled mask is
     checked at that centre (every tumour voxel keeps the liver and vessel margins, insertion.mask_clearance_ok), up to 40 draws.
  3. Physical size control: the generated patch is scaled isotropically so that its mask has the requested equivalent radius
     (small 5 / medium 10 / large 15 mm, or any value), then resampled onto the host spacing in one step.
  4. HU harmonisation from a 3-10 mm host shell around the placed tumour that excludes vessels and tumours, against a robust ring mean
     of the generated patch that excludes clipped air (insertion.harmonize_hu, mode offset); alpha is 0 on vessel voxels outside the
     tumour, so vessels keep their contrast.
  5. Mass effect: the paper's local scaling warp gamma' = [1 - (1 - gamma/gamma_max)^2 I/100] gamma, gamma_max = 1.3 R, I = 0 / 20 / 30
     for small / medium / large; the numerical Jacobian of the sampling map must stay >= 0.2 (I is lowered in steps of 5 otherwise).
     CT warped with a cubic spline clipped to the local range, liver and vessels by nearest neighbour, tumour labels with a smoothed
     indicator so that sub-voxel surface shifts are not lost (insertion.apply_mass_effect).
  6. Optional capsule ring (additive HU peak per contrast phase, off by default: LiTS does not record the phase).
  7. Quality control per insertion: tumour in liver, vessel overlap, radius vs request (expansion by the warp included), finite HU in
     range, tumour-to-parenchyma contrast and tumour std within bounds, one connected component, valid warp. A failed insertion is
     reverted (on_fail="reject") or kept and flagged (on_fail="flag"). Every insertion, accepted or not, is a row of qc_report.csv.

Conventions follow Syn_Img_Pipeline_fixed.py: host arrays are (X,Y,Z) in array-voxel space with per-axis spacing from the header zooms;
HU float32 while working, int16 on disk; labels 0 background, 1 liver, 2 tumour. Every write is confined to a local edit box, which is
backed up before the insertion, so voxels outside it are bit-identical to the host.

Env vars: LITS_INS_SIZE (small|medium|large|<radius mm>|keep, default medium), LITS_INS_N (tumours per host, default
LITS_GAN_INSERTIONS), LITS_INS_NO_VESSELS, LITS_INS_NO_MASS, LITS_INS_CAPSULE, LITS_INS_PHASE (default unknown),
LITS_INS_ON_FAIL (reject|flag), LITS_INS_OUT (default <results>/aug_improved), LITS_INS_MAX_HOSTS (0 = all).
"""
import os
import sys
import csv
import json
import math
import logging
from dataclasses import dataclass, field, asdict
import numpy as np
from scipy import ndimage as ndi

HERE = os.path.dirname(os.path.abspath(__file__))
if HERE not in sys.path:
    sys.path.insert(0, HERE)
import Syn_Img_Pipeline_fixed as S  # noqa: E402

SIZES_MM = {"small": 5.0, "medium": 10.0, "large": 15.0}
MASS_INTENSITY = {"small": 0.0, "medium": 20.0, "large": 30.0}
CAPSULE_HU = {"non_contrast": 0.0, "arterial": 60.0, "portal_venous": 120.0, "delayed": 80.0, "unknown": 60.0}
NEW = 3  # temporary label of the tumour being inserted (turned into 2 at the end)
STRUCT26 = np.ones((3, 3, 3), bool)

logger = logging.getLogger("gan_fixed")


@dataclass
class InsertParams:
    """Every setting of one insertion (lengths in mm, intensities in HU)."""
    target_radius_mm: float = None  # None keeps the generated size
    min_scale: float = 0.5  # bounds of the isotropic size rescale of a generated patch
    max_scale: float = 2.0
    patch_radius_tol_mm: float = 1.5  # samples within this of the best radius match are equally likely (needs samples radius_mm)
    vessels: bool = True
    vessel_sigma_mm: float = 1.5
    vessel_offset_hu: float = 15.0
    vessel_erode_mm: float = 5.0
    vessel_min_ml: float = 0.05
    margin_liver_mm: float = 3.0
    margin_vessel_mm: float = 3.0
    exclude_mm: float = S.EXCLUDE_MM
    fit_attempts: int = 40
    feather_mm: float = S.FEATHER_MM
    host_shell_mm: tuple = (3.0, 10.0)
    gen_ring_mm: tuple = (2.0, 10.0)
    mass_effect: bool = True
    mass_intensity: float = None  # None: by size class of the radius (MASS_INTENSITY)
    gamma_factor: float = 1.3
    min_jacobian: float = 0.2
    max_displacement_mm: float = 12.0
    intensity_step: float = 5.0
    capsule: bool = False
    capsule_sigma_mm: float = 0.8
    capsule_inner_mm: float = -1.0
    capsule_outer_mm: float = 1.5
    on_fail: str = "reject"  # reject | flag
    radius_tol: float = 0.25
    max_outside_liver_frac: float = 0.0
    max_vessel_overlap_frac: float = 0.02
    contrast_hu_bounds: tuple = (-120.0, 80.0)
    tumor_std_hu_bounds: tuple = (2.0, 80.0)

    def __post_init__(self):
        assert self.on_fail in ("reject", "flag"), self.on_fail
        assert 0 < self.min_scale <= 1 <= self.max_scale
        assert self.capsule_outer_mm > self.capsule_inner_mm and self.capsule_sigma_mm > 0
        assert 0 < self.min_jacobian <= 1 and self.intensity_step > 0


def size_class_of(radius_mm):
    return "small" if radius_mm <= 6.0 else ("medium" if radius_mm <= 14.0 else "large")


def equivalent_radius_mm(mask, spacing):
    v = float(np.count_nonzero(mask)) * float(np.prod(spacing))
    return (3.0 * v / (4.0 * math.pi)) ** (1.0 / 3.0)


def _bbox(mask, pad=(0, 0, 0)):
    idx = [np.flatnonzero(mask.any(axis=tuple(a for a in range(3) if a != ax))) for ax in range(3)]
    if any(len(i) == 0 for i in idx):
        return None
    return tuple(slice(max(int(i[0]) - int(p), 0), min(int(i[-1]) + 1 + int(p), n)) for i, p, n in zip(idx, pad, mask.shape))


def _robust_mean(v, k=2.5, n_iter=3):
    v = np.asarray(v, np.float64)
    for _ in range(n_iter):
        mu, sd = v.mean(), v.std()
        keep = np.abs(v - mu) <= k * sd
        if sd == 0 or keep.all() or keep.sum() < 5:
            break
        v = v[keep]
    return float(v.mean())


# -----------------------------
# 1. Vessels
# -----------------------------
def sigma_clipped(values, k=3.0, n_iter=10):
    """Mean and std after iterative k-sigma clipping started from median / MAD (std corrected for the truncation)."""
    v = np.asarray(values, np.float64)
    c, s = float(np.median(v)), 1.4826 * float(np.median(np.abs(v - np.median(v)))) or float(v.std())
    keep = np.abs(v - c) <= k * s if s > 0 else np.ones(v.shape, bool)
    for _ in range(n_iter):
        mu, sd = v[keep].mean(), v[keep].std()
        new = np.abs(v - mu) <= k * sd if sd > 0 else keep
        if not new.any() or np.array_equal(new, keep):
            break
        keep = new
    phi = math.exp(-0.5 * k * k) / math.sqrt(2 * math.pi)
    factor = math.sqrt(max(1.0 - 2.0 * k * phi / math.erf(k / math.sqrt(2.0)), 1e-12))
    return float(v[keep].mean()), float(v[keep].std()) / factor


def segment_vessels(hu, seg, spacing, p=None):
    """Bool (X,Y,Z) vessel mask inside the liver parenchyma (label 1) and its statistics; see the module docstring (item 1)."""
    p = p or InsertParams()
    sp = np.asarray(spacing, np.float64)
    par = seg == 1
    full = np.zeros(seg.shape, bool)
    if not par.any():
        return full, {"source": "empty_liver"}
    sig = p.vessel_sigma_mm / sp
    box = _bbox(par, np.ceil(4 * sig).astype(int) + 1)
    c, m = hu[box].astype(np.float32), par[box]
    num = ndi.gaussian_filter(np.where(m, c, 0).astype(np.float32), sig, mode="constant")
    den = ndi.gaussian_filter(m.astype(np.float32), sig, mode="constant")
    sm = np.where(den > 1e-3, num / np.maximum(den, 1e-3), c)
    depth = ndi.distance_transform_edt(np.pad(m, 1), sampling=sp)[1:-1, 1:-1, 1:-1]
    radius, region = p.vessel_erode_mm, depth > p.vessel_erode_mm
    while region.sum() < 200 and radius > 0.1:
        radius *= 0.5
        region = depth > radius
    if region.sum() < 200:
        region = m
    mean, std = sigma_clipped(sm[region])
    thr = mean + p.vessel_offset_hu
    cand = (sm > thr) & m
    lab, n = ndi.label(cand, structure=STRUCT26)
    removed = 0
    if n:
        ml = np.bincount(lab.ravel())[1:] * float(np.prod(sp)) / 1000.0
        keep = np.concatenate([[False], ml >= p.vessel_min_ml])
        removed = int((~keep[1:]).sum())
        cand = keep[lab]
    full[box] = cand
    liver_ml = float(par.sum()) * float(np.prod(sp)) / 1000.0
    vol_ml = float(cand.sum()) * float(np.prod(sp)) / 1000.0
    return full, {"source": "threshold", "parenchyma_mean_hu": mean, "parenchyma_std_hu": std, "threshold_hu": thr, "volume_ml": vol_ml,
                  "fraction_of_liver": vol_ml / liver_ml if liver_ml else 0.0, "components_removed": removed}


# -----------------------------
# 2. Placement maps
# -----------------------------
class HostMaps:
    """Depth in the liver, distance to vessels and distance to existing tumours (mm), on the liver bbox padded by one voxel (array
    borders count as outside the liver). Index maps with full-volume voxel indices through `origin`."""

    def __init__(self, seg, vessels, spacing):
        sp = tuple(float(s) for s in spacing)
        box = _bbox(seg > 0)
        assert box is not None, "host has no liver label"
        self.origin = np.array([s.start for s in box]) - 1
        liver = np.pad(seg[box] > 0, 1)
        self.depth = ndi.distance_transform_edt(liver, sampling=sp).astype(np.float32)
        tum = np.pad(seg[box] >= 2, 1)
        self.tdist = ndi.distance_transform_edt(~tum, sampling=sp).astype(np.float32) if tum.any() else np.full(liver.shape, np.inf, np.float32)
        ves = np.pad(vessels[box], 1) if vessels is not None else np.zeros_like(liver)
        self.vdist = ndi.distance_transform_edt(~ves, sampling=sp).astype(np.float32) if ves.any() else np.full(liver.shape, np.inf, np.float32)

    def stats(self, idx_full):
        """(depth, vessel distance, tumour distance) minima over full-volume voxel indices (n,3); depth 0 outside the maps."""
        loc = np.asarray(idx_full) - self.origin
        if np.any(loc < 0) or np.any(loc >= np.array(self.depth.shape)):
            return 0.0, 0.0, 0.0
        ix = tuple(loc.T)
        return float(self.depth[ix].min()), float(self.vdist[ix].min()), float(self.tdist[ix].min())


def candidate_centres(maps, radius_mm, cen, patch_shape, vol_shape, p):
    """Full-volume voxel indices of valid centres for a sphere of radius_mm (strict margins, window inside the volume), plus the voxel
    counts left after each criterion (to name the one that emptied the map)."""
    ok = maps.depth > radius_mm + p.margin_liver_mm
    counts = {"depth": int(ok.sum())}
    ok &= maps.vdist > radius_mm + p.margin_vessel_mm
    counts["vessel"] = int(ok.sum())
    ok &= maps.tdist > radius_mm + p.exclude_mm
    counts["tumour_gap"] = int(ok.sum())
    c = np.argwhere(ok) + maps.origin
    start = c - cen
    inside = np.all(start >= 0, 1) & np.all(start + np.asarray(patch_shape) <= np.asarray(vol_shape), 1)
    counts["window"] = int(inside.sum())
    return c[inside], counts


# -----------------------------
# 3. Size control
# -----------------------------
def prepare_patch(patch, mask, spacing, work_mm, target_radius_mm, p):
    """Scale the generated patch isotropically to target_radius_mm (clipped to [min_scale, max_scale]) and resample it to the host spacing
    in one step: S.resample_patch_to_host is told that a generator voxel measures work_mm * scale, so the physical size changes by
    `scale` and the mask volume is preserved exactly. Returns (patch_rs in [-1,1], mask_rs bool, scale)."""
    r0 = equivalent_radius_mm(mask, (work_mm,) * 3)
    scale = 1.0 if not target_radius_mm else float(np.clip(target_radius_mm / max(r0, 1e-6), p.min_scale, p.max_scale))
    img, m = S.resample_patch_to_host(patch, mask, spacing, work_mm * scale)
    return img, m, scale


def pick_sample(samples, target_radius_mm, rng, p):
    """Index of a sample: random, or (target and samples['radius_mm']) among the samples nearest to the target radius."""
    n = len(samples["patches"])
    r = samples.get("radius_mm")
    if target_radius_mm and r is not None and len(r) == n and n:
        err = np.abs(np.asarray(r, np.float64) - target_radius_mm)
        cand = np.flatnonzero(err <= err.min() + p.patch_radius_tol_mm + 1e-9)
        return int(cand[rng.integers(len(cand))])
    return int(rng.integers(n))


# -----------------------------
# 4. Blending with vessel-aware harmonisation
# -----------------------------
def blend_at(hu, seg, vessels, spacing, patch_rs, m_rs, start, rng, p):
    """Blend patch_rs into float32 `hu` IN PLACE with its window starting at `start`; the tumour is written as label NEW. Blending as in
    S.insert_one, with the host reference from a vessel-free shell and alpha 0 on vessel voxels outside the tumour. None if the mask
    has no liver voxel there."""
    sp = np.asarray(spacing, np.float64)
    ps = np.array(patch_rs.shape)
    win = tuple(slice(int(s), int(s + n)) for s, n in zip(start, ps))
    hw, sw, vw = hu[win].astype(np.float32), seg[win], vessels[win]
    liver_w = sw == 1
    m_eff = m_rs & liver_w
    if not m_eff.any():
        return None
    dil = ndi.distance_transform_edt(~m_eff, sampling=sp) <= 1.0
    alpha = ndi.gaussian_filter(dil.astype(np.float32), p.feather_mm / sp, mode="constant")
    alpha = np.maximum(alpha, m_eff) * S.border_taper(tuple(ps), sp) * liver_w
    alpha[vw & ~m_eff] = 0.0
    alpha[alpha < 0.01] = 0.0
    support = alpha > 0
    # harmonisation (offset mode): host shell around the placed tumour (no vessels, no tumours) vs a robust generated ring without air
    d = ndi.distance_transform_edt(~m_eff, sampling=sp)
    shell = (d >= p.host_shell_mm[0]) & (d <= p.host_shell_mm[1]) & liver_w & ~vw
    host_ref = _robust_mean(hw[shell]) if shell.sum() >= 30 else float(hw[liver_w & ~support].mean() if (liver_w & ~support).sum() >= 100 else hw[liver_w].mean())
    gen = S.from_unit(patch_rs)
    dg = ndi.distance_transform_edt(~m_rs, sampling=sp)
    ring = (dg > p.gen_ring_mm[0]) & (dg <= p.gen_ring_mm[1])
    ring_soft = ring & (gen > S.WIN_LO + 5.0)  # generated values at the window floor are air padding / clipped, not parenchyma
    ring = ring_soft if ring_soft.sum() >= 20 else (ring if ring.sum() >= 20 else ~m_rs)
    gen_ref = _robust_mean(gen[ring])
    shift = host_ref - gen_ref
    gen = gen + shift
    # host-matched high-frequency noise (same as S.insert_one)
    pad = np.ceil(24.0 / sp).astype(int)
    nsl = tuple(slice(max(0, w.start - q), min(n, w.stop + q)) for w, q, n in zip(win, pad, seg.shape))
    sig = S.liver_noise_std(hu, seg, spacing, nsl)
    sig = 0.0 if sig is None else sig
    sg = S.NOISE_SIGMA_MM / sp
    noise = ndi.gaussian_filter(rng.standard_normal(hw.shape).astype(np.float32), S.NOISE_PRE_MM / sp)
    noise = noise - ndi.gaussian_filter(noise, sg)
    noise *= sig / max(float(noise.std()), 1e-6)
    host_lp = ndi.gaussian_filter(hw, sg, mode="nearest")
    out = ((1 - alpha) * host_lp + alpha * ndi.gaussian_filter(gen, sg, mode="nearest")
           + np.sqrt(1 - alpha) * (hw - host_lp) + np.sqrt(alpha) * noise)
    hu[win][support] = out[support]
    sw[m_eff] = NEW
    return {"window": win, "support": support, "alpha": alpha, "mask": m_eff, "shift_hu": float(shift), "host_ref_hu": host_ref,
            "gen_ref_hu": gen_ref, "noise_std": float(sig), "n_label": int(m_eff.sum())}


# -----------------------------
# 5. Mass effect
# -----------------------------
def _ratio(gamma, gmax, intensity):
    u = np.clip(gamma / gmax, 0.0, 1.0)
    return 1.0 - (1.0 - u) ** 2 * (intensity / 100.0)


def radial_scale(gamma, gamma_max, intensity):
    """Paper formula gamma' = [1 - (1 - gamma/gamma_max)^2 I/100] gamma for gamma <= gamma_max, identity beyond."""
    g = np.asarray(gamma, np.float64)
    return np.where(g <= gamma_max, _ratio(g, gamma_max, intensity) * g, g)


def expected_radius_after_warp(radius_mm, gamma_max_mm, intensity):
    """Output radius of a sphere of input radius radius_mm (solves gamma'(g) = r by bisection; identity for I <= 0)."""
    k = min(max(float(intensity), 0.0) / 100.0, 0.99)
    if k <= 0 or radius_mm <= 0 or radius_mm >= gamma_max_mm:
        return float(radius_mm)
    lo, hi = float(radius_mm), float(gamma_max_mm)
    for _ in range(60):
        mid = 0.5 * (lo + hi)
        lo, hi = (mid, hi) if mid * (1 - (1 - mid / gamma_max_mm) ** 2 * k) < radius_mm else (lo, mid)
    return 0.5 * (lo + hi)


def compute_roi(shape, spacing, centre, reach_mm, pad_vox=6):
    half = reach_mm / np.asarray(spacing, np.float64)
    c = np.asarray(centre, np.float64)
    lo = np.maximum(np.floor(c - half).astype(int) - pad_vox, 0)
    hi = np.minimum(np.ceil(c + half).astype(int) + pad_vox + 1, np.asarray(shape))
    return tuple(slice(int(a), int(max(b, a))) for a, b in zip(lo, hi))


def sampling_jacobian(src, spacing):
    """Numerical determinant of d S / d x (both in mm) of a (3, rx, ry, rz) sampling map."""
    sp = np.asarray(spacing, np.float64)
    s_mm = src * sp[:, None, None, None]
    j = np.zeros((3, 3) + src.shape[1:])
    for i in range(3):
        for k in range(3):
            j[i, k] = np.gradient(s_mm[i], sp[k], axis=k) if src.shape[1 + k] >= 2 else float(i == k)
    a, b, c = j[0]
    d, e, f = j[1]
    g, h, k_ = j[2]
    return a * (e * k_ - f * h) - b * (d * k_ - f * g) + c * (d * h - e * g)


def _warp_indicator(ind, coords, moved):
    """Sub-voxel warp of a binary indicator: Gaussian-smoothed (1 voxel), resampled linearly, thresholded at the level that keeps its
    voxel count (insertion._warp_tumor_labels)."""
    k = int(ind.sum())
    if k == 0:
        return np.zeros(int(moved.sum()), bool)
    f = ndi.gaussian_filter(ind.astype(np.float32), 1.0, mode="nearest")
    flat = f.ravel()
    tau = float(np.clip(np.partition(flat, flat.size - k)[flat.size - k], 0.05, 0.95))
    return ndi.map_coordinates(f, coords[:, moved], order=1, mode="nearest") >= tau


def mass_effect(hu, seg, vessels, spacing, centre, radius_mm, intensity, p):
    """Local scaling warp around `centre` (float voxel index) IN PLACE on hu (cubic, clipped to the local range), seg (liver by nearest
    neighbour, tumour labels 2 and NEW by the smoothed indicator) and vessels (nearest). Only voxels with gamma < gamma_max change.
    Returns a dict (applied, intensity_requested / applied, gamma_max_mm, jacobian_min, max_displacement_mm, roi, reductions)."""
    sp = np.asarray(spacing, np.float64)
    gmax = p.gamma_factor * float(radius_mm)
    res = {"applied": False, "intensity_requested": float(intensity), "intensity_applied": 0.0, "gamma_max_mm": gmax, "jacobian_min": 1.0,
           "max_displacement_mm": 0.0, "roi": None, "reductions": []}
    if not p.mass_effect or intensity <= 0 or radius_mm <= 0:
        return res
    c = np.asarray(centre, np.float64)
    roi = compute_roi(hu.shape, sp, c, gmax)
    grid = np.stack(np.meshgrid(*[np.arange(s.start, s.stop, dtype=np.float64) for s in roi], indexing="ij"))
    rel = grid - c[:, None, None, None]
    gamma = np.sqrt(((rel * sp[:, None, None, None]) ** 2).sum(0))
    inside = gamma < gmax
    cur, src = float(intensity), None
    while cur > 0:
        cand = np.where(inside[None], c[:, None, None, None] + rel * _ratio(gamma, gmax, cur)[None], grid)
        jac = float(sampling_jacobian(cand, sp).min())
        disp = float(np.sqrt((((cand - grid) * sp[:, None, None, None]) ** 2).sum(0)).max())
        if np.isfinite(jac) and jac >= p.min_jacobian and disp <= p.max_displacement_mm:
            src = cand
            break
        res["reductions"].append({"intensity": cur, "jacobian_min": jac, "max_displacement_mm": disp})
        cur = max(cur - p.intensity_step, 0.0)
    if src is None:
        return res
    local = src - np.array([s.start for s in roi], np.float64)[:, None, None, None]
    moved = np.any(src != grid, axis=0)
    sub = hu[roi]
    near = tuple(np.clip(np.rint(local[i, moved]).astype(np.intp), 0, sub.shape[i] - 1) for i in range(3))
    lo_l = ndi.minimum_filter(sub, size=3, mode="nearest")[near]
    hi_l = ndi.maximum_filter(sub, size=3, mode="nearest")[near]
    vals = np.clip(ndi.map_coordinates(sub.astype(np.float64), local[:, moved], order=3, mode="nearest"), lo_l, hi_l)
    s_sub, v_sub = seg[roi], vessels[roi]
    liver_new = ndi.map_coordinates((s_sub > 0).astype(np.uint8), local[:, moved], order=0, mode="nearest") > 0
    tum = {lab: _warp_indicator(s_sub == lab, local, moved) for lab in (2, NEW)}
    lab_new = liver_new.astype(np.uint8)
    for lab in (2, NEW):
        lab_new[tum[lab]] = lab
    ves_new = ndi.map_coordinates(v_sub.astype(np.uint8), local[:, moved], order=0, mode="nearest") > 0
    sub[moved] = vals.astype(sub.dtype)
    s_sub[moved] = lab_new
    v_sub[moved] = ves_new & (lab_new == 1)
    res.update(applied=True, intensity_applied=cur, jacobian_min=jac, max_displacement_mm=disp, roi=roi)
    return res


# -----------------------------
# 6. Capsule
# -----------------------------
def signed_distance_mm(mask, spacing):
    """Signed distance (mm) to the voxel-face surface of a binary mask, negative inside (box distance, insertion.signed_distance_mm)."""
    sp = np.asarray(spacing, np.float64)
    m = np.pad(np.asarray(mask) > 0, 1)
    grid = np.indices(m.shape).astype(np.float64)

    def box(binary):
        _, idx = ndi.distance_transform_edt(binary, sampling=sp, return_indices=True)
        diff = np.abs(grid - idx) * sp[:, None, None, None] - 0.5 * sp[:, None, None, None]
        return np.sqrt((np.maximum(diff, 0.0) ** 2).sum(0))

    sd = np.where(m, -box(m), box(~m))
    return sd[1:-1, 1:-1, 1:-1].astype(np.float32)


def apply_capsule(hu, seg, spacing, phase, p):
    """Add the capsule ring around label NEW (additive peak CAPSULE_HU[phase], Gaussian profile zero at the band edges), inside the
    liver only, IN PLACE. Returns the peak HU (0 when nothing was added)."""
    peak = CAPSULE_HU.get(phase, CAPSULE_HU["unknown"])
    tm = seg == NEW
    if not p.capsule or peak == 0 or not tm.any():
        return 0.0
    sp = np.asarray(spacing, np.float64)
    reach = max(abs(p.capsule_inner_mm), abs(p.capsule_outer_mm))
    box = _bbox(tm, np.ceil(reach / sp).astype(int) + 2)
    sd = signed_distance_mm(tm[box], sp)
    lo, hi, sig = p.capsule_inner_mm, p.capsule_outer_mm, p.capsule_sigma_mm
    c0, w = 0.5 * (lo + hi), 0.5 * (hi - lo)
    g, g_edge = np.exp(-0.5 * ((sd - c0) / sig) ** 2), math.exp(-0.5 * (w / sig) ** 2)
    prof = np.where((sd > lo) & (sd < hi), np.maximum((g - g_edge) / (1 - g_edge), 0), 0)
    delta = (peak * prof).astype(np.float32)
    delta[~(seg[box] > 0)] = 0
    hu[box] += delta
    return float(peak)


# -----------------------------
# 7. Quality control
# -----------------------------
def _chk(passed, value, threshold, message):
    return {"passed": bool(passed), "value": value, "threshold": threshold, "message": message}


def qc_insertion(hu_b, seg_b, ves_b, hu_a, seg_a, spacing, expected_radius_mm, me, p, host_range):
    """Checks of one insertion inside its edit box (_b: backup before, _a: after; the new tumour is label NEW in seg_a)."""
    sp = np.asarray(spacing, np.float64)
    tum = seg_a == NEW
    n = int(tum.sum())
    checks = {"nonempty": _chk(n > 0, n, 1, f"{n} new tumour voxels")}
    if n:
        slack = me["max_displacement_mm"] if me["applied"] else 0.0
        host = seg_b > 0
        permitted = ndi.distance_transform_edt(~host, sampling=sp) <= slack if slack > 0 else host
        frac_out = float((tum & ~permitted).sum()) / n
        checks["tumor_in_liver"] = _chk(frac_out <= p.max_outside_liver_frac, frac_out, p.max_outside_liver_frac,
                                        f"{100 * frac_out:.2f}% of the tumour outside the host liver (+{slack:.1f} mm warp slack)")
        if p.vessels:
            ov = float((tum & ves_b).sum()) / n
            checks["vessel_overlap"] = _chk(ov <= p.max_vessel_overlap_frac, ov, p.max_vessel_overlap_frac, f"{100 * ov:.2f}% in vessels")
        r = equivalent_radius_mm(tum, sp)
        rel = abs(r - expected_radius_mm) / max(expected_radius_mm, 1e-6)
        checks["radius"] = _chk(rel <= p.radius_tol, {"actual_mm": r, "expected_mm": expected_radius_mm}, p.radius_tol,
                                f"equivalent radius {r:.2f} mm, expected {expected_radius_mm:.2f} mm")
        vals = hu_a[tum].astype(np.float64)
        d = ndi.distance_transform_edt(~tum, sampling=sp)
        shell = (d >= p.host_shell_mm[0]) & (d <= p.host_shell_mm[1]) & (seg_b == 1) & ~ves_b
        ref = float(hu_b[shell].mean()) if shell.sum() >= 30 else float(hu_b[seg_b == 1].mean())
        contrast, std = float(vals.mean() - ref), float(vals.std())
        lo, hi = p.contrast_hu_bounds
        checks["contrast"] = _chk(lo <= contrast <= hi, contrast, list(p.contrast_hu_bounds), f"tumour minus parenchyma {contrast:.1f} HU")
        lo, hi = p.tumor_std_hu_bounds
        checks["texture_std"] = _chk(lo <= std <= hi, std, list(p.tumor_std_hu_bounds), f"tumour HU std {std:.1f}")
        ncomp = int(ndi.label(tum, structure=STRUCT26)[1])
        checks["components"] = _chk(ncomp == 1, ncomp, 1, f"{ncomp} connected component(s)")
    fin = bool(np.isfinite(hu_a).all())
    lo_r, hi_r = min(host_range[0], -1024.0), max(host_range[1], 3071.0)
    in_range = fin and float(hu_a.min()) >= lo_r and float(hu_a.max()) <= hi_r
    checks["finite_range"] = _chk(in_range, [float(np.nanmin(hu_a)), float(np.nanmax(hu_a))], [lo_r, hi_r], "HU finite and in range")
    if me["applied"]:
        ok = me["jacobian_min"] >= p.min_jacobian and me["max_displacement_mm"] <= p.max_displacement_mm
        checks["displacement"] = _chk(ok, {"jacobian_min": me["jacobian_min"], "max_displacement_mm": me["max_displacement_mm"]},
                                      {"min_jacobian": p.min_jacobian, "max_displacement_mm": p.max_displacement_mm}, "warp valid")
    reasons = [k for k, v in checks.items() if not v["passed"]]
    return {"passed": not reasons, "reasons": reasons, "checks": checks}


# -----------------------------
# One insertion, several insertions, a host run
# -----------------------------
def insert_one_improved(hu, seg, vessels, spacing, samples, rng, p, phase="unknown", host_range=None):
    """One tumour into float32 hu / uint8 seg / bool vessels IN PLACE (all writes inside an edit box that is restored when QC rejects).
    Returns an info dict with accepted (bool), reason, qc and the insertion details."""
    sp = np.asarray(spacing, np.float64)
    host_range = host_range or (float(hu.min()), float(hu.max()))
    k = pick_sample(samples, p.target_radius_mm, rng, p)
    patch_rs, m_rs, scale = prepare_patch(samples["patches"][k], samples["masks"][k], spacing, samples["work_mm"], p.target_radius_mm, p)
    info = {"accepted": False, "reason": "", "sample": k, "scale": scale}
    if not m_rs.any():
        return dict(info, reason="mask_vanished")
    r_eq = equivalent_radius_mm(m_rs, sp)
    info["resampled_radius_mm"] = r_eq
    cen = np.rint(np.argwhere(m_rs).mean(0)).astype(int)
    m_idx = np.argwhere(m_rs)
    maps = HostMaps(seg, vessels if p.vessels else None, sp)
    cands, counts = candidate_centres(maps, r_eq, cen, m_rs.shape, seg.shape, p)
    info["candidate_counts"] = counts
    if not len(cands):
        empty = next((key for key in ("depth", "vessel", "tumour_gap", "window") if counts[key] == 0), "window")
        return dict(info, reason=f"no_site_{empty}")
    centre = None
    for _ in range(p.fit_attempts):
        c = cands[rng.integers(len(cands))]
        depth, vdist, tdist = maps.stats(m_idx + (c - cen))
        if depth >= p.margin_liver_mm and vdist >= p.margin_vessel_mm and tdist >= p.exclude_mm:
            centre, info["surface_clearance_mm"] = c, {"liver": depth, "vessel": vdist, "tumour": tdist}
            break
    if centre is None:
        return dict(info, reason="shape_clearance")
    start = centre - cen
    radius = p.target_radius_mm or r_eq
    intensity = p.mass_intensity if p.mass_intensity is not None else MASS_INTENSITY[size_class_of(radius)]
    gmax = p.gamma_factor * radius
    win = tuple(slice(int(s), int(s + n)) for s, n in zip(start, m_rs.shape))
    reach = gmax + max(abs(p.capsule_inner_mm), abs(p.capsule_outer_mm)) + 2.0
    roi = compute_roi(seg.shape, sp, start + np.argwhere(m_rs).mean(0), reach, pad_vox=8)
    box = tuple(slice(min(a.start, b.start), max(a.stop, b.stop)) for a, b in zip(win, roi))
    hu_b, seg_b, ves_b = hu[box].copy(), seg[box].copy(), vessels[box].copy()
    bl = blend_at(hu, seg, vessels, sp, patch_rs, m_rs, start, rng, p)
    if bl is None:
        return dict(info, reason="no_liver_voxels")
    warp_centre = start + np.argwhere(bl["mask"]).mean(0)  # centroid of the placed tumour (float voxel index)
    me = mass_effect(hu, seg, vessels, sp, warp_centre, radius, intensity, p)
    cap = apply_capsule(hu, seg, sp, phase, p)
    r_pre = equivalent_radius_mm(bl["mask"], sp)
    expected = expected_radius_after_warp(p.target_radius_mm or r_pre, gmax, me["intensity_applied"]) if me["applied"] else (p.target_radius_mm or r_pre)
    qc = qc_insertion(hu_b, seg_b, ves_b, hu[box], seg[box], sp, expected, me, p, host_range)
    accepted = qc["passed"] or p.on_fail == "flag"
    if accepted:
        sub = seg[box]
        n_new = int((sub == NEW).sum())
        sub[sub == NEW] = 2
    else:
        hu[box], seg[box], vessels[box] = hu_b, seg_b, ves_b
        n_new = 0
    info.update(accepted=accepted, flagged=accepted and not qc["passed"], reason="" if qc["passed"] else "qc:" + ";".join(qc["reasons"]),
                qc=qc, window=win, box=box, centre=centre, radius_requested_mm=p.target_radius_mm, radius_pre_warp_mm=r_pre,
                radius_expected_mm=expected, shift_hu=bl["shift_hu"], host_ref_hu=bl["host_ref_hu"], gen_ref_hu=bl["gen_ref_hu"],
                noise_std=bl["noise_std"], n_label=n_new, mass_effect={k_: v for k_, v in me.items() if k_ != "roi"}, capsule_hu=cap)
    return info


def insert_tumours(hu, seg, spacing, samples, rng, n=1, params=None, phase="unknown", vessels=None):
    """Up to n insertions into copies of a host. samples: dict with patches (k,P,P,P) in [-1,1], masks, work_mm and optionally
    radius_mm (an npz written by syn_training_improved.save_samples or Syn_Img_Pipeline_fixed). Returns (hu float32, seg uint8,
    vessels bool (warped), infos: one dict per attempted insertion)."""
    p = params or InsertParams()
    samples = dict(samples)
    samples["work_mm"] = float(samples.get("work_mm", 1.0))
    assert len(samples["patches"]), "no generated samples"
    out_hu, out_seg = hu.astype(np.float32).copy(), seg.astype(np.uint8).copy()
    if vessels is None:
        vessels = segment_vessels(out_hu, out_seg, spacing, p)[0] if p.vessels else np.zeros(seg.shape, bool)
    vessels = np.asarray(vessels, bool).copy()
    host_range = (float(out_hu.min()), float(out_hu.max()))
    infos = []
    for _ in range(n):
        info = insert_one_improved(out_hu, out_seg, vessels, spacing, samples, rng, p, phase, host_range)
        infos.append(info)
        logger.info(f"insertion: {'accepted' if info['accepted'] else 'rejected'}{' (flagged)' if info.get('flagged') else ''}"
                    f"{' ' + info['reason'] if info['reason'] else ''}; r {info.get('resampled_radius_mm', float('nan')):.1f} mm, "
                    f"mass effect I={info.get('mass_effect', {}).get('intensity_applied', 0):g}")
    return out_hu, out_seg, vessels, infos


QC_COLUMNS = ("case", "attempt", "accepted", "flagged", "reason", "sample", "scale", "radius_requested_mm", "resampled_radius_mm",
              "radius_expected_mm", "radius_actual_mm", "centre_x", "centre_y", "centre_z", "vessel_clearance_mm", "liver_clearance_mm",
              "mass_intensity", "jacobian_min", "max_displacement_mm", "shift_hu", "contrast_hu", "n_label")


def qc_row(cid, j, info):
    qc = info.get("qc", {}).get("checks", {})
    c = info.get("centre")
    cl = info.get("surface_clearance_mm", {})
    me = info.get("mass_effect", {})
    return {"case": cid, "attempt": j, "accepted": info["accepted"], "flagged": info.get("flagged", False), "reason": info["reason"],
            "sample": info.get("sample"), "scale": info.get("scale"), "radius_requested_mm": info.get("radius_requested_mm"),
            "resampled_radius_mm": info.get("resampled_radius_mm"), "radius_expected_mm": info.get("radius_expected_mm"),
            "radius_actual_mm": (qc.get("radius", {}).get("value") or {}).get("actual_mm"),
            "centre_x": None if c is None else int(c[0]), "centre_y": None if c is None else int(c[1]), "centre_z": None if c is None else int(c[2]),
            "vessel_clearance_mm": cl.get("vessel"), "liver_clearance_mm": cl.get("liver"), "mass_intensity": me.get("intensity_applied"),
            "jacobian_min": me.get("jacobian_min"), "max_displacement_mm": me.get("max_displacement_mm"), "shift_hu": info.get("shift_hu"),
            "contrast_hu": qc.get("contrast", {}).get("value"), "n_label": info.get("n_label", 0)}


def _fmt(v):
    if v is None:
        return ""
    if isinstance(v, float):
        return "inf" if math.isinf(v) else f"{v:.4g}"
    return str(v)


def run_insert_improved(samples_path, ids, out_dir, data_dir=S.DATA_DIR, n=S.INSERTIONS, params=None, rng=None, phase="unknown", fig_path=None):
    """Insert into the NIfTI hosts `ids` (the train part only: S.split_cases), write aug_volume-<id>.nii (int16),
    aug_segmentation-<id>.nii (uint8), qc_report.csv, insert_summary.json and a host / augmented / labels montage."""
    p = params or InsertParams()
    rng = rng or np.random.default_rng(S.SEED)
    assert not os.path.abspath(out_dir).startswith(os.path.abspath(data_dir)), "never write augmented files into the raw data dir"
    d = S.load_npz(samples_path)
    samples = {"patches": d["patches"], "masks": d["masks"], "work_mm": float(d["work_mm"]) if "work_mm" in d.files else 1.0,
               "radius_mm": d["radius_mm"] if "radius_mm" in d.files else None}
    if not len(samples["patches"]):
        logger.error("no generated samples: nothing to insert")
        return {"hosts_written": 0}
    os.makedirs(out_dir, exist_ok=True)
    cases, rows, shown, written = S.pair_cases(data_dir, ids), [], [], 0
    for cid in ids:
        vol, seg, sp, aff = S.load_case(*cases[cid])
        hu, seg_new, _, infos = insert_tumours(vol, seg, sp, samples, rng, n, p, phase)
        rows += [qc_row(cid, j, i) for j, i in enumerate(infos)]
        ok = [i for i in infos if i["accepted"]]
        if not ok:
            logger.warning(f"case {cid}: no tumour accepted ({[i['reason'] for i in infos]}), not written")
            continue
        S.save_nifti(S.to_int16(hu), aff, os.path.join(out_dir, f"aug_volume-{cid}.nii"), np.int16)
        S.save_nifti(seg_new, aff, os.path.join(out_dir, f"aug_segmentation-{cid}.nii"), np.uint8)
        written += 1
        if len(shown) < 4:
            shown.append((cid, S.to_int16(vol), S.to_int16(hu), seg, seg_new, ok[0], sp))
    with open(os.path.join(out_dir, "qc_report.csv"), "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(QC_COLUMNS))
        w.writeheader()
        for r in rows:
            w.writerow({k: _fmt(r.get(k)) for k in QC_COLUMNS})
    reasons = {}
    for r in rows:
        if r["reason"]:
            for x in r["reason"].replace("qc:", "").split(";"):
                reasons[x] = reasons.get(x, 0) + 1
    summary = {"hosts": len(ids), "hosts_written": written, "attempts": len(rows), "accepted": sum(r["accepted"] for r in rows),
               "flagged": sum(r["flagged"] for r in rows), "reasons": reasons, "params": {k: (list(v) if isinstance(v, tuple) else v) for k, v in asdict(p).items()},
               "phase": phase}
    S.write_json_atomic(os.path.join(out_dir, "insert_summary.json"), summary)
    if shown:
        S.save_insert_montage(fig_path or os.path.join(out_dir, "insertions.png"), shown)
    logger.info(f"improved insertion: {summary['accepted']}/{summary['attempts']} accepted, {written}/{len(ids)} hosts written -> {out_dir}; "
                f"reasons {reasons}")
    return summary


# -----------------------------
# Synthetic stand-ins (tests, notebook)
# -----------------------------
def toy_host(shape=(160, 160, 48), spacing=(0.8, 0.8, 2.5), seed=0):
    """Synthetic host: air, body (40 HU), ellipsoidal liver (110 HU, correlated noise), four bright tubular vessels (190 HU, label 1 like
    LiTS) and one existing tumour (50 HU, label 2). Returns (hu float32, seg uint8, spacing, true vessel mask)."""
    rng = np.random.default_rng(seed)
    sp = np.asarray(spacing, np.float64)
    g = np.stack(np.meshgrid(*[(np.arange(n) - n / 2) * s for n, s in zip(shape, sp)], indexing="ij"))
    body = (g[0] / 62) ** 2 + (g[1] / 58) ** 2 <= 1
    liver = (g[0] / 55) ** 2 + (g[1] / 46) ** 2 + (g[2] / 50) ** 2 <= 1
    hu = np.where(body, 40.0, -1000.0)
    hu = np.where(liver, 110.0, hu)
    ves = np.zeros(shape, bool)
    for a, b, r in ((-20, -15, 3.0), (25, 10, 2.5), (0, 30, 2.0), (-35, 20, 2.0)):
        ves |= ((g[0] - a) ** 2 + (g[1] - b) ** 2 <= r * r) & liver
    hu = np.where(ves, 190.0, hu)
    tum = ((g[0] - 30) ** 2 + (g[1] + 25) ** 2 + (g[2] - 10) ** 2 <= 8.0 ** 2) & liver
    hu = np.where(tum, 50.0, hu)
    noise = ndi.gaussian_filter(rng.standard_normal(shape), 0.8 / sp)
    hu = hu + noise / noise.std() * 10.0
    seg = np.where(liver, 1, 0).astype(np.uint8)
    seg[tum] = 2
    return hu.astype(np.float32), seg, tuple(float(s) for s in sp), ves


def toy_patch(p=64, radius_mm=10.0, work_mm=1.0, seed=0, tumour_hu=50.0, liver_hu=110.0):
    """Lobulated synthetic tumour patch in [-1,1] with its mask (a stand-in for a generated sample; insertion does not care where the
    patch came from)."""
    rng = np.random.default_rng(seed)
    g = np.stack(np.meshgrid(*[(np.arange(p) - (p - 1) / 2) * work_mm] * 3, indexing="ij"))
    rad = np.sqrt((g ** 2).sum(0))
    bump = ndi.gaussian_filter(rng.standard_normal((p,) * 3), 4.0 / work_mm)
    bump /= np.abs(bump).max() + 1e-8
    mask = rad <= radius_mm * (1 + 0.15 * bump)
    lab, n = ndi.label(mask, structure=STRUCT26)
    if n > 1:
        mask = lab == (np.argmax(np.bincount(lab.ravel())[1:]) + 1)
    tex = ndi.gaussian_filter(rng.standard_normal((p,) * 3), 1.0)
    hu = np.where(mask, tumour_hu, liver_hu) + tex / tex.std() * 8.0
    return S.to_unit(hu.astype(np.float32)), mask


# -----------------------------
# Entry point
# -----------------------------
def params_from_env():
    size = os.environ.get("LITS_INS_SIZE", "medium").strip().lower()
    target = None if size in ("keep", "none", "") else (SIZES_MM[size] if size in SIZES_MM else float(size))
    return InsertParams(target_radius_mm=target, vessels=os.environ.get("LITS_INS_NO_VESSELS", "0") != "1",
                        mass_effect=os.environ.get("LITS_INS_NO_MASS", "0") != "1", capsule=os.environ.get("LITS_INS_CAPSULE", "0") == "1",
                        on_fail=os.environ.get("LITS_INS_ON_FAIL", "reject"))


def main():
    assert S.GAN_SOURCE == "nifti", "insertion_improved.py writes NIfTI hosts; the cache source keeps using Syn_Img_Pipeline_fixed.py"
    S.setup_logging(S.RESULTS_DIR)
    S.set_seed(S.SEED)
    p = params_from_env()
    ids = S.split_cases(restrict=S.parse_volume_list(S.VOLUME_LIST) if S.VOLUME_LIST else None)
    max_hosts = int(os.environ.get("LITS_INS_MAX_HOSTS", 0))
    ids = ids[:max_hosts] if max_hosts else ids
    out = os.environ.get("LITS_INS_OUT", os.path.join(S.RESULTS_DIR, "aug_improved"))
    logger.info(f"improved insertion: {len(ids)} hosts, target radius {p.target_radius_mm} mm, vessels {p.vessels}, mass effect {p.mass_effect}, "
                f"capsule {p.capsule}, on_fail {p.on_fail}")
    run_insert_improved(os.path.join(S.RESULTS_DIR, "synthetic", "samples.npz"), ids, out, n=int(os.environ.get("LITS_INS_N", S.INSERTIONS)),
                        params=p, phase=os.environ.get("LITS_INS_PHASE", "unknown"))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile {CODE_DIR}/evaluate_synthesis.py
"""Evaluation of a finished Syn_Img_Pipeline_fixed.py run (NIfTI source): loss curves per GAN and real-vs-synthetic image metrics.

    LITS_DATA_DIR=... LITS_GAN_RESULTS_DIR=... python evaluate_synthesis.py

Reads <results>/train_metrics.json (loss_history; falls back to parsing pipeline.log), patches.npz, checkpoints/gan_last.pt,
synthetic/samples.npz and aug/. Writes <results>/evaluation/{evaluation.json, *.png}.

Real vs synthetic (tumour patches, 64^3 at 1 mm, compared in HU):
  every generator (dcgan 1, dcgan 2, wgan-gp, aggregator) is sampled N times and compared with the real tumour patches; the
  reference row "real B" is a second set of REAL patches from other volumes (volume-wise split A/B), i.e. the value a perfect
  generator would reach. Metrics: HU histogram Wasserstein-1 distance, radial power-spectrum log distance, gradient magnitude,
  Frechet distance + RBF-MMD^2 on standardised hand-crafted features (histogram, percentiles, gradients, spectrum; NOT an
  Inception FID), C2ST (random-forest real-vs-other accuracy, 5-fold CV; 0.5 = indistinguishable), SSIM of the central axial
  slice (to the nearest real patch = realism / memorisation, mean pairwise inside the set = diversity, high = mode collapse),
  mask-head gate acceptance, tumour volume and tumour-minus-surround contrast (HU).
Augmented volumes: inserted-tumour HU / contrast vs the host liver, compared with the real tumours of the same hosts, and the
  fraction of voxels that changed.
"""
import os
import re
import glob
import json
import numpy as np
import torch
import torch.nn.functional as F
from scipy import ndimage as ndi
from scipy import linalg
from scipy.stats import wasserstein_distance
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupKFold, cross_val_score
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import nibabel as nib
import Syn_Img_Pipeline_fixed as S

RES = S.RESULTS_DIR
OUT = os.path.join(RES, "evaluation")
N_SYN = int(os.environ.get("LITS_EVAL_N", 256))
GENS = ["dc1", "dc2", "wgan", "aggregator"]
LABEL = {"real_B": "real B (reference)", "dc1": "DCGAN 1", "dc2": "DCGAN 2", "wgan": "WGAN-GP", "aggregator": "Aggregator"}
LOSS_GROUPS = [("DCGAN 1", [("d1", "discriminator"), ("g1", "generator")]),
               ("DCGAN 2", [("d2", "discriminator"), ("g2", "generator")]),
               ("WGAN-GP", [("w_critic", "critic (incl. GP)"), ("w_g", "generator"), ("w_gp", "gradient penalty")]),
               ("Aggregator GAN", [("ag_d", "discriminator"), ("ag_g", "generator (adv.)"), ("ag_l1", "L1 to generator mean")]),
               ("Mask head", [("mask", "BCE + Dice")])]


# -----------------------------
# Loss curves
# -----------------------------
def loss_history():
    m = S.read_json(os.path.join(RES, "train_metrics.json"))
    if m.get("loss_history"):
        return m["loss_history"]
    hist = {}  # fallback: the "epoch k/N (..s) key=val ..." lines of pipeline.log; the last run of an epoch wins
    with open(os.path.join(RES, "pipeline.log")) as f:
        for line in f:
            r = re.search(r"epoch (\d+)/\d+ \((\d+)s\) (.*)$", line)
            if r:
                hist[int(r.group(1))] = dict({k: float(v) for k, v in re.findall(r"(\w+)=(-?[\d.]+(?:e-?\d+)?)", r.group(3))},
                                             epoch=int(r.group(1)), seconds=float(r.group(2)))
    return [hist[k] for k in sorted(hist)]


def plot_losses(hist, path):
    fig, axs = plt.subplots(1, len(LOSS_GROUPS), figsize=(4.2 * len(LOSS_GROUPS), 3.6))
    ep = [h["epoch"] for h in hist]
    for ax, (title, keys) in zip(axs, LOSS_GROUPS):
        for k, lab in keys:
            pts = [(e, h[k]) for e, h in zip(ep, hist) if k in h]
            if pts:
                ax.plot(*zip(*pts), marker="o", ms=3, lw=1.5, label=lab)
        ax.set_title(title, fontsize=11)
        ax.set_xlabel("epoch")
        ax.grid(alpha=0.3)
        ax.legend(fontsize=8)
    axs[0].set_ylabel("mean loss per epoch")
    plt.tight_layout()
    fig.savefig(path, dpi=120)
    plt.close(fig)


# -----------------------------
# Image features and metrics
# -----------------------------
def hu(x):
    return S.from_unit(np.asarray(x, np.float32))


def radial_spectrum(x, nb=16):
    """Radially averaged log power spectrum of one 3D patch (mean removed)."""
    p = np.abs(np.fft.fftshift(np.fft.fftn(x - x.mean()))) ** 2
    c = np.array(x.shape) // 2
    r = np.sqrt(sum((np.arange(n) - ci).reshape([-1 if a == ax else 1 for a in range(3)]) ** 2 for ax, (n, ci) in enumerate(zip(x.shape, c))))
    bins = np.linspace(1, x.shape[0] // 2, nb + 1)
    idx = np.digitize(r.ravel(), bins) - 1
    ok = (idx >= 0) & (idx < nb)
    s = np.bincount(idx[ok], p.ravel()[ok], nb) / np.maximum(np.bincount(idx[ok], minlength=nb), 1)
    return np.log10(s + 1e-6)


def features(h):
    """Hand-crafted per-patch features (HU input): histogram, percentiles, gradient / Laplacian statistics, power spectrum."""
    hist = np.histogram(np.clip(h, S.WIN_LO, S.WIN_HI), bins=24, range=(S.WIN_LO, S.WIN_HI))[0] / h.size
    g = np.sqrt(sum(a ** 2 for a in np.gradient(h)))
    lap = ndi.laplace(h)
    return np.concatenate([hist, np.percentile(h, [5, 25, 50, 75, 95]), [h.mean(), h.std(), g.mean(), g.std(), lap.var() ** 0.5],
                           radial_spectrum(h)])


def frechet(a, b):
    mu1, mu2 = a.mean(0), b.mean(0)
    eye = 1e-6 * np.eye(a.shape[1])  # some histogram bins are constant -> singular covariance
    s1, s2 = np.cov(a, rowvar=False) + eye, np.cov(b, rowvar=False) + eye
    cs = linalg.sqrtm(s1 @ s2)
    return float(((mu1 - mu2) ** 2).sum() + np.trace(s1 + s2 - 2 * np.real(cs)))


def mmd2(a, b):
    z = np.concatenate([a, b])
    d = ((z[:, None] - z[None]) ** 2).sum(-1)
    k = np.exp(-d / np.median(d[d > 0]))
    n = len(a)
    kxx, kyy, kxy = k[:n, :n], k[n:, n:], k[:n, n:]
    return float((kxx.sum() - np.trace(kxx)) / (n * (n - 1)) + (kyy.sum() - np.trace(kyy)) / (len(b) * (len(b) - 1)) - 2 * kxy.mean())


def c2st(a, b, ga, gb=None, seed=0):
    """Real-vs-other classifier accuracy. Folds are grouped by volume (ga / gb: volume id per patch; None = independent
    samples, e.g. generated): patches of one volume are strongly correlated, and a plain K-fold would let the classifier
    recognise the volume instead of telling the two sets apart."""
    n = min(len(a), len(b))
    rng = np.random.default_rng(seed)
    ia, ib = rng.permutation(len(a))[:n], rng.permutation(len(b))[:n]
    x = np.concatenate([a[ia], b[ib]])
    y = np.r_[np.zeros(n), np.ones(n)]
    g = np.r_[np.asarray(ga)[ia], np.asarray(gb)[ib] if gb is not None else 10 ** 6 + rng.integers(0, 5, n)]
    acc = cross_val_score(RandomForestClassifier(300, random_state=seed, n_jobs=-1), x, y, groups=g, cv=GroupKFold(5))
    return float(acc.mean()), float(acc.std())


def _ssim_maps(x, y, win=7):
    """Gaussian-window SSIM of 2D slices x (N,H,W) vs y (N,H,W) in the [-1,1] window (data range 2)."""
    g = torch.exp(-(torch.arange(win) - win // 2).float() ** 2 / (2 * 1.5 ** 2))
    k = (g[:, None] * g[None]) / g.sum() ** 2
    k = k[None, None]
    f = lambda t: F.conv2d(t[:, None], k)[:, 0]
    c1, c2 = (0.01 * 2) ** 2, (0.03 * 2) ** 2
    mx, my = f(x), f(y)
    vx, vy, cxy = f(x * x) - mx ** 2, f(y * y) - my ** 2, f(x * y) - mx * my
    return (((2 * mx * my + c1) * (2 * cxy + c2)) / ((mx ** 2 + my ** 2 + c1) * (vx + vy + c2))).mean(dim=(1, 2))


def ssim_matrix(a, b, chunk=4096):
    """SSIM of every pair (central axial slices). a (n,P,P), b (m,P,P) -> (n,m)."""
    a, b = torch.from_numpy(a).float(), torch.from_numpy(b).float()
    ii, jj = torch.meshgrid(torch.arange(len(a)), torch.arange(len(b)), indexing="ij")
    ii, jj = ii.ravel(), jj.ravel()
    out = torch.cat([_ssim_maps(a[ii[s:s + chunk]], b[jj[s:s + chunk]]) for s in range(0, len(ii), chunk)])
    return out.view(len(a), len(b)).numpy()


def mid(x):
    return np.ascontiguousarray(np.asarray(x, np.float32)[:, :, :, x.shape[-1] // 2])


def tumour_stats(patches, masks):
    """Tumour volume (voxels = mm^3 at 1 mm) and mean HU inside the tumour minus a 2-10 mm surrounding ring."""
    vol, con = [], []
    for p, m in zip(patches, masks):
        m = m > 0
        if not m.any():
            continue
        d = ndi.distance_transform_edt(~m)
        ring = (d > 2) & (d <= 10)
        h = hu(p)
        vol.append(int(m.sum()))
        con.append(float(h[m].mean() - h[ring].mean()))
    return np.array(vol), np.array(con)


def gate(model, x, min_vox):
    """Mask head + clean_mask on generated patches (x (n,P,P,P) in [-1,1]) -> (masks for accepted, acceptance rate)."""
    acc, masks = 0, np.zeros(x.shape, np.uint8)
    with torch.no_grad():
        model.train(False)
        for s in range(0, len(x), 8):
            prob = torch.sigmoid(model.mask(torch.from_numpy(x[s:s + 8])[:, None].to(model.device)))[:, 0].cpu().numpy()
            for j, p in enumerate(prob):
                m, _ = S.clean_mask(p, min_vox=min_vox)
                if m is not None:
                    masks[s + j], acc = m, acc + 1
    return masks, acc / len(x)


def sample_generators(model, n, seed):
    torch.manual_seed(seed)
    out = {g: [] for g in GENS}
    for s in range(0, n, 8):
        for k, v in model.sample_all(min(8, n - s)).items():
            out[k].append(v[:, 0].cpu().numpy())
    return {k: np.concatenate(v) for k, v in out.items()}


# -----------------------------
# Augmented volumes
# -----------------------------
def eval_augmented(data_dir, max_cases=None):
    rows = []
    files = sorted(glob.glob(os.path.join(RES, "aug", "aug_volume-*.nii")), key=S.case_id)[:max_cases]
    for f in files:
        cid = S.case_id(f)
        (vp, sp_), = S.pair_cases(data_dir, [cid]).values()
        v0, s0, _, _ = S.load_case(vp, sp_)
        v1 = np.asarray(nib.load(f).dataobj).astype(np.float32)
        s1 = np.rint(np.asarray(nib.load(os.path.join(RES, "aug", f"aug_segmentation-{cid}.nii")).dataobj)).astype(np.uint8)
        new = (s1 == 2) & (s0 != 2)
        if not new.any():
            continue
        liver = s1 == 1
        d = ndi.distance_transform_edt(~new)
        ring = (d > 2) & (d <= 6) & liver  # host liver right around the inserted tumour (voxel units)
        changed = np.abs(v1 - S.to_int16(v0)) > 0
        r = dict(case=cid, new_tumor_vox=int(new.sum()), inserted_hu=float(v1[new].mean()), host_liver_hu=float(v0[s0 == 1].mean()),
                 inserted_minus_ring_hu=float(v1[new].mean() - v1[ring].mean()) if ring.any() else None,
                 changed_vox_frac=float(changed.mean()), changed_outside_liver=int((changed & (s0 == 0)).sum()),
                 labels_changed_outside_new=int(((s1 != s0) & ~new).sum()))
        if (s0 == 2).any():
            real = s0 == 2
            dr = ndi.distance_transform_edt(~real)
            rr = (dr > 2) & (dr <= 6) & (s0 == 1)
            r.update(real_tumor_hu=float(v0[real].mean()), real_minus_ring_hu=float(v0[real].mean() - v0[rr].mean()) if rr.any() else None)
        rows.append(r)
        print(f"aug case {cid}: +{r['new_tumor_vox']} vox, inserted {r['inserted_hu']:.1f} HU, liver {r['host_liver_hu']:.1f} HU", flush=True)
    return rows


# -----------------------------
# Figures
# -----------------------------
def plot_montage(sets, path, n=8):
    rows = list(sets)
    fig, axs = plt.subplots(len(rows), n, figsize=(1.7 * n, 1.8 * len(rows)), squeeze=False)
    for r, k in enumerate(rows):
        x, m = sets[k]
        for j in range(n):
            ax = axs[r][j]
            ax.axis("off")
            z = x.shape[-1] // 2
            ax.imshow(x[j][:, :, z].T, cmap="gray", vmin=-1, vmax=1, origin="lower")
            if m is not None and m[j][:, :, z].any():
                ax.contour(m[j][:, :, z].T.astype(float), levels=[0.5], colors="r", linewidths=0.7)
            if j == 0:
                ax.set_title(k, fontsize=9, loc="left")
    plt.tight_layout()
    fig.savefig(path, dpi=110)
    plt.close(fig)


def plot_distributions(hu_vox, spectra, path):
    fig, axs = plt.subplots(1, 2, figsize=(11, 3.8))
    bins = np.linspace(S.WIN_LO, S.WIN_HI, 71)
    for k, v in hu_vox.items():
        axs[0].hist(np.clip(v, S.WIN_LO, S.WIN_HI), bins=bins, density=True, histtype="step", lw=1.6 if k == "real" else 1.1, label=k)
        axs[1].plot(np.mean(spectra[k], 0), lw=1.6 if k == "real" else 1.1, label=k)
    axs[0].set(title="HU distribution (voxels, window clipped)", xlabel="HU", ylabel="density")
    axs[1].set(title="radial power spectrum", xlabel="frequency bin (low -> high)", ylabel="log10 power")
    for a in axs:
        a.grid(alpha=0.3)
        a.legend(fontsize=8)
    plt.tight_layout()
    fig.savefig(path, dpi=120)
    plt.close(fig)


def plot_metric_bars(table, path):
    keys = [("hu_wasserstein", "HU Wasserstein-1 (HU)"), ("spectrum_log_dist", "spectrum log distance"), ("feature_frechet", "feature Frechet distance"),
            ("c2st_acc", "C2ST accuracy (0.5 ideal)"), ("ssim_to_nearest_real", "SSIM to nearest real"), ("ssim_pairwise_diversity", "pairwise SSIM (lower = diverse)")]
    names = list(table)
    fig, axs = plt.subplots(1, len(keys), figsize=(3.1 * len(keys), 3.4))
    for ax, (k, t) in zip(axs, keys):
        vals = [table[n].get(k, np.nan) for n in names]
        ax.bar(range(len(names)), vals, color=["#888" if n == "real_B" else "C%d" % i for i, n in enumerate(names)])
        ax.set_xticks(range(len(names)), [LABEL.get(n, n) for n in names], rotation=40, ha="right", fontsize=8)
        ax.set_title(t, fontsize=9)
        ax.grid(axis="y", alpha=0.3)
        if k == "c2st_acc":
            ax.axhline(0.5, color="k", ls="--", lw=0.8)
            ax.set_ylim(0.4, 1.0)
    plt.tight_layout()
    fig.savefig(path, dpi=120)
    plt.close(fig)


# -----------------------------
# Main
# -----------------------------
def main():
    os.makedirs(OUT, exist_ok=True)
    S.set_seed(S.SEED)
    res = {"results_dir": RES}
    hist = loss_history()
    plot_losses(hist, os.path.join(OUT, "losses_per_gan.png"))
    res["loss_history"] = hist
    print(f"loss curves: {len(hist)} epochs -> losses_per_gan.png", flush=True)

    d = S.load_npz(os.path.join(RES, "patches.npz"))
    imgs, masks, cids = d["images"].astype(np.float32), d["masks"], d["meta"][:, 0]
    work = float(d["work_mm"])
    # volume-wise split of the real patches: A = reference set, B = "perfect generator" baseline
    rng = np.random.default_rng(S.SEED)
    vols = rng.permutation(np.unique(cids))
    b_vols = set(vols[: len(vols) // 2].tolist())
    in_b = np.isin(cids, list(b_vols))
    real, real_m = imgs, masks
    model = S.MultiGAN.load(os.path.join(RES, "checkpoints", "gan_last.pt"), torch.device("cpu"))
    n = min(N_SYN, len(real))
    syn = sample_generators(model, n, S.SEED + 1)
    min_vox = S.vox_at(S.MIN_MASK_VOX, work)
    sets = {"real_B": (imgs[in_b], masks[in_b])}
    gate_rate = {}
    for g in GENS:
        m, gate_rate[g] = gate(model, syn[g], min_vox)
        sets[g] = (syn[g], m)
        print(f"{g}: mask-gate acceptance {gate_rate[g]:.2%}", flush=True)

    feat = lambda x: np.stack([features(hu(p)) for p in x])
    print("features ...", flush=True)
    f_real, f_a = feat(real), None
    f_a = f_real[~in_b]
    mu, sd = f_real.mean(0), f_real.std(0) + 1e-6
    z = lambda f: (f - mu) / sd
    spec = lambda x: np.stack([radial_spectrum(hu(p)) for p in x])
    s_real = spec(real)
    sub = lambda x, k=40000: np.random.default_rng(0).choice(hu(x).ravel(), k, replace=False)
    hu_real = sub(real)
    mid_real, mid_a = mid(real), mid(imgs[~in_b])
    rv, rc = tumour_stats(real, real_m)
    res["real"] = dict(patches=int(len(real)), volumes=int(len(np.unique(cids))), hu_mean=float(hu(real).mean()), hu_std=float(hu(real).std()),
                       tumour_vox_median=float(np.median(rv)), tumour_contrast_hu_mean=float(rc.mean()),
                       ssim_pairwise_diversity=float(np.mean(ssim_matrix(mid_real[:n], mid_real[:n])[np.triu_indices(min(n, len(real)), 1)])))
    table, hu_vox, spectra = {}, {"real": hu_real}, {"real": s_real}
    for k, (x, m) in sets.items():
        # real_B is compared with real A only (otherwise it would be compared with itself); generators with all real patches
        ref_f, ref_mid, ref_hu, ref_s = (f_a, mid_a, sub(imgs[~in_b]), spec(imgs[~in_b])) if k == "real_B" else (f_real, mid_real, hu_real, s_real)
        fx, sx = feat(x), spec(x)
        sm = ssim_matrix(mid(x), ref_mid)
        own = ssim_matrix(mid(x[:n]), mid(x[:n]))
        v, c = tumour_stats(x, m)
        acc, acc_sd = c2st(z(ref_f), z(fx), cids[~in_b], cids[in_b]) if k == "real_B" else c2st(z(ref_f), z(fx), cids)
        table[k] = dict(n=int(len(x)), hu_mean=float(hu(x).mean()), hu_std=float(hu(x).std()),
                        hu_wasserstein=float(wasserstein_distance(ref_hu, sub(x))),
                        spectrum_log_dist=float(np.sqrt(((ref_s.mean(0) - sx.mean(0)) ** 2).mean())),
                        grad_mean=float(fx[:, 24 + 5 + 2].mean()), feature_frechet=frechet(z(ref_f), z(fx)), feature_mmd2=mmd2(z(ref_f), z(fx)),
                        c2st_acc=acc, c2st_std=acc_sd, ssim_to_nearest_real=float(sm.max(1).mean()),
                        ssim_pairwise_diversity=float(own[np.triu_indices(len(own), 1)].mean()),
                        mask_gate_acceptance=gate_rate.get(k, 1.0), tumour_vox_median=float(np.median(v)) if len(v) else None,
                        tumour_contrast_hu_mean=float(c.mean()) if len(c) else None)
        hu_vox[LABEL[k]], spectra[LABEL[k]] = sub(x), sx
        print(f"{k}: " + " ".join(f"{a}={b:.3f}" for a, b in table[k].items() if isinstance(b, float)), flush=True)
    res["real_vs_synthetic"] = table
    res["real_grad_mean"] = float(f_real[:, 24 + 5 + 2].mean())
    plot_montage({"real": (real[:8], real_m[:8]), **{LABEL[g]: sets[g] for g in GENS}}, os.path.join(OUT, "real_vs_synthetic_montage.png"))
    plot_distributions(hu_vox, spectra, os.path.join(OUT, "hu_and_spectrum.png"))
    plot_metric_bars(table, os.path.join(OUT, "metrics_bars.png"))

    if os.path.isdir(os.path.join(RES, "aug")):
        print("augmented volumes ...", flush=True)
        rows = eval_augmented(S.DATA_DIR)
        res["augmented"] = rows
        if rows:
            fig, ax = plt.subplots(figsize=(5.5, 3.8))
            a = [r["inserted_minus_ring_hu"] for r in rows if r.get("inserted_minus_ring_hu") is not None]
            b = [r["real_minus_ring_hu"] for r in rows if r.get("real_minus_ring_hu") is not None]
            ax.boxplot([b, a], tick_labels=[f"real tumours (n={len(b)})", f"inserted tumours (n={len(a)})"])
            ax.axhline(0, color="k", lw=0.6)
            ax.set(ylabel="tumour mean HU minus surrounding liver (HU)", title="tumour contrast in the augmented hosts")
            ax.grid(axis="y", alpha=0.3)
            plt.tight_layout()
            fig.savefig(os.path.join(OUT, "augmented_contrast.png"), dpi=120)
            plt.close(fig)
    S.write_json_atomic(os.path.join(OUT, "evaluation.json"), res)
    print(f"done -> {OUT}", flush=True)


if __name__ == "__main__":
    main()

## Configure and import

Settings become the pipeline's environment variables, which are read at import time. After changing a setting, re-run from here; the
modules are re-imported.

In [ ]:
demo = MODE == "demo"
env = {
    "LITS_GAN_RESULTS_DIR": RESULTS, "LITS_GAN_SOURCE": "cache" if MODE == "cache" else "nifti", "LITS_GAN_SEED": str(SEED),
    "LITS_GAN_EPOCHS": str(8 if demo else EPOCHS), "LITS_GAN_RESUME": "1" if RESUME else "0",
    "LITS_GAN_MAX_SECONDS": str(0 if demo else int(MAX_TRAIN_MINUTES * 60)), "LITS_GAN_ES_PATIENCE": str(0 if demo else EARLY_STOP_PATIENCE),
    "LITS_GAN_MAX_VOLUMES": str(MAX_VOLUMES), "LITS_DEVICE": "auto", "LITS_GAN_PATCH": "32" if demo else "64",
    "LITS_GAN_WIDTH": "8" if demo else "32", "LITS_EVAL_N": "128",
}
if MODE == "nifti":
    env["LITS_DATA_DIR"] = NIFTI_DIR
elif MODE == "cache":
    env.update(LITS_CACHE_DIR=CACHE_DIR, LITS_PER_VOLUME=PER_VOLUME_CSV)
else:
    env["LITS_DATA_DIR"] = os.path.join(RESULTS, "toy_raw")  # synthetic hosts are written there below
os.environ.update(env)
for m in ("Syn_Img_Pipeline_fixed", "syn_training_improved", "insertion_improved", "evaluate_synthesis"):
    sys.modules.pop(m, None)
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)
import json, time, logging
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import syn_training_improved as T
import insertion_improved as I
S = T.S
%matplotlib inline
S.setup_logging(RESULTS)
S.set_seed(SEED)
DEVICE = S.pick_device()
S.setup_backends(DEVICE)
PATCHES = os.path.join(RESULTS, "patches.npz")
SAMPLES = os.path.join(RESULTS, "synthetic", "samples.npz")

def show(path, w=12, h=4):
    if os.path.exists(path):
        plt.figure(figsize=(w, h)); plt.imshow(plt.imread(path)); plt.axis("off"); plt.show()

print(f"mode {MODE}, device {DEVICE}, patch {S.PATCH}^3 at {S.WORK_MM} mm, width {S.WIDTH}, epochs {S.EPOCHS}")

## 1. Prepare tumour patches

Liver crop → isotropic resampling → fixed HU window → tumour patches per connected component plus healthy patches (see
`Syn_Img_Pipeline_fixed.prepare_patches`). With `RESUME` an existing `patches.npz` built from the same volumes is kept.

In [ ]:
if demo:
    rng = np.random.default_rng(SEED)
    imgs, masks, meta = [], [], []
    for i in range(48):
        x, m = I.toy_patch(S.PATCH, radius_mm=float(rng.uniform(3.5, 9.0)), seed=i, tumour_hu=float(rng.uniform(35, 70)),
                           liver_hu=float(rng.uniform(100, 120)))
        imgs.append(x); masks.append(m); meta.append((i // 6, 1))
    healthy = [S.to_unit((110 + 8 * rng.standard_normal((S.PATCH,) * 3)).astype(np.float32)) for _ in range(12)]
    S.save_npz_atomic(PATCHES, images=np.stack(imgs).astype(np.float16), masks=np.stack(masks).astype(np.uint8),
                      healthy=np.stack(healthy).astype(np.float16), meta=np.array(meta), healthy_cid=np.arange(12) % 8,
                      work_mm=np.float32(1.0), ids=np.arange(8))
    toy_raw = os.environ["LITS_DATA_DIR"]
    os.makedirs(toy_raw, exist_ok=True)
    for cid in (1, 2, 3):
        h_, s_, sp_, _ = I.toy_host(seed=cid)
        S.save_nifti(S.to_int16(h_), np.diag(list(sp_) + [1.0]), os.path.join(toy_raw, f"volume-{cid}.nii"), np.int16)
        S.save_nifti(s_, np.diag(list(sp_) + [1.0]), os.path.join(toy_raw, f"segmentation-{cid}.nii"), np.uint8)
    ids = [1, 2, 3]
else:
    source = S.make_source("cache") if MODE == "cache" else S.NiftiSource(S.DATA_DIR, S.WORK_MM)
    restrict = S.parse_volume_list(S.VOLUME_LIST) if S.VOLUME_LIST else None
    ids = source.split_cases(restrict)
    print(f"{len(ids)} usable train-part volumes: {ids[:15]}{' ...' if len(ids) > 15 else ''}")
    keep = False
    if RESUME and os.path.exists(PATCHES):
        z = S.load_npz(PATCHES)
        keep = "ids" in z.files and z["ids"].tolist() == ids and z["images"].shape[-1] == S.PATCH and float(z["work_mm"]) == S.WORK_MM
        print("patches.npz exists:", "same volumes, kept" if keep else "different settings, rebuilding")
    if not keep:
        t0 = time.time()
        S.prepare_patches(ids, PATCHES, size=S.PATCH, source=source)
        print(f"prepared in {(time.time() - t0) / 60:.1f} min")
z = S.load_npz(PATCHES)
print(f"{len(z['images'])} tumour / {len(z['healthy'])} healthy patches, {z['images'].shape[-1]}^3 at {float(z['work_mm'])} mm, "
      f"{len(np.unique(z['meta'][:, 0]))} volumes")
S.save_montage(os.path.join(RESULTS, "figures", "real_patches.png"), [("real tumour patches", z["images"][:8].astype(np.float32), z["masks"][:8] > 0)])
show(os.path.join(RESULTS, "figures", "real_patches.png"), 12, 2.4)

## 2. Train the improved multi-GAN

Two DCGANs, a WGAN-GP, the aggregator and the supervised mask head (as in `Syn_Img_Pipeline_fixed.py`), plus DiffAugment on every
discriminator input, an EMA of the generators, the WGAN drift term and a quality score per epoch. The score is the HU distance
between generated and real held-out tumours, lower is better. The best epoch is saved as `checkpoints/gan_best.pt`; early stopping
uses `EARLY_STOP_PATIENCE`. Training stops after the epoch that crosses `MAX_TRAIN_MINUTES`. Re-run with `RESUME = True` to continue.

In [ ]:
t0 = time.time()
model = T.train_improved(PATCHES, S.EPOCHS, DEVICE, RESULTS, resume=RESUME, max_seconds=S.MAX_SECONDS, patience=T.ES_PATIENCE,
                         quality_n=8 if demo else T.QUALITY_N, sample_every=S.EPOCHS if demo else S.SAMPLE_EVERY)
print(f"training cell: {(time.time() - t0) / 60:.1f} min")
m = S.read_json(os.path.join(RESULTS, "train_metrics.json"))
print("epochs done:", m.get("epochs_done"), "| stopped:", m.get("stopped"), "| best epoch:", m.get("best_epoch"))
h = pd.DataFrame(m["loss_history"]).set_index("epoch")
q = pd.DataFrame(m.get("quality_history", [])).set_index("epoch") if m.get("quality_history") else pd.DataFrame()
fig, axs = plt.subplots(1, 4, figsize=(17, 3.2))
for ax, (title, keys) in zip(axs, [("DCGANs", ["d1", "g1", "d2", "g2"]), ("WGAN-GP", ["w_critic", "w_gp", "w_g"]),
                                   ("Aggregator / mask head", ["ag_d", "ag_g", "ag_l1", "mask"]), ("quality score (HU, lower = better)", ["score"])]):
    for k in keys:
        src = q if k in q else h
        if k in src and src[k].notna().any():
            ax.plot(src.index, src[k], marker="o", ms=3, label=k)
    ax.set_title(title); ax.set_xlabel("epoch"); ax.grid(alpha=.3); ax.legend(fontsize=7)
plt.tight_layout(); plt.show()
for tag in ("train", "heldout"):
    if f"mask_head_{tag}" in m:
        print(f"mask head {tag}:", {k: round(v, 3) for k, v in m[f'mask_head_{tag}'].items()})
figs = sorted(glob.glob(os.path.join(RESULTS, "figures", "train_epoch*.png")))
if figs:
    show(figs[-1], 9, 9)

## 3. Size-controlled generation

Samples come from the best checkpoint (EMA weights). Each sample is labelled by the mask head, cleaned, measured and kept until every
size class has `SAMPLES_PER_CLASS` (small ≤ 6 mm, medium ≤ 14 mm, large ≤ 28 mm equivalent radius). The radius histogram shows which
sizes the GAN produces. A class it rarely produces may stay short.

In [ ]:
ck = os.path.join(RESULTS, "checkpoints", "gan_best.pt")
ck = ck if os.path.exists(ck) else os.path.join(RESULTS, "checkpoints", "gan_last.pt")
gen_model = T.ImprovedMultiGAN.load(ck, DEVICE)
st = {}
classes = ("small", "medium") if demo else ("small", "medium", "large")
by_size = T.generate_by_size(gen_model, 4 if demo else SAMPLES_PER_CLASS, classes, S.FUSION, S.WORK_MM,
                             max_attempts=64 if demo else None, stats=st)
if sum(len(v[0]) for v in by_size.values()) == 0:
    assert demo, "no generated sample passed the mask gate: train longer (see the quality score / gate_accept above)"
    print("demo: nothing passed the gate, using procedural stand-in patches")
    ps, ms = zip(*[I.toy_patch(S.PATCH, float(r), seed=j) for j, r in enumerate((4.5, 7.0, 9.0))])
    by_size = {"medium": (np.stack(ps), np.stack(ms).astype(np.uint8), np.array([I.equivalent_radius_mm(x, (1, 1, 1)) for x in ms], np.float32))}
T.save_samples(SAMPLES, by_size, S.WORK_MM)
S.write_json_atomic(os.path.join(RESULTS, "synthetic", "generation_stats.json"), st)
print(f"from {os.path.basename(ck)}: accepted {st.get('accepted')} in {st.get('tried')} attempts; rejects {st.get('rejects')}")
if st.get("radius_hist_mm"):
    e, c = st["radius_hist_mm"]["edges"], st["radius_hist_mm"]["counts"]
    plt.figure(figsize=(6, 2.6)); plt.bar(e[:-1], c, width=1.8, align="edge"); plt.xlabel("equivalent radius of gated samples (mm)")
    plt.ylabel("count"); plt.title("sizes the generator produces"); plt.show()
rows = [(f"{c} (r {r.min():.1f}-{r.max():.1f} mm)", p[:8], m_[:8] > 0) for c, (p, m_, r) in by_size.items() if len(p)]
S.save_montage(os.path.join(RESULTS, "figures", "generated_by_size.png"), rows)
show(os.path.join(RESULTS, "figures", "generated_by_size.png"), 12, 2.4 * len(rows))

## 4. Insert into healthy-liver hosts

**NIfTI and demo modes** use `insertion_improved.py`: a vessel mask, vessel-aware placement checked on the real tumour shape, the
requested physical radius (`INSERT_SIZE`), vessel-free HU harmonisation, mass effect, an optional capsule and nine QC checks per
insertion. Rejected insertions are reverted. Outputs go to `<results>/aug/`: `aug_volume-<id>.nii`, `aug_segmentation-<id>.nii`,
`qc_report.csv` and `insert_summary.json`. Only the first `MAX_HOSTS` train-part hosts are used, because each written volume is hundreds of MB.

**Cache mode** keeps the original slab insertion (`Syn_Img_Pipeline_fixed.run_insert_slabs` → `aug_slabs.npz`), which is the format the
segmenter's cache training reads. The improved insertion has no cache writer yet.

In [ ]:
rng = np.random.default_rng(SEED)
if MODE == "cache":
    d = S.load_npz(SAMPLES)
    st_ins = S.run_insert_slabs(source, ids, (d["patches"], d["masks"]), os.path.join(RESULTS, "aug_slabs.npz"), S.NUM_AUG, TUMOURS_PER_HOST,
                                S.MAX_ATTEMPTS or None, rng, float(d["work_mm"]), os.path.join(RESULTS, "figures", "slab_montage.png"))
    S.write_gan_summary(RESULTS)
    print(st_ins)
    show(os.path.join(RESULTS, "figures", "slab_montage.png"), 12, 8)
else:
    params = I.InsertParams(target_radius_mm=None if INSERT_SIZE == "keep" else I.SIZES_MM[INSERT_SIZE], vessels=VESSEL_AWARE,
                            mass_effect=MASS_EFFECT, capsule=CAPSULE)
    hosts = ids[:MAX_HOSTS] if MAX_HOSTS else ids
    aug_dir = os.path.join(RESULTS, "aug")
    t0 = time.time()
    summary = I.run_insert_improved(SAMPLES, hosts, aug_dir, data_dir=S.DATA_DIR, n=TUMOURS_PER_HOST, params=params, rng=rng)
    print(f"{(time.time() - t0) / 60:.1f} min:", json.dumps({k: summary.get(k) for k in ("hosts", "hosts_written", "attempts", "accepted", "reasons")}))
    if os.path.exists(os.path.join(aug_dir, "qc_report.csv")):
        display(pd.read_csv(os.path.join(aug_dir, "qc_report.csv")))
    show(os.path.join(aug_dir, "insertions.png"), 10, 10)

## 5. Evaluation

`evaluate_synthesis.py` compares every generator (DCGAN 1, DCGAN 2, WGAN-GP, aggregator) with the real patches: HU Wasserstein
distance, power spectrum, feature Fréchet / MMD, a real-vs-synthetic classifier (0.5 = indistinguishable), SSIM realism and diversity,
and tumour contrast. A second set of real patches ("real B") gives the value a perfect generator would reach. In NIfTI mode it also
measures the inserted tumours in `aug/`. It reads the EMA checkpoint. It is skipped in demo mode, because the toy set has too few
volumes for its cross-validation.

In [ ]:
if RUN_EVALUATION and not demo:
    r = subprocess.run([sys.executable, os.path.join(CODE_DIR, "evaluate_synthesis.py")], env=dict(os.environ), capture_output=True, text=True)
    print(r.stdout[-4000:])
    if r.returncode:
        print("evaluation failed:\n", r.stderr[-3000:])
    ev = os.path.join(RESULTS, "evaluation")
    if os.path.exists(os.path.join(ev, "evaluation.json")):
        res = S.read_json(os.path.join(ev, "evaluation.json"))
        display(pd.DataFrame(res["real_vs_synthetic"]).T[["hu_wasserstein", "spectrum_log_dist", "feature_frechet", "c2st_acc",
                                                           "ssim_to_nearest_real", "ssim_pairwise_diversity", "mask_gate_acceptance",
                                                           "tumour_contrast_hu_mean"]].round(3))
        for f in ("losses_per_gan.png", "real_vs_synthetic_montage.png", "hu_and_spectrum.png", "metrics_bars.png", "augmented_contrast.png"):
            show(os.path.join(ev, f), 14, 5)
else:
    print("evaluation skipped", "(demo mode)" if demo else "(RUN_EVALUATION is off)")

## 6. Download a summary

This zips the figures, metrics, CSV and JSON files, without volumes or checkpoints (those stay in the results folder on Drive).

In [ ]:
import zipfile
zpath = os.path.join(RESULTS, f"summary_{MODE}.zip")
with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in glob.glob(os.path.join(RESULTS, "**", "*"), recursive=True):
        if f.endswith((".png", ".json", ".csv", ".log")) and "toy_raw" not in f:
            zf.write(f, os.path.relpath(f, RESULTS))
print("summary:", zpath, f"({os.path.getsize(zpath) / 1e6:.1f} MB)")
if IN_COLAB:
    from google.colab import files
    files.download(zpath)